# Full Planner — Best Strategy Simulation Notebook

This notebook is dedicated to simulating, evaluating, and discovering the **best overall strategy** for SpaceInvasion Classic across full-empire progression (economy scaling, energy optimization, research unlocks, fleet composition, raiding, colonization, and action scheduling).

**Inputs** (read-only):

- `state/spaceinvasion.db` — the bot's telemetry, always read through a temporary copy so the live database is never opened;
- `spaceinvasion.core.game_data` — the wiki catalog of costs, build times, production and requirements;
- `notebooks/metadata/*.json` — static descriptions and planner settings that are not part of the catalog;
- `notebooks/metadata/config_optimizer_result.json` — the empire configuration that `config_optimizer.ipynb` saved, the destination toward which Section 11 plans.

**Layout**: Sections 0–1 set up packages, imports, helpers and static metadata. Sections 2–5 measure the attacks. Section 6 loads every colony, adds generic planets up to the planet limit, and builds the game-parameter tables for them. Section 7 builds the technology requirement graph, Section 8 tabulates the planet sizes and states, and Section 9 collects the highest research levels known in the universe. Section 10 prints everything that Sections 1 to 9 generated. Section 11 searches for the `W`-week action path toward the saved configuration that maximizes weekly-discounted resource production (with the income of the median attack) and prints its own data.

**Output**: the last cell saves the actions of those `W` weeks, in the order to take them and with a header listing the max level reached of each factor, as `notebooks/metadata/full_planner_path.txt`. It is the only file that this notebook writes.

The search for the empire configuration that builds the most Imperial starbases per month has its own notebook, `config_optimizer.ipynb`. It carries copies of the helpers it needs from this notebook, and a unit test fails when a copy differs from its original.

## 0. Package Installation

Installs the third-party packages this notebook imports, directly or through the `spaceinvasion` package: `numpy`, `pandas` and `beautifulsoup4`. The version ranges are the ones in `requirements.txt`. When the packages are already installed, the cell changes nothing.

In [48]:
!pip install -q "numpy>=1.26,<3" "pandas>=2.2,<4" "beautifulsoup4>=4.12,<5"

## 0.1 Imports

Every import the notebook uses is in this cell. The loop before the `spaceinvasion` imports puts the repository root on `sys.path`, so the notebook also runs when it is opened from the `notebooks/` folder.

In [49]:
from collections import deque
from collections.abc import Iterable, Iterator
import contextlib
import dataclasses
import datetime
import heapq
import json
import math
from pathlib import Path
import shutil
import sqlite3
import sys
import tempfile
from typing import Any

import numpy as np
import pandas as pd

# Make the repository importable when the notebook is opened from `notebooks/`.
for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "spaceinvasion" / "__init__.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from spaceinvasion.combat.fleet import DEFENSE_SLUG_BY_KEY, SHIP_SLUG_BY_KEY, _parse_coords_distance
from spaceinvasion.combat.ranking import POINTS_PER_RESOURCE_DIVISOR
from spaceinvasion.core import game_data
from spaceinvasion.core.logging_utils import redact_sensitive
from spaceinvasion.core.state import EnergyBalance, PlanetState, Resources
from spaceinvasion.db.storage import GameDatabase

## 1. Utils

Helpers shared by the sections below, grouped by purpose into one cell each (1.1–1.5 and 1.7). Cell 1.6 loads the static metadata from JSON files.

### 1.1 Workspace Paths & Shared Constants

Locates the repository and the database to read. `ACTIVE_DATA_ROOT` is the working directory when it contains its own `state/spaceinvasion.db` (isolated test fixtures); otherwise it is the repository root.

In [50]:
def resolve_repo_root() -> Path:
    """Return the directory that contains the `spaceinvasion` package, searching outward from the working directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "spaceinvasion" / "__init__.py").is_file():
            return candidate
    for entry in sys.path:
        if entry and (Path(entry).resolve() / "spaceinvasion" / "__init__.py").is_file():
            return Path(entry).resolve()
    for candidate in Path(game_data.__file__).resolve().parents:
        if (candidate / "spaceinvasion" / "__init__.py").is_file():
            return candidate
    raise RuntimeError("Cannot locate the repository root: no directory contains spaceinvasion/__init__.py")


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer the working directory when it has its own `state/spaceinvasion.db`; otherwise use the repository."""
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "state" / "spaceinvasion.db").is_file() else repo_root


REPO_ROOT: Path = resolve_repo_root()
ACTIVE_DATA_ROOT: Path = resolve_active_data_root(REPO_ROOT)
LIVE_DB_PATH: Path = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
METADATA_DIR: Path = REPO_ROOT / "notebooks" / "metadata"
RESOURCE_KEYS: tuple[str, ...] = ("pig_iron", "metal", "kryptonite", "spice")

print(f"REPO_ROOT        = {REPO_ROOT}")
print(f"ACTIVE_DATA_ROOT = {ACTIVE_DATA_ROOT}")
print(f"METADATA_DIR     = {METADATA_DIR}")

REPO_ROOT        = /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer
ACTIVE_DATA_ROOT = /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer
METADATA_DIR     = /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/notebooks/metadata


### 1.2 Formatting, Statistics & DataFrame Helpers

Display options and three small helpers: `fmt_duration` turns seconds into text, `compute_mean_and_std` returns the mean with the sample and population standard deviations (`numpy`), and `to_float_columns` gives numeric columns a float type even when a table is empty.

In [51]:
pd.set_option("display.max_columns", 28)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 64)


def fmt_duration(seconds: float) -> str:
    """Format seconds as `45s`, `3m 07s`, `5h 35m`, or `12d 04h`."""
    if not math.isfinite(seconds) or seconds <= 0.0:
        return "0s"
    sec_i = int(round(seconds))
    if sec_i < 60:
        return f"{sec_i}s"
    mins, rem_s = divmod(sec_i, 60)
    if mins < 60:
        return f"{mins}m {rem_s:02d}s"
    hrs, rem_m = divmod(mins, 60)
    if hrs < 48:
        return f"{hrs}h {rem_m:02d}m"
    days, rem_h = divmod(hrs, 24)
    return f"{days}d {rem_h:02d}h"


def compute_mean_and_std(values: Any) -> tuple[float, float, float]:
    """Return `(mean, sample std with ddof=1, population std with ddof=0)`; non-finite values are ignored."""
    arr = np.asarray(values, dtype=np.float64).ravel()
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return 0.0, 0.0, 0.0
    mean_val = float(np.mean(arr))
    sample_std = float(np.std(arr, ddof=1)) if arr.size >= 2 else 0.0
    pop_std = float(np.std(arr, ddof=0))
    return mean_val, sample_std, pop_std


def to_float_columns(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Convert `columns` to float64 (missing or non-numeric values become 0.0), so empty tables are typed too."""
    for column in columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0.0).astype(np.float64)
    return frame

### 1.3 Game Catalog Helpers & Formulas

Helpers that read `game_data` in a fixed order (`research_catalog`, `unit_keys`), normalize costs, format requirements, and repair the build times that the wiki truncates at very high levels (`build_monotonic_base_times`). The game's time formulas are here too:

- buildings: `base_time / ((b7 + 1) * 2^b17 * speed)`
- research: `base_time / ((b8 + 1) * speed)`
- ships and defenses: `base_time / (max(1, b9) * 2^b17 * speed)`

where `b7` is the Development center, `b8` the Research lab, `b9` the Weapons factory, `b17` the Microsystem accelerator and `speed` the universe speed factor.

`warehouse_capacity` is the storage formula `100,000 * 2^level`, and `solar_cell_energy_per_unit` reads the catalog's Solar cell temperature table.

In [52]:
def research_catalog() -> list[tuple[str, dict[str, Any]]]:
    """Researches as `(key, info)` in numeric key order (`r1`, `r2`, ...); wiki entries without an id are left out."""
    return sorted(game_data.RESEARCH_BY_ID.items(), key=lambda entry: int(entry[0][1:]))


def unit_keys(slug_by_key: dict[str, str]) -> list[str]:
    """Ship or defense keys (`s1`, `d1`, ...) in numeric order."""
    return sorted(slug_by_key, key=lambda key: int(key[1:]))


def full_cost(raw_cost: Any) -> dict[str, Any]:
    """Copy a cost mapping, adding a zero for any of the four resources it leaves out."""
    cost = dict(raw_cost or {})
    for resource in RESOURCE_KEYS:
        cost.setdefault(resource, 0)
    return cost


def cost_points(cost: dict[str, Any]) -> float:
    """Ranking points earned by spending `cost`: one point per 1,000 resources."""
    return float(sum(int(cost[resource]) for resource in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)


def format_prerequisites(requirements: list[dict[str, Any]] | tuple[dict[str, Any], ...]) -> str:
    """Format tech-tree requirements as `r2 L1 (Energy technology), b7 L3 (Development center)`."""
    if not requirements:
        return "None"
    parts: list[str] = []
    for requirement in requirements:
        category = str(requirement.get("category") or "")
        slug = str(requirement.get("slug") or "")
        level = int(requirement.get("level") or 0)
        if category == "buildings":
            building_id = (game_data.BUILDINGS.get(slug) or {}).get("id")
            code = f"b{building_id}" if building_id is not None else slug
        elif category == "research":
            research_id = (game_data.RESEARCHES.get(slug) or {}).get("id")
            code = str(research_id) if research_id else slug
        else:
            code = slug or str(requirement.get("name") or "?")
        parts.append(f"{code} L{level} ({requirement.get('name', slug)})")
    return ", ".join(parts)


def build_monotonic_base_times(levels_map: dict[int, dict[str, Any]], max_level: int) -> dict[int, float]:
    """Return `{level: base_time_s}`, repairing levels where the wiki's scientific notation lost its exponent.

    At very high levels the scraped time drops to a bare mantissa (for example 1.37 s after 9e13 s). Such a
    level is rebuilt from the previous one, scaled by the same ratio as its Spice-equivalent cost.
    """
    times: dict[int, float] = {}
    previous_time = 0.0
    previous_cost = 0.0
    for level in range(1, max_level + 1):
        row = levels_map.get(level) or {}
        raw_time = float(row.get("base_time_s") or 0.0)
        cost = float(game_data.spice_equivalent(row.get("cost") or {}))
        if level > 1 and previous_time > 0.0 and raw_time < previous_time:
            ratio = (cost / previous_cost) if previous_cost > 0.0 else 1.51
            fixed_time = previous_time * max(1.05, ratio)
        else:
            fixed_time = max(0.0, raw_time)
        times[level] = fixed_time
        previous_time = fixed_time
        previous_cost = cost
    return times


def building_time_divisor(dev_center_level: int, microsystem_level: int, speedup_factor: float) -> float:
    """Divisor of a building's base time: `(b7 + 1) * 2^b17 * speed`, never below 1."""
    return max(1.0, float(dev_center_level + 1) * float(2 ** min(microsystem_level, 40)) * float(speedup_factor))


def research_time_divisor(research_lab_level: int, speedup_factor: float) -> float:
    """Divisor of a research's base time: `(b8 + 1) * speed`, never below 1."""
    return max(1.0, float(research_lab_level + 1) * float(speedup_factor))


def shipyard_time_divisor(weapons_factory_level: int, microsystem_level: int, speedup_factor: float) -> float:
    """Divisor of a ship's or defense's base time: `max(1, b9) * 2^b17 * speed`, never below 1."""
    return max(
        1.0,
        float(max(1, weapons_factory_level)) * float(2 ** min(microsystem_level, 40)) * float(speedup_factor),
    )


def warehouse_capacity(level: int) -> int:
    """Storage capacity of a warehouse at `level`: `100,000 * 2^level` (level 0 is every planet's base capacity)."""
    return int(100_000 * (2 ** max(0, int(level))))


def solar_cell_energy_per_unit(temperature_c: float) -> float:
    """Base energy of one orbital Solar cell (`s9`), interpolated from the catalog's temperature table."""
    energy_by_temperature = game_data.SHIPS[SHIP_SLUG_BY_KEY["s9"]]["solar_cell_energy_by_temp"]
    return float(game_data.interpolate_temp_value(energy_by_temperature, temperature_c))

### 1.4 Read-Only Database Access & JSON Parsing

`open_readonly_db_copy` is the only way this notebook touches SQLite. It copies the database (with its WAL sidecars) into a temporary directory, lets `GameDatabase` bring the copy to the current schema, and yields a query-only connection to the copy. The live file is never opened, locked or modified.

`read_latest_game_state` returns the bot's latest full game state, which the telemetry loader of Section 2 and the roster loader of Section 1.7 both read. `parse_json_object` and `parse_level_map` decode the JSON text columns.

In [53]:
def parse_json_object(raw: Any) -> dict[str, Any]:
    """Return `raw` as a dict when it is a dict or the JSON text of an object; otherwise `{}`."""
    if isinstance(raw, dict):
        return dict(raw)
    if isinstance(raw, str) and raw.strip():
        try:
            parsed = json.loads(raw)
        except ValueError:
            return {}
        return dict(parsed) if isinstance(parsed, dict) else {}
    return {}


def parse_level_map(raw: Any) -> dict[str, int]:
    """Parse a `{key: level or count}` object into non-negative ints, dropping non-numeric values."""
    return {
        str(key): max(0, int(value or 0))
        for key, value in parse_json_object(raw).items()
        if isinstance(value, (int, float))
    }


def has_database(db_path: Path) -> bool:
    """True when `db_path` is a file large enough to hold a SQLite header."""
    return db_path.is_file() and db_path.stat().st_size >= 100


@contextlib.contextmanager
def open_readonly_db_copy(db_path: Path) -> Iterator[sqlite3.Connection]:
    """Yield a query-only connection to a temporary copy of `db_path`, migrated to the current schema.

    The live database is only copied (with its WAL sidecars), never opened, so the notebook cannot lock
    or change it. Opening the copy once through `GameDatabase` applies any pending forward-only
    migrations, which lets the notebook's SQL rely on the current schema.
    """
    with tempfile.TemporaryDirectory(prefix="si_fp_ro_") as tmp_dir:
        tmp_db = Path(tmp_dir) / db_path.name
        shutil.copy2(db_path, tmp_db)
        tmp_db.chmod(0o600)
        for suffix in ("-wal", "-shm", "-journal"):
            sidecar = Path(f"{db_path}{suffix}")
            if sidecar.is_file():
                tmp_sidecar = Path(f"{tmp_db}{suffix}")
                shutil.copy2(sidecar, tmp_sidecar)
                tmp_sidecar.chmod(0o600)
        GameDatabase(tmp_db, backup_on_upgrade=False).close()
        with contextlib.closing(sqlite3.connect(tmp_db)) as conn:
            conn.row_factory = sqlite3.Row
            conn.execute("PRAGMA query_only = ON")
            yield conn


SQL_LATEST_GAME_STATE_META = """
SELECT value
FROM meta
WHERE key = 'latest_game_state'
LIMIT 1
"""


def read_latest_game_state(conn: sqlite3.Connection) -> dict[str, Any]:
    """Return the bot's most recent full game state (`meta['latest_game_state']`), or `{}` when it is absent."""
    row = conn.execute(SQL_LATEST_GAME_STATE_META).fetchone()
    return parse_json_object(row[0]) if row and row[0] else {}

### 1.5 Full-Data Printing

`print_section_data` prints every row, column and element of the objects it is given, under a header with the date and time of the printout. Section 10 uses it to print everything that Sections 1 to 9 generate. Floats of 1 or more are never switched to scientific notation, and all text passes through `redact_sensitive`.

In [54]:
def format_full_float(value: float) -> str:
    """Format a float for full printouts without scientific notation for large values.

    `|x| >= 1` keeps its shortest exact decimal text, capped at 6 decimals; `|x| < 1` keeps 6 significant digits.
    """
    val = float(value)
    if not math.isfinite(val):
        return str(val)
    if val == 0.0:
        return "0"
    magnitude = abs(val)
    if magnitude < 1.0:
        return f"{val:.6g}"
    if magnitude >= 1e16:
        return f"{val:.0f}"
    whole, _, frac = repr(val).partition(".")
    if len(frac) > 6:
        return f"{val:.6f}".rstrip("0").rstrip(".")
    return whole if frac in ("", "0") else f"{whole}.{frac}"


def _float_column_formatter(column: pd.Series) -> Any:
    """Return one formatter for a float column so that its decimal points line up."""
    texts = [format_full_float(v) for v in column.to_numpy(dtype=np.float64) if math.isfinite(v)]
    if any("e" in text for text in texts):
        return format_full_float
    decimals = max((len(text.partition(".")[2]) for text in texts), default=0)
    return lambda v: f"{float(v) + 0.0:.{decimals}f}"


def _json_default(obj: Any) -> Any:
    """Convert numpy values, dataclasses, and sets for `json.dumps`; fall back to `str`."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return dataclasses.asdict(obj)
    if isinstance(obj, (set, frozenset)):
        return sorted(obj, key=str)
    return str(obj)


def _compact_json(value: Any) -> str:
    """Single-line JSON text for one value (falls back to `repr` for unserializable objects)."""
    try:
        return json.dumps(value, default=_json_default)
    except (TypeError, ValueError):
        return repr(value)


def describe_data_object(obj: Any) -> str:
    """One-line type and size summary of a generated object."""
    if isinstance(obj, pd.DataFrame):
        return f"pandas.DataFrame, shape=({obj.shape[0]}, {obj.shape[1]})"
    if isinstance(obj, np.ndarray):
        return f"numpy.ndarray, shape={obj.shape}, dtype={obj.dtype}"
    if isinstance(obj, (dict, list, tuple, set, frozenset)):
        return f"{type(obj).__name__}, len={len(obj)}"
    return type(obj).__name__


def render_data_item(name: str, obj: Any) -> str:
    """Render one named object in full: every row, column, element, and cell text, with no `...` truncation."""
    head = f"[{name}] {describe_data_object(obj)}"
    if isinstance(obj, pd.DataFrame):
        if obj.empty:
            return f"{head}\n(no rows) columns: {list(obj.columns)}"
        formatters: dict[Any, Any] = {}
        if obj.columns.is_unique:
            formatters = {
                col: _float_column_formatter(obj[col])
                for col in obj.columns
                if pd.api.types.is_float_dtype(obj[col])
            }
        show_index = not isinstance(obj.index, pd.RangeIndex)
        if show_index and not isinstance(obj.index, pd.MultiIndex) and pd.api.types.is_float_dtype(obj.index):
            formatters["__index__"] = format_full_float
        table = obj.to_string(
            index=show_index,
            formatters=formatters or None,
            float_format=format_full_float,
        )
        return f"{head}\n{table}"
    if isinstance(obj, np.ndarray):
        body = np.array2string(
            obj,
            threshold=obj.size + 1,
            max_line_width=160,
            formatter={"float_kind": format_full_float},
        )
        return f"{head}\n{body}"
    if isinstance(obj, dict):
        if not obj:
            return f"{head}\n{{}}"
        if all(isinstance(v, pd.DataFrame) for v in obj.values()):
            note = "registry of DataFrames; each table is printed in full by its own section"
            rows = [f"  {key}: shape=({tbl.shape[0]}, {tbl.shape[1]})" for key, tbl in obj.items()]
            return f"{head} ({note})\n" + "\n".join(rows)
        rows = [f"  {json.dumps(str(key))}: {_compact_json(value)}" for key, value in obj.items()]
        return f"{head}\n{{\n" + ",\n".join(rows) + "\n}"
    if isinstance(obj, (list, tuple)):
        if all(v is None or isinstance(v, (str, int, float, bool)) for v in obj):
            return f"{head}\n{_compact_json(list(obj))}"
        return f"{head} (one JSON value per line)\n" + "\n".join(_compact_json(v) for v in obj)
    return f"{head} = {obj!r}"


def print_section_data(section_title: str, items: list[tuple[str, Any]]) -> None:
    """Print everything one section generated, in full, stamped with the date and time of the printout.

    Each item is `(name, obj)`. The whole text passes through `redact_sensitive` before printing, so a
    session token can never reach a notebook output.
    """
    printed_at = datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
    rule = "=" * 100
    header = f"{rule}\n{section_title}\nPrinted: {printed_at} | {len(items)} item(s)\n{rule}"
    blocks = [render_data_item(name, obj) for name, obj in items]
    print(redact_sensitive(header + "\n" + ("\n" + "-" * 100 + "\n").join(blocks)))

### 1.6 Static Metadata (`notebooks/metadata/*.json`)

Static data is kept in JSON files instead of code cells:

| File | Content |
|---|---|
| `buildings.json` | category, nominal cost scale factor and primary effect of `b1`–`b21` |
| `technologies.json` | domain, effect, per-level bonus and scope of `r1`–`r15` |
| `ships.json` | role of each ship |
| `defenses.json` | role of each defense and whether it can dig in |
| `resources.json` | mine and warehouse building of each resource, Mining technology flag, notes |
| `planner_settings.json` | planner settings and fallback assumptions: table level caps, empire size, cold-start colony, flight-time fallback |

The per-entity files repeat each entity's name. Loading stops with an error when a file misses an entity, has an unknown one, lacks a field, or names an entity differently from `game_data`, so a description can never end up on the wrong entity.

`technology_multiplier` turns a research level into its multiplier, using the `per_level_bonus_pct` of `technologies.json`.

In [55]:
def load_metadata(file_name: str) -> dict[str, Any]:
    """Read one JSON document from `notebooks/metadata/`."""
    return json.loads((METADATA_DIR / file_name).read_text(encoding="utf-8"))


def load_keyed_metadata(
    file_name: str,
    section: str,
    expected_keys: Iterable[str],
    required_fields: tuple[str, ...],
) -> dict[str, dict[str, Any]]:
    """Load `{key: entry}` metadata; it must cover exactly `expected_keys`, each entry with every required field."""
    entries = load_metadata(file_name)[section]
    expected = set(expected_keys)
    if set(entries) != expected:
        raise ValueError(
            f"{file_name}: missing {sorted(expected - set(entries))}, unknown {sorted(set(entries) - expected)}"
        )
    for key, entry in entries.items():
        absent = [field for field in required_fields if field not in entry]
        if absent:
            raise ValueError(f"{file_name}: {key} lacks {absent}")
    return entries


def load_catalog_metadata(
    file_name: str,
    section: str,
    catalog_names: dict[str, str],
    required_fields: tuple[str, ...],
) -> dict[str, dict[str, Any]]:
    """Load per-entity metadata and check each entry's `name` against the game catalog (`{key: name}`)."""
    entries = load_keyed_metadata(file_name, section, catalog_names, ("name", *required_fields))
    for key, entry in entries.items():
        if entry["name"] != catalog_names[key]:
            raise ValueError(
                f"{file_name}: {key} is named {entry['name']!r}, but game_data calls it {catalog_names[key]!r}"
            )
    return entries


BUILDING_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "buildings.json",
    "buildings",
    {f"b{building_id}": str(info["name"]) for building_id, info in game_data.BUILDING_BY_ID.items()},
    ("category", "cost_scale_factor", "primary_effect"),
)
TECHNOLOGY_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "technologies.json",
    "technologies",
    {key: str(info["name"]) for key, info in research_catalog()},
    ("domain", "effect_metric", "per_level_bonus_pct", "applies_to", "strategic_effect"),
)
TECHNOLOGY_COST_SCALE_FACTOR: float = float(load_metadata("technologies.json")["cost_scale_factor"])
SHIP_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "ships.json",
    "ships",
    {key: str(game_data.SHIPS[slug]["name"]) for key, slug in SHIP_SLUG_BY_KEY.items()},
    ("role",),
)
DEFENSE_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "defenses.json",
    "defenses",
    {key: str(game_data.DEFENSES[slug]["name"]) for key, slug in DEFENSE_SLUG_BY_KEY.items()},
    ("role", "can_dig_in"),
)
RESOURCE_METADATA: dict[str, dict[str, Any]] = load_keyed_metadata(
    "resources.json",
    "resources",
    RESOURCE_KEYS,
    ("mine_building_key", "warehouse_building_key", "boosted_by_mining_tech", "notes"),
)
PLANNER_SETTINGS: dict[str, Any] = load_metadata("planner_settings.json")


def technology_multiplier(research_key: str, level: int) -> float:
    """Multiplier a research gives at `level`: `1 + per_level_bonus_pct / 100 * level` (1.0 for an unknown key)."""
    bonus_per_level = float(TECHNOLOGY_METADATA.get(research_key, {}).get("per_level_bonus_pct", 0.0)) / 100.0
    return 1.0 + bonus_per_level * float(level)


print(
    f"Static metadata loaded from {METADATA_DIR}: {len(BUILDING_METADATA)} buildings, "
    f"{len(TECHNOLOGY_METADATA)} technologies, {len(SHIP_METADATA)} ships, {len(DEFENSE_METADATA)} defenses, "
    f"{len(RESOURCE_METADATA)} resources, and the planner settings"
)

Static metadata loaded from /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/notebooks/metadata: 21 buildings, 15 technologies, 15 ships, 11 defenses, 4 resources, and the planner settings


### 1.7 Planet Roster Helpers

`load_live_planet_states` reads every owned planet from the temporary database copy. It prefers the bot's latest full game state, which records the real warehouse capacities. Without one, it takes the newest `planet_snapshots` row of each planet and derives the capacities from the warehouse levels.

`build_padded_empire_planets` appends generic planets to the owned ones until the empire has the given number of normal planets. Asteroids are kept but do not count toward that limit. A generic planet (`empty_colony_slot`) is an empty colony slot: a planet with the given number of free fields, no resources, level-0 warehouses and nothing built. Its `planet_id` is negative (`-1`, `-2`, …) and its coordinates are `EMPTY:01`, `EMPTY:02`, ….

Sections 6 and 8 work on this roster.

In [56]:
SQL_LATEST_PLANETS_PER_ID = """
SELECT
    p.planet_id,
    p.coords,
    p.name,
    p.planet_type,
    p.temperature,
    p.fields_used,
    p.fields_max,
    p.pig_iron,
    p.metal,
    p.kryptonite,
    p.spice,
    p.energy_produced,
    p.energy_used,
    p.buildings_json,
    p.ships_json,
    p.defense_json,
    p.economy_rates_json
FROM planet_snapshots p
INNER JOIN (
    SELECT planet_id, MAX(id) AS max_id
    FROM planet_snapshots
    GROUP BY planet_id
) latest ON latest.max_id = p.id
ORDER BY p.planet_id ASC
"""


def snapshot_row_to_planet_fields(row: sqlite3.Row) -> dict[str, Any]:
    """Reshape a `planet_snapshots` row into the nested layout the saved game state uses for a planet."""
    return {
        "planet_id": row["planet_id"],
        "name": row["name"],
        "coords": row["coords"],
        "planet_type": row["planet_type"],
        "temperature": row["temperature"],
        "fields_used": row["fields_used"],
        "fields_max": row["fields_max"],
        "resources": {resource: row[resource] for resource in RESOURCE_KEYS},
        "energy": {"produced": row["energy_produced"], "used": row["energy_used"]},
        "buildings": row["buildings_json"],
        "ships": row["ships_json"],
        "defense": row["defense_json"],
        "economy_rates": row["economy_rates_json"],
    }


def planet_state_from_fields(fields: dict[str, Any]) -> PlanetState:
    """Build a typed `PlanetState` from one planet's saved fields.

    Gaps are filled from the building levels: used fields default to the sum of all levels, and a missing
    warehouse capacity to `warehouse_capacity(level)`. A missing size defaults to `fallback_planet_fields`.
    """
    buildings = parse_level_map(fields.get("buildings"))
    resources = parse_json_object(fields.get("resources"))
    storage = parse_json_object(fields.get("storage"))
    energy = parse_json_object(fields.get("energy"))
    fields_used = int(fields.get("fields_used") or sum(buildings.values()))
    fallback_fields = int(PLANNER_SETTINGS["empire"]["fallback_planet_fields"])
    return PlanetState(
        planet_id=int(fields.get("planet_id") or 0),
        name=str(fields.get("name") or "Colony"),
        coords=str(fields.get("coords") or "00:000:00"),
        planet_type=str(fields.get("planet_type") or "normal"),
        temperature=int(fields.get("temperature") or 0),
        fields_used=fields_used,
        fields_max=int(fields.get("fields_max") or max(fallback_fields, fields_used)),
        resources=Resources(**{resource: int(resources.get(resource) or 0) for resource in RESOURCE_KEYS}),
        storage=Resources(
            **{
                resource: int(
                    storage.get(resource)
                    or warehouse_capacity(buildings.get(RESOURCE_METADATA[resource]["warehouse_building_key"], 0))
                )
                for resource in RESOURCE_KEYS
            }
        ),
        energy=EnergyBalance(produced=int(energy.get("produced") or 0), used=int(energy.get("used") or 0)),
        buildings=buildings,
        ships=parse_level_map(fields.get("ships")),
        defense=parse_level_map(fields.get("defense")),
        economy_rates=parse_level_map(fields.get("economy_rates")),
    )


def load_live_planet_states(db_path: Path) -> list[PlanetState]:
    """Load every owned planet from a temporary copy of `db_path`.

    The bot's latest full game state is preferred because it records the real warehouse capacities. Without
    it, the newest `planet_snapshots` row of each planet is used and capacities come from the warehouse levels.
    """
    if not has_database(db_path):
        return []
    with open_readonly_db_copy(db_path) as conn:
        saved_planets = read_latest_game_state(conn).get("planets") or []
        if isinstance(saved_planets, list) and saved_planets:
            return [planet_state_from_fields(planet) for planet in saved_planets if isinstance(planet, dict)]
        return [
            planet_state_from_fields(snapshot_row_to_planet_fields(row))
            for row in conn.execute(SQL_LATEST_PLANETS_PER_ID)
        ]


def empty_colony_slot(slot_number: int, fields: int, temperature_c: int) -> PlanetState:
    """A generic planet that is not colonized yet: `fields` free fields, base warehouses, nothing built."""
    base_storage = warehouse_capacity(0)
    return PlanetState(
        planet_id=-slot_number,
        name=f"Empty Colony Slot {slot_number}",
        coords=f"EMPTY:{slot_number:02d}",
        planet_type="normal",
        temperature=int(temperature_c),
        fields_used=0,
        fields_max=int(fields),
        resources=Resources(0, 0, 0, 0),
        storage=Resources(base_storage, base_storage, base_storage, base_storage),
        energy=EnergyBalance(0, 0),
        buildings={f"b{building_id}": 0 for building_id in sorted(game_data.BUILDING_BY_ID)},
        ships={},
        defense={},
        economy_rates={},
    )


def build_padded_empire_planets(
    live_planets: list[PlanetState],
    *,
    max_planets: int,
    empty_planet_size: int,
    empty_temp_c: int,
) -> list[PlanetState]:
    """Return the owned planets followed by generic planets (empty colony slots).

    Slots are added until the empire has `max_planets` normal planets; asteroids and other planet types are
    kept but do not count toward that limit.
    """
    owned_normal_planets = sum(1 for planet in live_planets if planet.planet_type == "normal")
    return [
        *live_planets,
        *(
            empty_colony_slot(slot_number, empty_planet_size, empty_temp_c)
            for slot_number in range(1, max(0, int(max_planets) - owned_normal_planets) + 1)
        ),
    ]

## 2. Attack Telemetry from SQLite

Loads the reference colony and the attack history from `state/spaceinvasion.db`. The tables and their relationships are documented in [docs/database-mer.md](../docs/database-mer.md). Every query runs on the temporary copy opened by `open_readonly_db_copy`.

### 2.1 SQL Queries & Table Columns

| Query | Tables | Returns |
|---|---|---|
| `SQL_LATEST_COLONY_SNAPSHOT` | `planet_snapshots` | coordinates, building and research levels of the most recently snapshotted planet |
| `SQL_BATTLE_REPORTS_MER_JOIN` | `battle_reports`, `game_messages`, `spy_targets` | loot, losses and rubble of every battle with the target's status; `spy_targets` rows for own planets and merged targets are left out |
| `SQL_RAID_DISPATCH_ORIGIN_JOIN` | `raid_history`, `action_history` | every raid dispatch with the colony that launched it: the executed `ATTACK` action within 5 s of the dispatch, or the query parameter when there is none |
| `SQL_RECYCLE_HARVESTS` | `recycle_reports` | recycler harvests |
| `SQL_ACTIVE_BASH_BLOCKS` | `bash_blocks` | number of targets still bash-blocked at the given time |

The `*_COLUMNS` tuples list the columns of `attacks_df` and `recycles_df` and which of them are numeric.

In [57]:
SQL_LATEST_COLONY_SNAPSHOT = """
SELECT coords, buildings_json, researches_json
FROM planet_snapshots
ORDER BY ts DESC, id DESC
LIMIT 1
"""

SQL_BATTLE_REPORTS_MER_JOIN = """
SELECT
    br.id,
    br.ts,
    br.msg_uid,
    br.battle_id,
    br.msg_date_text,
    br.target_coords,
    br.outcome,
    br.rounds,
    br.pig_iron,
    br.metal,
    br.kryptonite,
    br.spice,
    br.loot_spice_eq,
    br.attacker_lost_spice,
    br.defender_lost_spice,
    br.debris_spice_eq,
    gm.ingested_ts AS msg_ingested_ts,
    gm.deleted_in_game,
    COALESCE(st.status_color, 'unknown') AS target_status_color,
    COALESCE(st.is_inactive, 0) AS target_is_inactive,
    COALESCE(st.is_spacia, 0) AS target_is_spacia,
    COALESCE(st.bash_hits, 0) AS target_bash_hits
FROM battle_reports br
LEFT JOIN game_messages gm ON gm.msg_uid = br.msg_uid
LEFT JOIN spy_targets st
       ON st.coords = br.target_coords
      AND st.is_own_planet = 0
      AND st.merged_into = ''
ORDER BY br.msg_date_text ASC, br.id ASC
"""

# The `BETWEEN` range (instead of `ABS(ah.ts - rh.attacked_ts) <= 5.0`) lets SQLite use `idx_action_history_ts`.
SQL_RAID_DISPATCH_ORIGIN_JOIN = """
SELECT
    rh.id,
    rh.target_coords,
    rh.attacked_ts,
    rh.is_exempt,
    rh.outcome,
    rh.loot_json,
    COALESCE(
        (
            SELECT ah.planet_coords
            FROM action_history ah
            WHERE ah.ts BETWEEN rh.attacked_ts - 5.0 AND rh.attacked_ts + 5.0
              AND ah.kind = 'ATTACK'
              AND ah.executed = 1
              AND ah.dry_run = 0
              AND ah.planet_coords IS NOT NULL
              AND ah.planet_coords != ''
            ORDER BY ABS(ah.ts - rh.attacked_ts) ASC
            LIMIT 1
        ),
        ?
    ) AS origin_coords
FROM raid_history rh
ORDER BY rh.attacked_ts ASC, rh.id ASC
"""

SQL_RECYCLE_HARVESTS = """
SELECT
    coords,
    recycler_count,
    harvested_pig_iron,
    harvested_metal,
    harvested_kryptonite,
    harvested_spice,
    harvested_spice_eq,
    ts
FROM recycle_reports
ORDER BY ts ASC, id ASC
"""

SQL_ACTIVE_BASH_BLOCKS = """
SELECT COUNT(DISTINCT target_coords)
FROM bash_blocks
WHERE blocked_until_ts > ?
"""

ATTACK_DF_COLUMNS: tuple[str, ...] = (
    "event_ts",
    "origin_coords",
    "target_coords",
    "target_class",
    "won",
    "outcome",
    "pig_iron",
    "metal",
    "kryptonite",
    "spice",
    "spice_eq",
    "attacker_lost_spice",
    "net_spice_eq",
    "debris_spice_eq",
    "one_way_sec",
    "round_trip_sec",
    "time_source",
)
ATTACK_FLOAT_COLUMNS: tuple[str, ...] = (
    "event_ts",
    "pig_iron",
    "metal",
    "kryptonite",
    "spice",
    "spice_eq",
    "attacker_lost_spice",
    "net_spice_eq",
    "debris_spice_eq",
    "one_way_sec",
    "round_trip_sec",
)

RECYCLE_DF_COLUMNS: tuple[str, ...] = (
    "coords",
    "recycler_count",
    "harvested_pig_iron",
    "harvested_metal",
    "harvested_kryptonite",
    "harvested_spice",
    "harvested_spice_eq",
    "ts",
)
RECYCLE_FLOAT_COLUMNS: tuple[str, ...] = tuple(column for column in RECYCLE_DF_COLUMNS if column != "coords")

### 2.2 Telemetry Loader

`load_attack_telemetry` returns the reference colony and one record per attack. Dispatches and battle reports are timed differently: a raid dispatch is stored in epoch seconds, a battle report carries the server's wall-clock text without a time zone. The loader reconciles them in three steps:

1. `infer_server_tz_offset_sec` finds the whole-hour offset between the two clocks (the median over all report/dispatch pairs on the same target).
2. `attack_records_from_battles` pairs each battle report with its dispatch. The flight time is the observed gap between dispatch and impact (`observed_dispatch_pair`). A report without a dispatch gets the median observed seconds per square root of distance, applied to its own distance (`calibrated_distance`). When no pair was observed at all, the rough estimate `estimate_physics_one_way_seconds` is used (`physics_formula`).
3. `attack_records_from_raids` covers a database that has dispatches but no battle report yet: one record per dispatch, timed by the same rough estimate.

In [58]:
def estimate_physics_one_way_seconds(origin_coords: str, target_coords: str) -> float:
    """Rough OGame-style one-way flight time in seconds: `10 + 3500 / factor * sqrt(10 * distance / speed)`.

    Only a fallback for attacks without an observed flight time. `speed` and `factor` are the assumptions
    under `flight_time_fallback` in `planner_settings.json`; they are not measured from the game.
    """
    assumptions = PLANNER_SETTINGS["flight_time_fallback"]
    distance = max(1.0, float(_parse_coords_distance(origin_coords, target_coords)))
    speed = max(100.0, float(assumptions["min_ship_speed"]))
    factor = min(100.0, max(1.0, float(assumptions["fleet_speed_factor"])))
    return float(round(10.0 + (3500.0 / factor) * math.sqrt((10.0 * distance) / speed)))


def parse_server_clock(text: Any) -> float | None:
    """Read a battle report's `YYYY-MM-DD HH:MM:SS` server clock text as if it were UTC (`None` when malformed)."""
    try:
        parsed = datetime.datetime.strptime(str(text or "").strip(), "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None
    return parsed.replace(tzinfo=datetime.timezone.utc).timestamp()


def infer_server_tz_offset_sec(
    battle_rows: list[dict[str, Any]],
    server_clock_ts: list[float | None],
    raid_rows: list[dict[str, Any]],
) -> float:
    """Estimate the offset, in seconds, to add to a battle report's server clock to get epoch time.

    Every report/dispatch pair on the same target votes for the whole-hour offset (at most 14 h) that puts
    the impact 30 s to 1 h after the dispatch. The median vote wins; without votes the offset is 0.
    """
    votes: list[float] = []
    for battle, clock_ts in zip(battle_rows, server_clock_ts):
        if clock_ts is None:
            continue
        for raid in raid_rows:
            if str(raid.get("target_coords") or "") != str(battle.get("target_coords") or ""):
                continue
            dispatch_ts = float(raid.get("attacked_ts") or 0.0)
            offset_hours = round((dispatch_ts - clock_ts) / 3600.0)
            if abs(offset_hours) <= 14 and 30.0 <= (clock_ts + offset_hours * 3600.0 - dispatch_ts) <= 3600.0:
                votes.append(offset_hours * 3600.0)
    return float(np.median(np.asarray(votes, dtype=np.float64))) if votes else 0.0


def attack_record(
    *,
    event_ts: float,
    origin_coords: str,
    target_coords: str,
    target_class: str,
    won: bool,
    outcome: str,
    loot: dict[str, float],
    spice_eq: float,
    attacker_lost_spice: float,
    debris_spice_eq: float,
    one_way_sec: float,
    time_source: str,
) -> dict[str, Any]:
    """Assemble one `attacks_df` row (`loot` holds the four resources; net yield and round trip are derived)."""
    return {
        "event_ts": event_ts,
        "origin_coords": origin_coords,
        "target_coords": target_coords,
        "target_class": target_class,
        "won": won,
        "outcome": outcome,
        **loot,
        "spice_eq": spice_eq,
        "attacker_lost_spice": attacker_lost_spice,
        "net_spice_eq": spice_eq - attacker_lost_spice,
        "debris_spice_eq": debris_spice_eq,
        "one_way_sec": float(one_way_sec),
        "round_trip_sec": float(2.0 * one_way_sec),
        "time_source": time_source,
    }


def attack_records_from_battles(
    battle_rows: list[dict[str, Any]],
    server_clock_ts: list[float | None],
    raid_rows: list[dict[str, Any]],
    default_origin: str,
    tz_offset_sec: float,
) -> list[dict[str, Any]]:
    """Build one record per battle report, timing each attack by pairing the report with its raid dispatch."""
    used_raids: set[int] = set()
    seconds_per_sqrt_distance: list[float] = []
    paired: list[tuple[dict[str, Any], float, float | None, str]] = []
    for battle, clock_ts in zip(battle_rows, server_clock_ts):
        target = str(battle.get("target_coords") or "")
        impact_ts = (clock_ts + tz_offset_sec) if clock_ts is not None else float(battle.get("ts") or 0.0)
        flight_sec: float | None = None
        origin = default_origin
        paired_raid: int | None = None
        for raid_index, raid in enumerate(raid_rows):
            if raid_index in used_raids or str(raid.get("target_coords") or "") != target:
                continue
            gap = impact_ts - float(raid.get("attacked_ts") or 0.0)
            # Keep the latest unused dispatch that left 30 s to 2 h before the impact.
            if 30.0 <= gap <= 7200.0 and (flight_sec is None or gap < flight_sec):
                flight_sec = gap
                origin = str(raid.get("origin_coords") or default_origin)
                paired_raid = raid_index
        if paired_raid is not None and flight_sec is not None:
            used_raids.add(paired_raid)
            distance = max(1.0, float(_parse_coords_distance(origin, target)))
            seconds_per_sqrt_distance.append(max(1.0, flight_sec - 10.0) / math.sqrt(distance))
        paired.append((battle, impact_ts, flight_sec, origin))

    calibration = (
        float(np.median(np.asarray(seconds_per_sqrt_distance, dtype=np.float64)))
        if seconds_per_sqrt_distance
        else None
    )
    records: list[dict[str, Any]] = []
    for battle, impact_ts, flight_sec, origin in paired:
        target = str(battle.get("target_coords") or "")
        if flight_sec is not None:
            one_way_sec, time_source = flight_sec, "observed_dispatch_pair"
        elif calibration is not None:
            distance = max(1.0, float(_parse_coords_distance(origin, target)))
            one_way_sec, time_source = 10.0 + calibration * math.sqrt(distance), "calibrated_distance"
        else:
            one_way_sec, time_source = estimate_physics_one_way_seconds(origin, target), "physics_formula"

        outcome = str(battle.get("outcome") or "").strip().lower()
        won = outcome in ("win", "victory")
        if battle.get("target_is_spacia"):
            target_class = "spacia"
        elif battle.get("target_is_inactive") or str(battle.get("target_status_color") or "") == "yellow":
            target_class = "yellow_inactive"
        else:
            target_class = "active_colony"
        records.append(
            attack_record(
                event_ts=impact_ts,
                origin_coords=origin,
                target_coords=target,
                target_class=target_class,
                won=won,
                outcome=outcome or ("win" if won else "loss"),
                loot={resource: float(battle.get(resource) or 0.0) for resource in RESOURCE_KEYS},
                spice_eq=float(battle["loot_spice_eq"]),
                attacker_lost_spice=float(battle.get("attacker_lost_spice") or 0.0),
                debris_spice_eq=float(battle.get("debris_spice_eq") or 0.0),
                one_way_sec=one_way_sec,
                time_source=time_source,
            )
        )
    return records


def attack_records_from_raids(raid_rows: list[dict[str, Any]], default_origin: str) -> list[dict[str, Any]]:
    """Build one record per raid dispatch (used while no battle report exists), timed by the physics estimate."""
    records: list[dict[str, Any]] = []
    for raid in raid_rows:
        target = str(raid.get("target_coords") or "")
        origin = str(raid.get("origin_coords") or default_origin)
        raw_loot = parse_json_object(raid.get("loot_json"))
        loot = {resource: float(raw_loot.get(resource) or 0.0) for resource in RESOURCE_KEYS}
        outcome = str(raid.get("outcome") or "").lower()
        won = "defeat" not in outcome and "loss" not in outcome
        records.append(
            attack_record(
                event_ts=float(raid.get("attacked_ts") or 0.0),
                origin_coords=origin,
                target_coords=target,
                target_class="exempt_farm" if raid.get("is_exempt") else "active_colony",
                won=won,
                outcome="win" if won else "loss",
                loot=loot,
                spice_eq=float(game_data.spice_equivalent(loot)),
                attacker_lost_spice=0.0,
                debris_spice_eq=0.0,
                one_way_sec=estimate_physics_one_way_seconds(origin, target),
                time_source="physics_formula",
            )
        )
    return records


def load_attack_telemetry(db_path: Path) -> dict[str, Any]:
    """Read the origin colony and attack history from a temporary copy of `db_path`; derive one record per attack.

    Without a database, or without a planet snapshot in it, the origin colony and the universe speed fall
    back to `cold_start_colony` in `planner_settings.json`.
    """
    cold_start = PLANNER_SETTINGS["cold_start_colony"]
    origin_coords = str(cold_start["origin_coords"])
    universe_speedup_factor = float(cold_start["speedup_factor"])
    colony_buildings: dict[str, int] = {}
    colony_researches: dict[str, int] = {}
    battle_rows: list[dict[str, Any]] = []
    raid_rows: list[dict[str, Any]] = []
    recycle_rows: list[dict[str, Any]] = []
    active_bash_blocks = 0

    if has_database(db_path):
        with open_readonly_db_copy(db_path) as conn:
            snapshot = conn.execute(SQL_LATEST_COLONY_SNAPSHOT).fetchone()
            if snapshot:
                if snapshot["coords"]:
                    origin_coords = str(snapshot["coords"])
                colony_buildings = parse_level_map(snapshot["buildings_json"])
                colony_researches = parse_level_map(snapshot["researches_json"])

            saved_speed = read_latest_game_state(conn).get("speedup_factor")
            if isinstance(saved_speed, (int, float)) and math.isfinite(float(saved_speed)) and float(saved_speed) > 0.0:
                universe_speedup_factor = float(saved_speed)

            battle_rows = [dict(row) for row in conn.execute(SQL_BATTLE_REPORTS_MER_JOIN)]
            raid_rows = [dict(row) for row in conn.execute(SQL_RAID_DISPATCH_ORIGIN_JOIN, (origin_coords,))]
            recycle_rows = [dict(row) for row in conn.execute(SQL_RECYCLE_HARVESTS)]
            latest_event_ts = max(
                [0.0]
                + [float(raid.get("attacked_ts") or 0.0) for raid in raid_rows]
                + [float(battle.get("ts") or 0.0) for battle in battle_rows]
            )
            active_bash_blocks = int(conn.execute(SQL_ACTIVE_BASH_BLOCKS, (latest_event_ts,)).fetchone()[0])

    if battle_rows:
        server_clock_ts = [parse_server_clock(battle.get("msg_date_text")) for battle in battle_rows]
        tz_offset_sec = infer_server_tz_offset_sec(battle_rows, server_clock_ts, raid_rows)
        records = attack_records_from_battles(battle_rows, server_clock_ts, raid_rows, origin_coords, tz_offset_sec)
    else:
        records = attack_records_from_raids(raid_rows, origin_coords)
    records.sort(key=lambda record: float(record["event_ts"]))

    dispatch_timestamps = sorted(
        float(raid["attacked_ts"])
        for raid in raid_rows
        if isinstance(raid.get("attacked_ts"), (int, float)) and math.isfinite(float(raid["attacked_ts"]))
    )
    if not dispatch_timestamps:
        dispatch_timestamps = [float(record["event_ts"]) for record in records if float(record["event_ts"]) > 0.0]

    fleet_base_level = colony_buildings.get("b10", 0)
    return {
        "records": records,
        "dispatch_timestamps": dispatch_timestamps,
        "origin_coords": origin_coords,
        "colony_buildings": colony_buildings,
        "colony_researches": colony_researches,
        "universe_speedup_factor": universe_speedup_factor,
        "fleet_base_level": fleet_base_level,
        "fleet_slot_capacity": 1 + fleet_base_level,
        "engine_tech_levels": {key: colony_researches.get(key, 0) for key in ("r4", "r5", "r6")},
        "recycle_rows": recycle_rows,
        "active_bash_blocks": active_bash_blocks,
        "source_label": (
            "sqlite:mer_joined(battle_reports+raid_history+spy_targets)" if battle_rows or raid_rows else "cold_start"
        ),
    }

### 2.3 Load Telemetry into DataFrames

Runs the loader and names its results:

- `telemetry_bundle` — everything the loader returned;
- `origin_planet_coords`, `colony_buildings`, `fleet_slot_capacity` (1 + Fleet base level) — the reference colony;
- `colony_researches` — the account's research levels, the same on every planet;
- `attacks_df` — one row per attack; `recycles_df` — one row per recycler harvest;
- `attack_dispatch_ts_arr` — the dispatch times in epoch seconds.

The reference colony is the planet with the newest snapshot. With a single planet that is the home planet; with several planets it is whichever one was snapshotted last. It provides the default origin of an attack, the fleet slots, and the building levels that the graph of Section 7 treats as current. The parameter tables of Section 6 do not use it: they read every colony.

In [59]:
telemetry_bundle: dict[str, Any] = load_attack_telemetry(LIVE_DB_PATH)
telemetry_source: str = telemetry_bundle["source_label"]
origin_planet_coords: str = telemetry_bundle["origin_coords"]
colony_buildings: dict[str, int] = telemetry_bundle["colony_buildings"]
colony_researches: dict[str, int] = telemetry_bundle["colony_researches"]
fleet_slot_capacity: int = int(telemetry_bundle["fleet_slot_capacity"])

attacks_df = to_float_columns(
    pd.DataFrame(telemetry_bundle["records"], columns=list(ATTACK_DF_COLUMNS)),
    ATTACK_FLOAT_COLUMNS,
)
attacks_df["won"] = attacks_df["won"].astype(bool)
recycles_df = to_float_columns(
    pd.DataFrame(telemetry_bundle["recycle_rows"], columns=list(RECYCLE_DF_COLUMNS)),
    RECYCLE_FLOAT_COLUMNS,
)
attack_dispatch_ts_arr: np.ndarray = np.asarray(telemetry_bundle["dispatch_timestamps"], dtype=np.float64)

print(
    pd.DataFrame(
        [
            ("telemetry source", telemetry_source),
            ("origin colony", origin_planet_coords),
            ("universe speed", f"{telemetry_bundle['universe_speedup_factor']:.1f}x"),
            ("fleet slots", f"{fleet_slot_capacity} (Fleet base b10 at level {telemetry_bundle['fleet_base_level']})"),
            (
                "engine research r4 / r5 / r6",
                " / ".join(str(telemetry_bundle["engine_tech_levels"][key]) for key in ("r4", "r5", "r6")),
            ),
            ("attacks_df", f"{len(attacks_df)} attacks on {attacks_df['target_coords'].nunique()} targets"),
            ("targets with an active bash block", telemetry_bundle["active_bash_blocks"]),
            ("recycles_df", f"{len(recycles_df)} recycler harvests"),
        ],
        columns=["property", "value"],
    ).to_string(index=False)
)

                         property                                                      value
                 telemetry source sqlite:mer_joined(battle_reports+raid_history+spy_targets)
                    origin colony                                                  02:380:12
                   universe speed                                                      20.0x
                      fleet slots                              7 (Fleet base b10 at level 6)
     engine research r4 / r5 / r6                                                  8 / 7 / 0
                       attacks_df                                     7 attacks on 2 targets
targets with an active bash block                                                         35
                      recycles_df                                       14 recycler harvests


## 3. Mean & Standard Deviation of Resources Achieved in Attacks

`attack_resource_stats_df` holds, for all attacks and for winning attacks only, the mean, the sample standard deviation (`ddof=1`) and the population standard deviation (`ddof=0`) of:

- the loot per resource (`pig_iron`, `metal`, `kryptonite`, `spice`) and its Spice equivalent `spice_eq` at the Merchant parity `6 : 2.5 : 3 : 1`;
- the net yield `net_spice_eq`: loot minus the attacker's fleet losses;
- the rubble left in orbit, `debris_spice_eq`.

The same numbers are available as dictionaries (`attack_resource_mean`, `attack_resource_std`, `attack_resource_pop_std`, `winning_attack_resource_mean`, `winning_attack_resource_std`). `recycle_spice_eq_mean` and `recycle_spice_eq_std` describe the recycler harvests in `recycles_df`. Everything is computed with `numpy` on the `pandas` tables of Section 2.

In [60]:
METRIC_KEYS: tuple[str, ...] = (*RESOURCE_KEYS, "spice_eq", "net_spice_eq", "debris_spice_eq")


def metric_stats(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Mean, sample std (`ddof=1`) and population std (`ddof=0`) of each metric column; zeros when undefined."""
    values = frame[list(METRIC_KEYS)].to_numpy(dtype=np.float64)
    zeros = np.zeros(len(METRIC_KEYS), dtype=np.float64)
    return {
        "mean": np.mean(values, axis=0) if len(frame) >= 1 else zeros,
        "sample_std": np.std(values, axis=0, ddof=1) if len(frame) >= 2 else zeros,
        "pop_std": np.std(values, axis=0, ddof=0) if len(frame) >= 1 else zeros,
    }


def build_attack_resource_stats(all_attacks: pd.DataFrame, winning_attacks: pd.DataFrame) -> pd.DataFrame:
    """One row per metric with the statistics of all attacks and of winning attacks."""
    all_stats, win_stats = metric_stats(all_attacks), metric_stats(winning_attacks)
    return pd.DataFrame(
        {
            "metric": list(METRIC_KEYS),
            "samples_all": len(all_attacks),
            "mean_all": all_stats["mean"],
            "sample_std_all": all_stats["sample_std"],
            "pop_std_all": all_stats["pop_std"],
            "samples_wins": len(winning_attacks),
            "mean_wins": win_stats["mean"],
            "sample_std_wins": win_stats["sample_std"],
            "pop_std_wins": win_stats["pop_std"],
        }
    )


def stats_column_by_metric(stats: pd.DataFrame, column: str) -> dict[str, float]:
    """`{metric: value}` for one column of an `attack_resource_stats_df` table."""
    return {str(metric): float(value) for metric, value in zip(stats["metric"], stats[column])}


winning_attacks_df = attacks_df.loc[attacks_df["won"]].reset_index(drop=True)
attack_resource_stats_df = build_attack_resource_stats(attacks_df, winning_attacks_df)

attack_resource_mean: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "mean_all")
attack_resource_std: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "sample_std_all")
attack_resource_pop_std: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "pop_std_all")
winning_attack_resource_mean: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "mean_wins")
winning_attack_resource_std: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "sample_std_wins")

recycle_spice_eq_mean, recycle_spice_eq_std, _ = compute_mean_and_std(
    recycles_df["harvested_spice_eq"].to_numpy(dtype=np.float64)
)

print(attack_resource_stats_df.round(2).to_string(index=False))
print(
    f"\nRecycler harvests ({len(recycles_df)}): mean {recycle_spice_eq_mean:,.2f} SpiceEq, "
    f"sample std {recycle_spice_eq_std:,.2f} SpiceEq"
)

         metric  samples_all  mean_all  sample_std_all  pop_std_all  samples_wins  mean_wins  sample_std_wins  pop_std_wins
       pig_iron            7   4065.86         4575.74      4236.31             6    4743.50          4611.68       4209.87
          metal            7   5377.29         5413.95      5012.34             6    6273.50          5331.62       4867.08
     kryptonite            7   5705.14         5930.12      5490.23             6    6656.00          5882.60       5370.05
          spice            7   2328.00         2527.64      2340.14             6    2716.00          2530.25       2309.79
       spice_eq            7   7058.27         6697.76      6200.92             6    8234.65          6496.73       5930.67
   net_spice_eq            7  -2532.16        29097.11     26938.69             6    8234.65          6496.73       5930.67
debris_spice_eq            7   2582.86         6553.12      6067.01             6     106.67           165.25        150.85

Recycle

## 4. Mean & Standard Deviation of Attack Time

How long an attack keeps a fleet slot busy, and how often attacks are sent. `attack_time_stats_df` reports the count, mean, sample standard deviation (`ddof=1`), population standard deviation (`ddof=0`) and median, in seconds, of three `numpy` arrays:

- `one_way_times_arr` — flight time from dispatch to impact;
- `round_trip_times_arr` — twice the one-way time, which is how long the fleet slot stays occupied;
- `inter_attack_intervals_arr` — the positive gaps between consecutive dispatches.

The scalars `attack_one_way_time_*`, `attack_time_*` (round trip) and `inter_attack_time_*` hold the unrounded mean, sample and population standard deviation.

In [61]:
def positive_gaps(timestamps: np.ndarray) -> np.ndarray:
    """Positive differences between consecutive timestamps (sorted first)."""
    gaps = np.diff(np.sort(timestamps))
    return gaps[gaps > 0.0]


def time_stats_row(label: str, samples: np.ndarray) -> dict[str, Any]:
    """One `attack_time_stats_df` row: count, mean, standard deviations and median of `samples` (seconds)."""
    mean, sample_std, pop_std = compute_mean_and_std(samples)
    return {
        "time_metric": label,
        "samples": int(samples.size),
        "mean_sec": round(mean, 2),
        "mean_text": fmt_duration(mean),
        "sample_std_sec": round(sample_std, 2),
        "sample_std_text": fmt_duration(sample_std),
        "pop_std_sec": round(pop_std, 2),
        "median_sec": round(float(np.median(samples)) if samples.size > 0 else 0.0, 2),
    }


one_way_times_arr: np.ndarray = attacks_df["one_way_sec"].to_numpy(dtype=np.float64)
round_trip_times_arr: np.ndarray = attacks_df["round_trip_sec"].to_numpy(dtype=np.float64)
inter_attack_intervals_arr: np.ndarray = positive_gaps(attack_dispatch_ts_arr)

attack_one_way_time_mean_sec, attack_one_way_time_std_sec, attack_one_way_time_pop_std_sec = (
    compute_mean_and_std(one_way_times_arr)
)
attack_time_mean_sec, attack_time_std_sec, attack_time_pop_std_sec = compute_mean_and_std(round_trip_times_arr)
inter_attack_time_mean_sec, inter_attack_time_std_sec, inter_attack_time_pop_std_sec = (
    compute_mean_and_std(inter_attack_intervals_arr)
)

attack_time_stats_df = pd.DataFrame(
    [
        time_stats_row("One-Way Flight Time (Dispatch -> Impact)", one_way_times_arr),
        time_stats_row("Round-Trip Attack Time (Slot Occupation)", round_trip_times_arr),
        time_stats_row("Inter-Attack Dispatch Interval", inter_attack_intervals_arr),
    ]
)

print(attack_time_stats_df.to_string(index=False))

                             time_metric  samples  mean_sec mean_text  sample_std_sec sample_std_text  pop_std_sec  median_sec
One-Way Flight Time (Dispatch -> Impact)        7    151.47    2m 31s          113.25          1m 53s       104.85      115.58
Round-Trip Attack Time (Slot Occupation)        7    302.93    5m 03s          226.49          3m 46s       209.69      231.17
          Inter-Attack Dispatch Interval        6  13249.65    3h 40m        21626.47          6h 00m     19742.18     3354.80


## 5. Attack Success Rate

With `W` wins in `N` attacks:

$$\hat{p} = \frac{W}{N}, \qquad \mathrm{SE}(\hat{p}) = \sqrt{\frac{\hat{p}\,(1 - \hat{p})}{N}}, \qquad \tilde{p} = \frac{W + 1}{N + 2}$$

- `attack_success_rate` is $\hat{p}$, `attack_success_rate_std_err` its binomial standard error, and `attack_success_rate_smoothed` the Laplace-smoothed $\tilde{p}$, which stays away from 0 % and 100 % when `N` is small.
- `attack_class_stats_df` and `attack_target_stats_df` break attempts, wins, losses, success rate and loot down by target class and by target (`pandas` `groupby`). `per_class_success_rate` and `per_target_success_rate` are the same numbers as dictionaries.
- `expected_raids_per_slot_hr` is 3600 divided by the mean round trip. Multiplied by the mean loot it gives the gross and net Spice-equivalent yield per fleet-slot hour, and with all `fleet_slot_capacity` slots the ceiling `max_empire_gross_spice_eq_hr`.
- `ATTACK_SIM_PARAMS` collects the scalars of Sections 3–5 for the simulation; `ATTACK_STATS_TABLES` registers the tables by name.

In [62]:
def summarize_attacks_by(attacks: pd.DataFrame, group_column: str) -> pd.DataFrame:
    """Attempts, wins, losses, success rate and loot statistics of `attacks` per value of `group_column`."""
    if attacks.empty:
        return pd.DataFrame(
            columns=[
                group_column,
                "attempts",
                "wins",
                "success_rate",
                "mean_spice_eq",
                "sample_std_spice_eq",
                "mean_net_spice_eq",
                "sample_std_net_spice_eq",
                "losses",
            ]
        )
    summary = (
        attacks.assign(won_num=attacks["won"].astype(np.float64))
        .groupby(group_column, as_index=False)
        .agg(
            attempts=("won_num", "count"),
            wins=("won_num", "sum"),
            success_rate=("won_num", "mean"),
            mean_spice_eq=("spice_eq", "mean"),
            sample_std_spice_eq=("spice_eq", "std"),
            mean_net_spice_eq=("net_spice_eq", "mean"),
            sample_std_net_spice_eq=("net_spice_eq", "std"),
        )
        .fillna(0.0)
        .sort_values(group_column)
        .reset_index(drop=True)
    )
    summary["wins"] = summary["wins"].astype(int)
    summary["losses"] = summary["attempts"] - summary["wins"]
    return summary


def success_rates_by(summary: pd.DataFrame, group_column: str) -> dict[str, dict[str, Any]]:
    """`{group value: counts, success rate and loot means}` view of a `summarize_attacks_by` table."""
    return {
        str(row[group_column]): {
            "attempts": int(row["attempts"]),
            "wins": int(row["wins"]),
            "losses": int(row["losses"]),
            "success_rate": float(row["success_rate"]),
            "mean_spice_eq": float(row["mean_spice_eq"]),
            "std_spice_eq": float(row["sample_std_spice_eq"]),
            "mean_net_spice_eq": float(row["mean_net_spice_eq"]),
        }
        for row in summary.to_dict("records")
    }


attack_attempts: int = int(len(attacks_df))
attack_wins: int = int(attacks_df["won"].sum())
attack_losses: int = attack_attempts - attack_wins

attack_success_rate: float = float(attacks_df["won"].mean()) if attack_attempts > 0 else 0.0
attack_success_rate_std_err: float = (
    float(np.sqrt(attack_success_rate * (1.0 - attack_success_rate) / float(attack_attempts)))
    if attack_attempts > 0
    else 0.0
)
attack_success_rate_smoothed: float = float(attack_wins + 1) / float(attack_attempts + 2)

attack_class_stats_df = summarize_attacks_by(attacks_df, "target_class")
attack_target_stats_df = summarize_attacks_by(attacks_df, "target_coords")
per_class_success_rate: dict[str, dict[str, Any]] = success_rates_by(attack_class_stats_df, "target_class")
per_target_success_rate: dict[str, dict[str, Any]] = success_rates_by(attack_target_stats_df, "target_coords")

expected_raids_per_slot_hr: float = 3600.0 / attack_time_mean_sec if attack_time_mean_sec > 0.0 else 0.0
expected_spice_eq_per_slot_hr: float = expected_raids_per_slot_hr * attack_resource_mean["spice_eq"]
expected_net_spice_eq_per_slot_hr: float = expected_raids_per_slot_hr * attack_resource_mean["net_spice_eq"]
max_empire_gross_spice_eq_hr: float = expected_spice_eq_per_slot_hr * float(fleet_slot_capacity)

attack_summary_stats_df = pd.DataFrame(
    [
        {
            "metric": "Total Attacks",
            "value": float(attack_attempts),
            "display": f"{attack_attempts} ({attack_wins} wins / {attack_losses} losses)",
        },
        {
            "metric": "Empirical Success Rate",
            "value": round(attack_success_rate, 6),
            "display": f"{attack_success_rate:.2%} (+/- {attack_success_rate_std_err:.2%} SE)",
        },
        {
            "metric": "Laplace-Smoothed Success Rate",
            "value": round(attack_success_rate_smoothed, 6),
            "display": f"{attack_success_rate_smoothed:.2%}",
        },
        {
            "metric": "Max Sorties per Fleet Slot / Hour",
            "value": round(expected_raids_per_slot_hr, 4),
            "display": (
                f"{expected_raids_per_slot_hr:.2f} raids/hr (mean round-trip {fmt_duration(attack_time_mean_sec)})"
            ),
        },
        {
            "metric": "Gross Loot per Slot-Hour (SpiceEq)",
            "value": round(expected_spice_eq_per_slot_hr, 2),
            "display": f"{expected_spice_eq_per_slot_hr:,.1f} SpiceEq/hr",
        },
        {
            "metric": "Net Loot per Slot-Hour (SpiceEq)",
            "value": round(expected_net_spice_eq_per_slot_hr, 2),
            "display": f"{expected_net_spice_eq_per_slot_hr:+,.1f} SpiceEq/hr",
        },
        {
            "metric": f"Full Capacity Ceiling ({fleet_slot_capacity} Slots)",
            "value": round(max_empire_gross_spice_eq_hr, 2),
            "display": f"{max_empire_gross_spice_eq_hr:,.1f} gross SpiceEq/hr",
        },
    ]
)

ATTACK_STATS_TABLES: dict[str, pd.DataFrame] = {
    "attacks": attacks_df,
    "recycles": recycles_df,
    "attack_resource_stats": attack_resource_stats_df,
    "attack_time_stats": attack_time_stats_df,
    "attack_summary_stats": attack_summary_stats_df,
    "attack_class_stats": attack_class_stats_df,
    "attack_target_stats": attack_target_stats_df,
}

ATTACK_SIM_PARAMS: dict[str, Any] = {
    "attempts": attack_attempts,
    "wins": attack_wins,
    "losses": attack_losses,
    "success_rate": attack_success_rate,
    "success_rate_std_err": attack_success_rate_std_err,
    "success_rate_smoothed": attack_success_rate_smoothed,
    "fleet_slot_capacity": fleet_slot_capacity,
    "resource_mean": dict(attack_resource_mean),
    "resource_std": dict(attack_resource_std),
    "winning_resource_mean": dict(winning_attack_resource_mean),
    "winning_resource_std": dict(winning_attack_resource_std),
    "recycle_spice_eq_mean": recycle_spice_eq_mean,
    "recycle_spice_eq_std": recycle_spice_eq_std,
    "one_way_time_mean_sec": attack_one_way_time_mean_sec,
    "one_way_time_std_sec": attack_one_way_time_std_sec,
    "round_trip_time_mean_sec": attack_time_mean_sec,
    "round_trip_time_std_sec": attack_time_std_sec,
    "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
    "inter_attack_time_std_sec": inter_attack_time_std_sec,
    "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
    "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
    "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
    "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
}

print(attack_summary_stats_df[["metric", "display"]].to_string(index=False))
if not attack_class_stats_df.empty:
    print("\nBy target class (attack_class_stats_df)")
    print(attack_class_stats_df.round(3).to_string(index=False))
if not attack_target_stats_df.empty:
    print("\nBy target (attack_target_stats_df)")
    print(attack_target_stats_df.round(3).to_string(index=False))

                            metric                                 display
                     Total Attacks                   7 (6 wins / 1 losses)
            Empirical Success Rate                  85.71% (+/- 13.23% SE)
     Laplace-Smoothed Success Rate                                  77.78%
 Max Sorties per Fleet Slot / Hour 11.88 raids/hr (mean round-trip 5m 03s)
Gross Loot per Slot-Hour (SpiceEq)                     83,879.4 SpiceEq/hr
  Net Loot per Slot-Hour (SpiceEq)                    -30,091.8 SpiceEq/hr
   Full Capacity Ceiling (7 Slots)              587,156.0 gross SpiceEq/hr

By target class (attack_class_stats_df)
   target_class  attempts  wins  success_rate  mean_spice_eq  sample_std_spice_eq  mean_net_spice_eq  sample_std_net_spice_eq  losses
         spacia         3     3          1.00       7821.400             8892.062           7821.400                 8892.062       0
yellow_inactive         4     3          0.75       6485.925             5999.023         -

## 6. Game Parameters Catalog

Tables of the game's parameters, built with `pandas` from `spaceinvasion.core.game_data` and the metadata of Section 1.6: costs, build and research times, production, energy, storage, combat statistics and technology effects of every building, technology, ship and defense.

Build times, the Spice mine's output and the Solar cells' energy depend on the planet. Section 6 therefore works on the whole empire: every colony you own, followed by generic planets (empty colony slots) until the roster has `MAX_EMPIRE_PLANETS` normal planets. It produces two kinds of tables:

- **Catalog and level tables** (`buildings_levels_df`, `technologies_levels_df`, `ships_df`, `defenses_df` and the two catalogs) list every entity and level for a generic planet: universe speed and research levels applied, no accelerator buildings, the temperature `EMPTY_COLONY_DEFAULT_TEMP_C`.
- **Per-planet tables** (`planet_buildings_df`, `planet_technologies_df`, `planet_ships_df`, `planet_defenses_df`) have one row per planet and entity, computed with that planet's own building levels and temperature.

This cell loads the roster and collects the inputs:

- `live_planet_states` and `empire_planet_states` — your colonies, and your colonies followed by the generic planets, as `PlanetState` objects;
- `GAME_PARAM_CONTEXT` — what is the same on every planet: the universe speed and the research levels `r1_mining_tech` to `r10_defense_systems`;
- `PLANET_PARAM_CONTEXTS` — one context per planet of the roster: the account-wide values, the planet's temperature, and the levels of the buildings that shorten build and research times (`b7_dev_center`, `b8_research_lab`, `b9_weapons_factory`, `b17_microsystem`). `GENERIC_PLANET_CONTEXT` is the context of a generic planet;
- `planet_contexts_df` — the planets' own inputs as a table, with the resulting divisors of the base times (formulas in Section 1.3);
- `merchant_rates_df` — the Merchant parity (`6 Pig iron : 2.5 Metal : 3 Kryptonite : 1 Spice`), the ranking value of a resource unit (1 point per 1,000 resources spent) and the mine of each resource.

`MAX_BUILDING_TABLE_LEVEL` and `MAX_RESEARCH_TABLE_LEVEL` cap the level tables. `MAX_EMPIRE_PLANETS`, `EMPTY_COLONY_SIZE_FIELDS` and `EMPTY_COLONY_DEFAULT_TEMP_C` describe the roster. All five come from `planner_settings.json`. The planet limit is a setting, not a value read from the game: 11 is the default of the bot's own colony cap (`StrategyConfig.max_planets`).

Every table builder takes a context, so a full table can be rebuilt for any planet or state: `build_building_tables(PLANET_PARAM_CONTEXTS[0], MAX_BUILDING_TABLE_LEVEL)` returns the catalog and level table of the first planet, and `build_ship_table({**GENERIC_PLANET_CONTEXT, "r8_weapons_tech": 5})` the ships with Weapons technology 5.

In [63]:
MAX_BUILDING_TABLE_LEVEL: int = int(PLANNER_SETTINGS["table_level_caps"]["buildings"])
MAX_RESEARCH_TABLE_LEVEL: int = int(PLANNER_SETTINGS["table_level_caps"]["researches"])
MAX_EMPIRE_PLANETS: int = int(PLANNER_SETTINGS["empire"]["max_planets"])
EMPTY_COLONY_SIZE_FIELDS: int = int(PLANNER_SETTINGS["empire"]["empty_colony_fields"])
EMPTY_COLONY_DEFAULT_TEMP_C: int = int(PLANNER_SETTINGS["empire"]["empty_colony_temperature_c"])


def planet_param_context(
    account_context: dict[str, Any],
    slot_index: int,
    planet: PlanetState,
    *,
    is_empty_slot: bool,
) -> dict[str, Any]:
    """Inputs of the table builders for one planet: the account-wide context plus the planet's own values."""
    return {
        **account_context,
        "slot_index": slot_index,
        "planet_id": int(planet.planet_id),
        "coords": planet.coords,
        "planet_status": "empty_colony_slot" if is_empty_slot else "colonized",
        "planet_name": planet.name,
        "planet_type": planet.planet_type,
        "planet_temperature_c": int(planet.temperature),
        "b7_dev_center": planet.buildings.get("b7", 0),
        "b8_research_lab": planet.buildings.get("b8", 0),
        "b9_weapons_factory": planet.buildings.get("b9", 0),
        "b17_microsystem": planet.buildings.get("b17", 0),
    }


def planet_identity(context: dict[str, Any]) -> dict[str, Any]:
    """The four columns that identify a planet in every per-planet table."""
    return {column: context[column] for column in ("slot_index", "planet_id", "coords", "planet_status")}


def planet_context_row(context: dict[str, Any]) -> dict[str, Any]:
    """One `planet_contexts_df` row: a planet's own inputs and the time divisors that follow from them."""
    speedup = context["speedup_factor"]
    return {
        **planet_identity(context),
        "planet_name": context["planet_name"],
        "planet_type": context["planet_type"],
        "temperature_c": context["planet_temperature_c"],
        "b7_dev_center": context["b7_dev_center"],
        "b8_research_lab": context["b8_research_lab"],
        "b9_weapons_factory": context["b9_weapons_factory"],
        "b17_microsystem": context["b17_microsystem"],
        "building_time_divisor": building_time_divisor(context["b7_dev_center"], context["b17_microsystem"], speedup),
        "research_time_divisor": research_time_divisor(context["b8_research_lab"], speedup),
        "shipyard_time_divisor": shipyard_time_divisor(
            context["b9_weapons_factory"], context["b17_microsystem"], speedup
        ),
    }


def next_level_row(
    context: dict[str, Any],
    current_level: int,
    level_row: dict[str, Any],
    extra: dict[str, Any],
) -> dict[str, Any]:
    """Row of a per-planet table: the planet, then a level-table row for the level after `current_level`.

    The `level` column of `level_row` becomes `next_level`; `extra` and `current_level` are placed before it.
    """
    row = planet_identity(context)
    for column, value in level_row.items():
        if column == "level":
            row.update(extra)
            row["current_level"] = current_level
            row["next_level"] = value
        else:
            row[column] = value
    return row


# Every owned colony, then generic planets (empty colony slots) until the empire has MAX_EMPIRE_PLANETS normal planets.
live_planet_states: list[PlanetState] = load_live_planet_states(LIVE_DB_PATH)
empire_planet_states: list[PlanetState] = build_padded_empire_planets(
    live_planet_states,
    max_planets=MAX_EMPIRE_PLANETS,
    empty_planet_size=EMPTY_COLONY_SIZE_FIELDS,
    empty_temp_c=EMPTY_COLONY_DEFAULT_TEMP_C,
)

# Account-wide inputs: the universe speed and the research levels are the same on every planet.
GAME_PARAM_CONTEXT: dict[str, Any] = {
    "wiki_synced_at_utc": game_data.WIKI_SYNCED_AT_UTC,
    "speedup_factor": float(telemetry_bundle["universe_speedup_factor"]),
    "r1_mining_tech": colony_researches.get("r1", 0),
    "r2_energy_tech": colony_researches.get("r2", 0),
    "r4_rocket_engine": colony_researches.get("r4", 0),
    "r5_diffusion_engine": colony_researches.get("r5", 0),
    "r6_warp_engine": colony_researches.get("r6", 0),
    "r8_weapons_tech": colony_researches.get("r8", 0),
    "r9_shield_tech": colony_researches.get("r9", 0),
    "r10_defense_systems": colony_researches.get("r10", 0),
    "points_per_resource_divisor": float(POINTS_PER_RESOURCE_DIVISOR),
}
# One builder context per planet of the roster, and one for a generic planet, the baseline of the level tables.
PLANET_PARAM_CONTEXTS: list[dict[str, Any]] = [
    planet_param_context(GAME_PARAM_CONTEXT, slot_index, planet, is_empty_slot=slot_index > len(live_planet_states))
    for slot_index, planet in enumerate(empire_planet_states, start=1)
]
GENERIC_PLANET_CONTEXT: dict[str, Any] = planet_param_context(
    GAME_PARAM_CONTEXT,
    0,
    empty_colony_slot(0, EMPTY_COLONY_SIZE_FIELDS, EMPTY_COLONY_DEFAULT_TEMP_C),
    is_empty_slot=True,
)
planet_contexts_df = pd.DataFrame([planet_context_row(context) for context in PLANET_PARAM_CONTEXTS])

merchant_rates_df = pd.DataFrame(
    [
        {
            "resource": resource,
            "units_per_1_spice": float(game_data.MERCHANT_RATES_IN_SPICE[resource]),
            "spice_eq_per_unit": round(1.0 / float(game_data.MERCHANT_RATES_IN_SPICE[resource]), 6),
            "points_per_unit": 1.0 / float(POINTS_PER_RESOURCE_DIVISOR),
            "mine_building_key": RESOURCE_METADATA[resource]["mine_building_key"],
            "boosted_by_mining_tech_r1": bool(RESOURCE_METADATA[resource]["boosted_by_mining_tech"]),
            "notes": RESOURCE_METADATA[resource]["notes"],
        }
        for resource in RESOURCE_KEYS
    ]
)

print(
    f"{len(live_planet_states)} owned planet(s) + {len(empire_planet_states) - len(live_planet_states)} generic "
    f"{EMPTY_COLONY_SIZE_FIELDS}-field planet(s), universe speed {GAME_PARAM_CONTEXT['speedup_factor']:.1f}x\n"
)
print(planet_contexts_df.to_string(index=False))
print()
print(merchant_rates_df.to_string(index=False))

1 owned planet(s) + 10 generic 300-field planet(s), universe speed 20.0x

 slot_index  planet_id    coords     planet_status          planet_name planet_type  temperature_c  b7_dev_center  b8_research_lab  b9_weapons_factory  b17_microsystem  building_time_divisor  research_time_divisor  shipyard_time_divisor
          1      12476 02:380:12         colonized               Colony      normal             31              9                3                   4                0                  200.0                   80.0                   80.0
          2         -1  EMPTY:01 empty_colony_slot  Empty Colony Slot 1      normal              0              0                0                   0                0                   20.0                   20.0                   20.0
          3         -2  EMPTY:02 empty_colony_slot  Empty Colony Slot 2      normal              0              0                0                   0                0                   20.0                   20.0  

### 6.1 Buildings (`b1`–`b21`)

- `buildings_catalog_df` — one row per building: category, whether it can be built on an asteroid, number of levels in the catalog, cost scale factor, level-1 cost and time, prerequisites and primary effect.
- `buildings_levels_df` — one row per building level on a generic planet, up to `MAX_BUILDING_TABLE_LEVEL`:
  - cost per resource, `cost_spice_eq`, and `points_gain` (cost / 1,000);
  - `base_time_s` from the catalog and `effective_build_time_s`. On a specific planet, the time is `base_time_s` divided by that planet's `building_time_divisor` in `planet_contexts_df`;
  - mines: `hourly_prod_1x` (universe speed 1, no research) and `hourly_prod_effective` (universe speed, and +2 % per Mining technology level except for the Blast furnace). The Spice mine's output also follows the planet's temperature: the colder, the more;
  - feedstock: `pig_iron_consumed_hr_effective` (the Blast furnace uses 2 Pig iron per Metal) and `spice_consumed_hr` (fuel of the Thermosolar plant);
  - `net_spice_eq_hr_effective`, its gain over the previous level `delta_net_spice_eq_hr`, and `marginal_roi_hours`, the hours the level needs to pay back its cost (empty when the level adds no net output);
  - energy produced (positive) or consumed (negative): `energy_at_level_base`, and `energy_at_level_effective` with +1 % per Energy technology level on production only;
  - `storage_capacity` of the warehouses (`100,000 * 2^level`) and `time_reduction_multiplier` of the accelerator buildings (`1 / (level + 1)` for `b7` and `b8`, `1 / level` for `b9`, `0.5^level` for `b17`).
- `planet_buildings_df` — one row per planet and building: the `current_level` on that planet, the `next_level`, and the level-table columns of that next level computed for the planet, with its own build time and Spice mine output. `delta_net_spice_eq_hr` and `energy_delta_from_prev` are the gains over the current level. `buildable_on_planet` is false for a building that the catalog does not allow on an asteroid. A building already at the last level of the catalog has no row.

In [64]:
def building_time_reduction(key: str, level: int) -> float:
    """Share of the base time that an accelerator building at `level` leaves (1.0 for every other building)."""
    if key in ("b7", "b8"):  # Development center (buildings) and Research lab (research)
        return 1.0 / float(level + 1)
    if key == "b9":  # Weapons factory (ships and defenses)
        return 1.0 / float(max(1, level))
    if key == "b17":  # Microsystem accelerator (buildings, ships and defenses)
        return 0.5 ** float(level)
    return 1.0


def produced_resource(building_key: str) -> str:
    """Resource that a building produces (`"none"` for a building that is not a mine)."""
    return next(
        (resource for resource, meta in RESOURCE_METADATA.items() if meta["mine_building_key"] == building_key),
        "none",
    )


def building_level_economy(building_id: int, level: int, context: dict[str, Any]) -> dict[str, float]:
    """Hourly output, feedstock, net Spice-equivalent value and energy of one building level on the context's planet.

    Level 0, a building that is not built, yields zeros.
    """
    row = (game_data.BUILDING_BY_ID[building_id].get("levels") or {}).get(level) or {}
    merchant_rates = game_data.MERCHANT_RATES_IN_SPICE
    resource = produced_resource(f"b{building_id}")
    speedup = float(context["speedup_factor"])

    # Hourly output scales with universe speed; Mining technology adds +2 % per level (not to the Blast furnace).
    # Only the Spice mine depends on the planet: its output follows the temperature.
    if resource != "none":
        temperature = int(context["planet_temperature_c"])
        production_1x = float(
            game_data.mine_production_per_hour(building_id, level, temperature=temperature, mining_tech_level=0)
        )
        production = speedup * float(
            game_data.mine_production_per_hour(
                building_id, level, temperature=temperature, mining_tech_level=int(context["r1_mining_tech"])
            )
        )
    else:
        production_1x = 0.0
        production = 0.0
    # Feedstock per hour: the Blast furnace's Pig iron (2 per Metal) scales with universe speed like its output;
    # the Thermosolar plant's Spice fuel is the catalog value, not multiplied by universe speed.
    pig_iron_used = float(row.get("pig_iron_used") or 0.0) * speedup
    spice_used = float(row.get("spice_used") or 0.0)
    gross_spice_eq_hr = production / float(merchant_rates[resource]) if resource in merchant_rates else 0.0
    feedstock_spice_eq_hr = pig_iron_used / float(merchant_rates["pig_iron"]) + spice_used / float(
        merchant_rates["spice"]
    )
    # Energy: Energy technology boosts production (positive values) only; consumption is unchanged.
    energy_base = float(row.get("energy") or 0.0)
    energy_multiplier = technology_multiplier("r2", context["r2_energy_tech"])
    return {
        "production_1x": production_1x,
        "production": production,
        "pig_iron_used": pig_iron_used,
        "spice_used": spice_used,
        "net_spice_eq_hr": gross_spice_eq_hr - feedstock_spice_eq_hr,
        "energy_base": energy_base,
        "energy": (energy_base * energy_multiplier) if energy_base > 0.0 else energy_base,
    }


def building_level_row(building_id: int, level: int, base_time: float, context: dict[str, Any]) -> dict[str, Any]:
    """One level-table row: cost, time, production, energy and storage of `level` on the context's planet."""
    info = game_data.BUILDING_BY_ID[building_id]
    key = f"b{building_id}"
    row = (info.get("levels") or {}).get(level) or {}
    cost = game_data.building_cost(building_id, level)
    cost_spice_eq = float(game_data.spice_equivalent(cost))
    build_time = base_time / building_time_divisor(
        context["b7_dev_center"], context["b17_microsystem"], context["speedup_factor"]
    )
    economy = building_level_economy(building_id, level, context)
    previous = building_level_economy(building_id, level - 1, context)
    # Net value gained over the previous level, and how long this level takes to pay for itself.
    delta_net_spice_eq_hr = economy["net_spice_eq_hr"] - previous["net_spice_eq_hr"]
    roi_hours = (cost_spice_eq / delta_net_spice_eq_hr) if delta_net_spice_eq_hr > 0.0 else float("inf")
    return {
        "building_id": building_id,
        "key": key,
        "slug": str(info["slug"]),
        "name": str(info["name"]),
        "category": BUILDING_METADATA[key]["category"],
        "level": level,
        "pig_iron": int(cost["pig_iron"]),
        "metal": int(cost["metal"]),
        "kryptonite": int(cost["kryptonite"]),
        "spice": int(cost["spice"]),
        "cost_spice_eq": round(cost_spice_eq, 2),
        "points_gain": round(cost_points(cost), 3),
        "base_time_s": round(base_time, 2),
        "effective_build_time_s": round(build_time, 2),
        "effective_build_time_text": fmt_duration(build_time),
        "produced_resource": produced_resource(key),
        "hourly_prod_1x": round(economy["production_1x"], 2),
        "hourly_prod_effective": round(economy["production"], 2),
        "pig_iron_consumed_hr_effective": round(economy["pig_iron_used"], 2),
        "spice_consumed_hr": round(economy["spice_used"], 2),
        "net_spice_eq_hr_effective": round(economy["net_spice_eq_hr"], 2),
        "delta_net_spice_eq_hr": round(delta_net_spice_eq_hr, 2),
        "marginal_roi_hours": round(roi_hours, 2) if math.isfinite(roi_hours) else None,
        "energy_at_level_base": round(economy["energy_base"], 2),
        "energy_at_level_effective": round(economy["energy"], 2),
        "energy_delta_from_prev": round(economy["energy"] - previous["energy"], 2),
        "storage_capacity": int(row.get("storage_capacity") or 0),
        "time_reduction_multiplier": building_time_reduction(key, level),
    }


def build_building_tables(context: dict[str, Any], max_level: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build the building catalog (one row per building) and the level table (one row per building level)."""
    time_divisor = building_time_divisor(
        context["b7_dev_center"], context["b17_microsystem"], context["speedup_factor"]
    )
    catalog_rows: list[dict[str, Any]] = []
    level_rows: list[dict[str, Any]] = []
    for building_id in sorted(game_data.BUILDING_BY_ID):
        info = game_data.BUILDING_BY_ID[building_id]
        key = f"b{building_id}"
        levels = info.get("levels") or {}
        table_levels = int(info.get("max_table_level") or len(levels))
        base_times = build_monotonic_base_times(levels, table_levels)
        level_1_cost = game_data.building_cost(building_id, 1)
        level_1_time = float(base_times.get(1, 0.0))
        catalog_rows.append(
            {
                "building_id": building_id,
                "key": key,
                "slug": str(info["slug"]),
                "name": str(info["name"]),
                "category": BUILDING_METADATA[key]["category"],
                "asteroid_ok": bool(info.get("asteroid_ok", False)),
                "max_table_level": table_levels,
                "cost_scale_factor": BUILDING_METADATA[key]["cost_scale_factor"],
                "l1_pig_iron": int(level_1_cost["pig_iron"]),
                "l1_metal": int(level_1_cost["metal"]),
                "l1_kryptonite": int(level_1_cost["kryptonite"]),
                "l1_spice": int(level_1_cost["spice"]),
                "l1_cost_spice_eq": round(float(game_data.spice_equivalent(level_1_cost)), 2),
                "l1_base_time_s": round(level_1_time, 2),
                "l1_effective_time_s": round(level_1_time / time_divisor, 2),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
                "primary_effect": BUILDING_METADATA[key]["primary_effect"],
            }
        )
        level_rows.extend(
            building_level_row(building_id, level, float(base_times.get(level, 0.0)), context)
            for level in range(1, min(table_levels, max_level) + 1)
        )
    return pd.DataFrame(catalog_rows), pd.DataFrame(level_rows)


def build_planet_building_table(planets: list[PlanetState], contexts: list[dict[str, Any]]) -> pd.DataFrame:
    """One row per planet and building: the level built there and the next level, computed for that planet.

    The columns after `next_level` are the level-table columns of that next level. A building already at the
    last level of the catalog has no row.
    """
    rows: list[dict[str, Any]] = []
    for planet, context in zip(planets, contexts):
        for building_id in sorted(game_data.BUILDING_BY_ID):
            info = game_data.BUILDING_BY_ID[building_id]
            levels = info.get("levels") or {}
            current_level = max(0, int(planet.buildings.get(f"b{building_id}", 0) or 0))
            next_level = current_level + 1
            if next_level > int(info.get("max_table_level") or len(levels)):
                continue
            base_time = float(build_monotonic_base_times(levels, next_level).get(next_level, 0.0))
            # The catalog marks the few buildings an asteroid can hold; a normal planet takes every building.
            buildable = planet.planet_type != "asteroid" or bool(info.get("asteroid_ok", False))
            rows.append(
                next_level_row(
                    context,
                    current_level,
                    building_level_row(building_id, next_level, base_time, context),
                    {"buildable_on_planet": buildable},
                )
            )
    return pd.DataFrame(rows)


buildings_catalog_df, buildings_levels_df = build_building_tables(GENERIC_PLANET_CONTEXT, MAX_BUILDING_TABLE_LEVEL)
planet_buildings_df = build_planet_building_table(empire_planet_states, PLANET_PARAM_CONTEXTS)

print(
    f"buildings_catalog_df: {buildings_catalog_df.shape}, buildings_levels_df: {buildings_levels_df.shape} "
    f"(generic planet), planet_buildings_df: {planet_buildings_df.shape} (next level on each planet)\n"
)
print(
    buildings_catalog_df[
        [
            "key",
            "name",
            "category",
            "asteroid_ok",
            "cost_scale_factor",
            "l1_cost_spice_eq",
            "l1_base_time_s",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\nNext level of the mines and the Fusion plant on the owned planets and on the first generic planet:")
print(
    planet_buildings_df[
        planet_buildings_df["key"].isin(["b1", "b2", "b3", "b4", "b5"])
        & planet_buildings_df["buildable_on_planet"]
        & (planet_buildings_df["slot_index"] <= len(live_planet_states) + 1)
    ][
        [
            "slot_index",
            "coords",
            "planet_status",
            "key",
            "name",
            "current_level",
            "next_level",
            "cost_spice_eq",
            "effective_build_time_text",
            "hourly_prod_effective",
            "delta_net_spice_eq_hr",
            "marginal_roi_hours",
            "energy_delta_from_prev",
        ]
    ].to_string(index=False)
)

buildings_catalog_df: (21, 17), buildings_levels_df: (975, 28) (generic planet), planet_buildings_df: (231, 34) (next level on each planet)

key                    name          category  asteroid_ok  cost_scale_factor  l1_cost_spice_eq  l1_base_time_s                                            prerequisites
 b1           Pig iron mine     resource_mine        False               1.51             21.33            93.0                                                     None
 b2           Blast furnace           smelter        False               1.51             20.83            90.0                                                     None
 b3         Kryptonite mine     resource_mine        False               1.51             33.33           136.0                                                     None
 b4              Spice mine     resource_mine        False               1.51            114.67           272.0                                                     None
 b5           

### 6.2 Technologies (`r1`–`r15`)

A research applies to the whole account: every planet, asteroid and fleet. Each level costs twice the previous one, and the research time is divided by `(b8 + 1) * speed` of the planet that researches it.

- `technologies_catalog_df` — one row per research: domain, current level, level-1 cost and time, the quantity it changes (`effect_metric`), `per_level_bonus_pct`, what it applies to, prerequisites and strategic effect. The descriptive columns come from `technologies.json`.
- `technologies_levels_df` — one row per research level, up to `MAX_RESEARCH_TABLE_LEVEL`: cost, `points_gain`, the base research time and the time on a generic planet, the cumulative bonus `bonus_pct_total`, `multiplier_at_level` (`1 + bonus_pct_total / 100`) and a readable `effect_at_level`.
- `planet_technologies_df` — one row per planet and research: the account's `current_level`, the `next_level`, and the level-table columns of that next level with the research time in the planet's own Research lab (`research_lab_level`). It shows where the next level is researched fastest. A planet whose `research_lab_level` is 0 cannot research; its time is the base time at universe speed. A research already at the last level of the catalog has no row.

Per-level bonuses: +2 % mine output for `r1` (not the Blast furnace), +1 % energy output for `r2`, +10 % ship speed for the engine researches `r4`, `r5` and `r6`, and +10 % attack, shield and hull for `r8`, `r9` and `r10`.

In [65]:
def technology_effect_at_level(key: str, level: int) -> str:
    """Human-readable effect of research `key` at `level`."""
    bonus_pct = float(TECHNOLOGY_METADATA[key]["per_level_bonus_pct"])
    if bonus_pct > 0.0:
        return f"+{bonus_pct * float(level):.0f}% ({technology_multiplier(key, level):.2f}x)"
    if key == "r15":  # Black hole research: scanner jam of 4 h + 1 h per level; level 2 unlocks the Imperial starbase
        return f"Scanner jam {4 + level}h" + (" + Starbase unlocked" if level >= 2 else "")
    return f"Tier L{level} unlock / capability"


def technology_identity(key: str, info: dict[str, Any]) -> dict[str, Any]:
    """The columns that identify a research in the catalog and level tables."""
    return {
        "research_id": int(key[1:]),
        "key": key,
        "slug": str(info["slug"]),
        "name": str(info["name"]),
        "domain": str(TECHNOLOGY_METADATA[key]["domain"]),
    }


def technology_level_row(
    key: str,
    info: dict[str, Any],
    level: int,
    base_time: float,
    time_divisor: float,
) -> dict[str, Any]:
    """One level-table row of research `key`: cost, research time and cumulative effect at `level`."""
    cost = full_cost(((info.get("levels") or {}).get(level) or {}).get("cost"))
    research_time = base_time / time_divisor
    bonus_pct = float(TECHNOLOGY_METADATA[key]["per_level_bonus_pct"])
    return {
        **technology_identity(key, info),
        "level": level,
        "pig_iron": int(cost["pig_iron"]),
        "metal": int(cost["metal"]),
        "kryptonite": int(cost["kryptonite"]),
        "spice": int(cost["spice"]),
        "cost_spice_eq": round(float(game_data.spice_equivalent(cost)), 2),
        "points_gain": round(cost_points(cost), 3),
        "base_time_s": round(base_time, 2),
        "effective_research_time_s": round(research_time, 2),
        "effective_research_time_text": fmt_duration(research_time),
        "bonus_pct_total": round(bonus_pct * float(level), 2),
        "multiplier_at_level": round(technology_multiplier(key, level), 4),
        "effect_at_level": technology_effect_at_level(key, level),
    }


def build_technology_tables(
    context: dict[str, Any],
    research_levels: dict[str, int],
    max_level: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build the research catalog (one row per research) and the level table (one row per research level)."""
    time_divisor = research_time_divisor(context["b8_research_lab"], context["speedup_factor"])
    catalog_rows: list[dict[str, Any]] = []
    level_rows: list[dict[str, Any]] = []
    for key, info in research_catalog():
        meta = TECHNOLOGY_METADATA[key]
        levels = info.get("levels") or {}
        table_levels = int(info.get("max_table_level") or len(levels))
        base_times = build_monotonic_base_times(levels, table_levels)
        level_1_cost = full_cost((levels.get(1) or {}).get("cost") or info.get("base_cost"))
        level_1_time = float(base_times.get(1, 0.0))
        catalog_rows.append(
            {
                **technology_identity(key, info),
                "current_level": research_levels.get(key, 0),
                "max_table_level": table_levels,
                "cost_scale_factor": TECHNOLOGY_COST_SCALE_FACTOR,
                "l1_pig_iron": int(level_1_cost["pig_iron"]),
                "l1_metal": int(level_1_cost["metal"]),
                "l1_kryptonite": int(level_1_cost["kryptonite"]),
                "l1_spice": int(level_1_cost["spice"]),
                "l1_cost_spice_eq": round(float(game_data.spice_equivalent(level_1_cost)), 2),
                "l1_base_time_s": round(level_1_time, 2),
                "l1_effective_time_s": round(level_1_time / time_divisor, 2),
                "effect_metric": str(meta["effect_metric"]),
                "per_level_bonus_pct": float(meta["per_level_bonus_pct"]),
                "applies_to": str(meta["applies_to"]),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
                "strategic_effect": str(meta["strategic_effect"]),
            }
        )
        level_rows.extend(
            technology_level_row(key, info, level, float(base_times.get(level, 0.0)), time_divisor)
            for level in range(1, min(table_levels, max_level) + 1)
        )
    return pd.DataFrame(catalog_rows), pd.DataFrame(level_rows)


def build_planet_technology_table(contexts: list[dict[str, Any]], research_levels: dict[str, int]) -> pd.DataFrame:
    """One row per planet and research: the next research level, timed with that planet's Research lab.

    Research levels belong to the account, so `current_level` is the same on every planet; only the research
    time differs. A research already at the last level of the catalog has no row.
    """
    rows: list[dict[str, Any]] = []
    for context in contexts:
        time_divisor = research_time_divisor(context["b8_research_lab"], context["speedup_factor"])
        for key, info in research_catalog():
            levels = info.get("levels") or {}
            current_level = max(0, int(research_levels.get(key, 0) or 0))
            next_level = current_level + 1
            if next_level > int(info.get("max_table_level") or len(levels)):
                continue
            base_time = float(build_monotonic_base_times(levels, next_level).get(next_level, 0.0))
            rows.append(
                next_level_row(
                    context,
                    current_level,
                    technology_level_row(key, info, next_level, base_time, time_divisor),
                    {"research_lab_level": context["b8_research_lab"]},
                )
            )
    return pd.DataFrame(rows)


technologies_catalog_df, technologies_levels_df = build_technology_tables(
    GENERIC_PLANET_CONTEXT, colony_researches, MAX_RESEARCH_TABLE_LEVEL
)
planet_technologies_df = build_planet_technology_table(PLANET_PARAM_CONTEXTS, colony_researches)

print(
    f"technologies_catalog_df: {technologies_catalog_df.shape}, "
    f"technologies_levels_df: {technologies_levels_df.shape} (generic planet), "
    f"planet_technologies_df: {planet_technologies_df.shape} (next level on each planet)\n"
)
print(
    technologies_catalog_df[
        [
            "key",
            "name",
            "domain",
            "current_level",
            "l1_cost_spice_eq",
            "l1_base_time_s",
            "per_level_bonus_pct",
            "prerequisites",
            "strategic_effect",
        ]
    ].to_string(index=False)
)
print("\nNext level of every research on the first planet of the roster:")
print(
    planet_technologies_df[planet_technologies_df["slot_index"] == 1][
        [
            "coords",
            "research_lab_level",
            "key",
            "name",
            "current_level",
            "next_level",
            "cost_spice_eq",
            "effective_research_time_text",
            "effect_at_level",
        ]
    ].to_string(index=False)
)

technologies_catalog_df: (15, 20), technologies_levels_df: (445, 18) (generic planet), planet_technologies_df: (165, 24) (next level on each planet)

key                name           domain  current_level  l1_cost_spice_eq  l1_base_time_s  per_level_bonus_pct                                       prerequisites                                                                   strategic_effect
 r1   Mining technology          economy              9            333.33          2083.0                  2.0     r2 L4 (Energy technology), b8 L2 (Research lab)                            +2%/lvl global Pig iron, Kryptonite & Spice mine output
 r2   Energy technology          economy              4            826.67          1833.0                  1.0                                b8 L1 (Research lab)                  +1%/lvl global energy output; gates b9, b6, r1, shields & turrets
 r3     Engine research       propulsion              6            570.00          1000.0                  0.0  

### 6.3 Spaceships (`s1`–`s16`)

`ships_df` has one row per ship:

- unit cost per resource, `cost_spice_eq`, and `points_value` (cost / 1,000);
- `base_build_time_s`, and `effective_build_time_s` and `units_per_hour_effective` on a generic planet;
- combat: `hull_base`, `shield_base` and `attack_base`, and the `*_effective` values with +10 % per level of Defense systems (`r10`), Shield technology (`r9`) and Weapons technology (`r8`). SpaceInvasion Classic has no rapid fire: every unit fires once per combat round;
- propulsion: `engine`, the research that speeds it up (`engine_tech_key`), `speed_base`, and `speed_effective` with +10 % per level of that research;
- logistics: `cargo`, `fuel`, and the ratios `cargo_per_spice_eq`, `attack_per_spice_eq` and `hull_per_spice_eq`;
- `energy_output_base` and `energy_output_effective`: only Solar cells (`s9`) produce energy. Their output depends on the planet's temperature, and they use no building field.

`planet_ships_df` has one row per planet and ship: the ships `stationed` there, the build time and hourly throughput in that planet's Weapons factory, and the Solar cell energy at that planet's temperature. A planet without a Weapons factory is timed as if it had level 1.

`combat_multipliers` and `planet_unit_rows`, defined here, are used again for the defenses in 6.4.

In [66]:
def combat_multipliers(context: dict[str, Any]) -> dict[str, float]:
    """Attack, shield and hull multipliers from the Weapons (`r8`), Shield (`r9`) and Defense systems (`r10`) levels."""
    return {
        "attack": technology_multiplier("r8", context["r8_weapons_tech"]),
        "shield": technology_multiplier("r9", context["r9_shield_tech"]),
        "hull": technology_multiplier("r10", context["r10_defense_systems"]),
    }


def build_ship_table(context: dict[str, Any]) -> pd.DataFrame:
    """Build `ships_df`: cost, build time, combat stats, propulsion, cargo and orbital energy of every ship."""
    time_divisor = shipyard_time_divisor(
        context["b9_weapons_factory"], context["b17_microsystem"], context["speedup_factor"]
    )
    multiplier = combat_multipliers(context)
    energy_multiplier = technology_multiplier("r2", context["r2_energy_tech"])
    temperature = float(context["planet_temperature_c"])
    # A ship's `engine` stat is the name of the research that speeds it up (Rocket, Diffusion or Warp engine).
    engine_key_by_name = {str(info["name"]): key for key, info in research_catalog()}
    engine_levels = {
        "r4": context["r4_rocket_engine"],
        "r5": context["r5_diffusion_engine"],
        "r6": context["r6_warp_engine"],
    }

    rows: list[dict[str, Any]] = []
    for key in unit_keys(SHIP_SLUG_BY_KEY):
        slug = SHIP_SLUG_BY_KEY[key]
        info = game_data.SHIPS[slug]
        stats = info.get("stats") or {}
        cost = full_cost(info.get("base_cost"))
        cost_spice_eq = float(game_data.spice_equivalent(cost))
        base_time = float(info.get("base_build_time_s") or 0.0)
        build_time = base_time / time_divisor

        hull, shield, attack = (int(stats.get(stat) or 0) for stat in ("hull", "shield", "attack"))
        speed = int(stats.get("speed") or 0)
        cargo = int(stats.get("cargo") or 0)
        engine = str(stats.get("engine") or ("Stationary" if speed == 0 else "Unknown"))
        engine_key = engine_key_by_name.get(engine, "none")
        engine_level = int(engine_levels.get(engine_key, 0))
        # Only Solar cells (`s9`) produce energy; they orbit the planet and use no building field.
        energy_base = solar_cell_energy_per_unit(temperature) if key == "s9" else 0.0

        rows.append(
            {
                "ship_id": int(key[1:]),
                "key": key,
                "slug": slug,
                "name": str(info["name"]),
                "role": str(SHIP_METADATA[key]["role"]),
                "pig_iron": int(cost["pig_iron"]),
                "metal": int(cost["metal"]),
                "kryptonite": int(cost["kryptonite"]),
                "spice": int(cost["spice"]),
                "cost_spice_eq": round(cost_spice_eq, 2),
                "points_value": round(cost_points(cost), 3),
                "base_build_time_s": round(base_time, 2),
                "effective_build_time_s": round(build_time, 2),
                "effective_build_time_text": fmt_duration(build_time),
                "units_per_hour_effective": round((3600.0 / build_time) if build_time > 0.0 else 0.0, 2),
                "hull_base": hull,
                "shield_base": shield,
                "attack_base": attack,
                "hull_effective": round(float(hull) * multiplier["hull"], 1),
                "shield_effective": round(float(shield) * multiplier["shield"], 1),
                "attack_effective": round(float(attack) * multiplier["attack"], 1),
                "engine": engine,
                "engine_tech_key": engine_key,
                "speed_base": speed,
                "speed_effective": round(float(speed) * technology_multiplier(engine_key, engine_level), 1),
                "cargo": cargo,
                "fuel": int(stats.get("fuel") or 0),
                "energy_output_base": round(energy_base, 2),
                "energy_output_effective": round(energy_base * energy_multiplier, 2),
                "fields_used": 0,
                "cargo_per_spice_eq": round(float(cargo) / cost_spice_eq, 3) if cost_spice_eq > 0.0 else 0.0,
                "attack_per_spice_eq": (
                    round((float(attack) * multiplier["attack"]) / cost_spice_eq, 4) if cost_spice_eq > 0.0 else 0.0
                ),
                "hull_per_spice_eq": (
                    round((float(hull) * multiplier["hull"]) / cost_spice_eq, 2) if cost_spice_eq > 0.0 else 0.0
                ),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
            }
        )
    return pd.DataFrame(rows)


SHIP_PLANET_COLUMNS: tuple[str, ...] = (
    "effective_build_time_s",
    "effective_build_time_text",
    "units_per_hour_effective",
    "energy_output_base",
    "energy_output_effective",
)


def planet_unit_rows(
    context: dict[str, Any],
    stationed: dict[str, int],
    unit_table: pd.DataFrame,
    planet_columns: tuple[str, ...],
) -> list[dict[str, Any]]:
    """Rows of a per-planet unit table: the units stationed on the planet and the unit columns that depend on it."""
    identity = planet_identity(context)
    return [
        {
            **identity,
            "key": unit["key"],
            "name": unit["name"],
            "stationed": max(0, int(stationed.get(unit["key"], 0) or 0)),
            **{column: unit[column] for column in planet_columns},
        }
        for unit in unit_table.to_dict("records")
    ]


def build_planet_ship_table(planets: list[PlanetState], contexts: list[dict[str, Any]]) -> pd.DataFrame:
    """One row per planet and ship: ships stationed there, build time in its Weapons factory, Solar cell output."""
    return pd.DataFrame(
        [
            row
            for planet, context in zip(planets, contexts)
            for row in planet_unit_rows(context, planet.ships, build_ship_table(context), SHIP_PLANET_COLUMNS)
        ]
    )


ships_df = build_ship_table(GENERIC_PLANET_CONTEXT)
planet_ships_df = build_planet_ship_table(empire_planet_states, PLANET_PARAM_CONTEXTS)

print(f"ships_df: {ships_df.shape} (generic planet), planet_ships_df: {planet_ships_df.shape} (each planet)\n")
print(
    ships_df[
        [
            "key",
            "name",
            "role",
            "cost_spice_eq",
            "base_build_time_s",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "speed_effective",
            "cargo",
            "fuel",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\nBuild time per planet (seconds), one column per ship:")
print(
    planet_ships_df.pivot(index=["slot_index", "coords"], columns="key", values="effective_build_time_s")[
        list(ships_df["key"])
    ].to_string()
)

ships_df: (15, 34) (generic planet), planet_ships_df: (165, 12) (each planet)

key              name                    role  cost_spice_eq  base_build_time_s  attack_effective  shield_effective  hull_effective  speed_effective   cargo  fuel                                                                prerequisites
 s1   Small transport         cargo_transport        2200.00            1700.00               2.0              10.0          6000.0           5400.0    6000    10                               b9 L2 (Weapons factory), r4 L4 (Rocket engine)
 s2   Large transport         cargo_transport        5200.00            4400.00               5.0              25.0         14000.0           9000.0   30000    15                               b9 L6 (Weapons factory), r4 L6 (Rocket engine)
 s3       Transmitter      armed_heavy_hauler       18333.33           14166.67             130.0              70.0         50000.0          11900.0  250000    25                            b9 L8 (Weap

### 6.4 Planetary Defenses (`d1`–`d11`) & Table Registry

`defenses_df` has one row per defense: unit cost, the base build time and the build time on a generic planet, base and effective hull, shield and attack (the same multipliers as the ships), whether the unit can dig in (`can_dig_in`, from `defenses.json`), and the ratios `attack_per_spice_eq` and `hull_plus_shield_per_spice_eq`.

`planet_defenses_df` has one row per planet and defense: the units `stationed` there, and the build time and hourly throughput in that planet's Weapons factory.

`GAME_PARAMS_TABLES` registers the twelve tables of Section 6 by name, and `game_params_summary_df` lists their shapes and first columns.

In [67]:
def build_defense_table(context: dict[str, Any]) -> pd.DataFrame:
    """Build `defenses_df`: cost, build time and combat stats of every planetary defense."""
    time_divisor = shipyard_time_divisor(
        context["b9_weapons_factory"], context["b17_microsystem"], context["speedup_factor"]
    )
    multiplier = combat_multipliers(context)

    rows: list[dict[str, Any]] = []
    for key in unit_keys(DEFENSE_SLUG_BY_KEY):
        slug = DEFENSE_SLUG_BY_KEY[key]
        info = game_data.DEFENSES[slug]
        stats = info.get("stats") or {}
        cost = full_cost(info.get("base_cost"))
        cost_spice_eq = float(game_data.spice_equivalent(cost))
        base_time = float(info.get("base_build_time_s") or 0.0)
        build_time = base_time / time_divisor

        hull, shield, attack = (int(stats.get(stat) or 0) for stat in ("hull", "shield", "attack"))
        hull_effective = float(hull) * multiplier["hull"]
        shield_effective = float(shield) * multiplier["shield"]
        attack_effective = float(attack) * multiplier["attack"]

        rows.append(
            {
                "defense_id": int(key[1:]),
                "key": key,
                "slug": slug,
                "name": str(info["name"]),
                "role": str(DEFENSE_METADATA[key]["role"]),
                "can_dig_in": bool(DEFENSE_METADATA[key]["can_dig_in"]),
                "pig_iron": int(cost["pig_iron"]),
                "metal": int(cost["metal"]),
                "kryptonite": int(cost["kryptonite"]),
                "spice": int(cost["spice"]),
                "cost_spice_eq": round(cost_spice_eq, 2),
                "points_value": round(cost_points(cost), 3),
                "base_build_time_s": round(base_time, 2),
                "effective_build_time_s": round(build_time, 2),
                "effective_build_time_text": fmt_duration(build_time),
                "units_per_hour_effective": round((3600.0 / build_time) if build_time > 0.0 else 0.0, 2),
                "hull_base": hull,
                "shield_base": shield,
                "attack_base": attack,
                "hull_effective": round(hull_effective, 1),
                "shield_effective": round(shield_effective, 1),
                "attack_effective": round(attack_effective, 1),
                "attack_per_spice_eq": round(attack_effective / cost_spice_eq, 4) if cost_spice_eq > 0.0 else 0.0,
                "hull_plus_shield_per_spice_eq": (
                    round((hull_effective + shield_effective) / cost_spice_eq, 2) if cost_spice_eq > 0.0 else 0.0
                ),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
            }
        )
    return pd.DataFrame(rows)


DEFENSE_PLANET_COLUMNS: tuple[str, ...] = (
    "effective_build_time_s",
    "effective_build_time_text",
    "units_per_hour_effective",
)


def build_planet_defense_table(planets: list[PlanetState], contexts: list[dict[str, Any]]) -> pd.DataFrame:
    """One row per planet and defense: units stationed there and the build time in its Weapons factory."""
    return pd.DataFrame(
        [
            row
            for planet, context in zip(planets, contexts)
            for row in planet_unit_rows(
                context, planet.defense, build_defense_table(context), DEFENSE_PLANET_COLUMNS
            )
        ]
    )


defenses_df = build_defense_table(GENERIC_PLANET_CONTEXT)
planet_defenses_df = build_planet_defense_table(empire_planet_states, PLANET_PARAM_CONTEXTS)

GAME_PARAMS_TABLES: dict[str, pd.DataFrame] = {
    "merchant_rates": merchant_rates_df,
    "planet_contexts": planet_contexts_df,
    "buildings_catalog": buildings_catalog_df,
    "buildings_levels": buildings_levels_df,
    "planet_buildings": planet_buildings_df,
    "technologies_catalog": technologies_catalog_df,
    "technologies_levels": technologies_levels_df,
    "planet_technologies": planet_technologies_df,
    "ships": ships_df,
    "planet_ships": planet_ships_df,
    "defenses": defenses_df,
    "planet_defenses": planet_defenses_df,
}
game_params_summary_df = pd.DataFrame(
    [
        {
            "table_name": name,
            "dataframe_var": f"{name}_df",
            "rows": int(table.shape[0]),
            "columns": int(table.shape[1]),
            "sample_columns": ", ".join(list(table.columns[:6])),
        }
        for name, table in GAME_PARAMS_TABLES.items()
    ]
)

print(
    f"defenses_df: {defenses_df.shape} (generic planet), "
    f"planet_defenses_df: {planet_defenses_df.shape} (each planet)\n"
)
print(
    defenses_df[
        [
            "key",
            "name",
            "role",
            "can_dig_in",
            "cost_spice_eq",
            "base_build_time_s",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "attack_per_spice_eq",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\nBuild time per planet (seconds), one column per defense:")
print(
    planet_defenses_df.pivot(index=["slot_index", "coords"], columns="key", values="effective_build_time_s")[
        list(defenses_df["key"])
    ].to_string()
)
print("\nGAME_PARAMS_TABLES:")
print(game_params_summary_df.to_string(index=False))

defenses_df: (11, 25) (generic planet), planet_defenses_df: (121, 10) (each planet)

key                  name             role  can_dig_in  cost_spice_eq  base_build_time_s  attack_effective  shield_effective  hull_effective  attack_per_spice_eq                                                                     prerequisites
 d1                Mortar planetary_turret        True         800.00            1000.00              35.0              15.0          2000.0               0.0437                                                           b9 L1 (Weapons factory)
 d2  Laser cannon battery planetary_turret        True       15166.67           11333.33            1000.0             400.0         40000.0               0.0659   b9 L8 (Weapons factory), r11 L10 (Targeting systems), r2 L8 (Energy technology)
 d3            Ion cannon planetary_turret        True        3466.67            1533.33             125.0             125.0         10000.0               0.0361                      

## 7. Technology Requirement Graph

A building, research, ship or defense can require other buildings and researches at given levels. This section keeps those requirements in memory as a directed acyclic graph, so questions such as "is this unlocked?" or "what has to be built first?" are answered without reading the catalog again.

### 7.1 `TechnologyRequirementGraph`

A node is an entity key (`b7`, `r2`, `s7`, `d10`). An edge `source -> target` with a level means that the target needs the source at that level. For example, the Weapons factory `b9` needs the Development center `b7` at level 3 and Energy technology `r2` at level 1.

The graph is built once from `game_data` and exposes:

- `nodes` — category, slug, name and highest catalog level of each entity;
- `prereqs_of[key]` and `unlocks_of[key]` — the direct requirements of an entity, and the entities that directly require it, as `(key, level)` pairs;
- `topological_order` and `topological_layer[key]` — an order in which every entity comes after its prerequisites, and each entity's depth in the tree. Kahn's algorithm produces both and fails if the requirements contain a cycle;
- `ancestors_of[key]` and `descendants_of[key]` — every direct and indirect prerequisite, and everything the entity helps to unlock.

The queries take the current building and research levels as arguments:

- `is_unlocked` and `direct_missing` — whether the direct requirements are met, and which ones are not;
- `transitive_requirements` — the minimum level of every direct and indirect prerequisite;
- `plan_unlock_sequence` — the ordered upgrades that unlock a target, with their cost and time.

In [68]:
class TechnologyRequirementGraph:
    """In-memory directed acyclic graph of the prerequisites of every building, research, ship and defense.

    Nodes are entity keys (`b7`, `r2`, `s7`, `d10`). An edge `source -> target` with a level means that the
    target needs the source at that level. Ships and defenses have no levels of their own.
    """

    PLAN_COLUMNS: tuple[str, ...] = (
        "step",
        "key",
        "name",
        "category",
        "topological_layer",
        "from_level",
        "to_level",
        "levels_added",
        "pig_iron",
        "metal",
        "kryptonite",
        "spice",
        "step_cost_spice_eq",
        "step_effective_time_s",
        "step_effective_time_text",
        "cum_cost_spice_eq",
        "cum_effective_time_s",
        "cum_effective_time_text",
    )

    def __init__(self) -> None:
        self.nodes: dict[str, dict[str, Any]] = {}
        self._requirements: dict[str, list[dict[str, Any]]] = {}
        self._base_times: dict[str, dict[int, float]] = {}
        for building_id in sorted(game_data.BUILDING_BY_ID):
            self._add_node(f"b{building_id}", "buildings", game_data.BUILDING_BY_ID[building_id])
        for key, info in research_catalog():
            self._add_node(key, "research", info)
        for key in unit_keys(SHIP_SLUG_BY_KEY):
            self._add_node(key, "ships", game_data.SHIPS[SHIP_SLUG_BY_KEY[key]])
        for key in unit_keys(DEFENSE_SLUG_BY_KEY):
            self._add_node(key, "defense", game_data.DEFENSES[DEFENSE_SLUG_BY_KEY[key]])

        self.prereqs_of, self.unlocks_of = self._link_requirements()
        self.topological_order, self.topological_layer = self._sort_topologically()
        self._topological_rank = {key: rank for rank, key in enumerate(self.topological_order)}
        self.ancestors_of, self.descendants_of = self._close_transitively()

    def __repr__(self) -> str:
        edges = sum(len(requirements) for requirements in self.prereqs_of.values())
        return f"TechnologyRequirementGraph(nodes={len(self.nodes)}, edges={edges})"

    def _add_node(self, key: str, category: str, info: dict[str, Any]) -> None:
        """Register one catalog entity with its requirements and its repaired base times."""
        levels = info.get("levels") or {}
        max_level = int(info.get("max_table_level") or len(levels))
        self.nodes[key] = {
            "key": key,
            "category": category,
            "entity_id": int(key[1:]),
            "slug": str(info["slug"]),
            "name": str(info["name"]),
            "max_level": max_level,
        }
        self._requirements[key] = list(info.get("requirements") or [])
        self._base_times[key] = build_monotonic_base_times(levels, max_level)

    def _link_requirements(
        self,
    ) -> tuple[dict[str, tuple[tuple[str, int], ...]], dict[str, tuple[tuple[str, int], ...]]]:
        """Turn each catalog requirement into an edge, indexed by target (`prereqs_of`) and source (`unlocks_of`)."""
        key_by_slug = {(node["category"], node["slug"]): key for key, node in self.nodes.items()}
        prereqs: dict[str, list[tuple[str, int]]] = {key: [] for key in self.nodes}
        unlocks: dict[str, list[tuple[str, int]]] = {key: [] for key in self.nodes}
        for target, requirements in self._requirements.items():
            for requirement in requirements:
                source = key_by_slug.get((str(requirement["category"]), str(requirement["slug"])))
                if source is None:
                    raise KeyError(f"Unresolved tech-tree requirement {requirement!r} on {target}")
                level = int(requirement["level"])
                prereqs[target].append((source, level))
                unlocks[source].append((target, level))
        return (
            {key: tuple(edges) for key, edges in prereqs.items()},
            {key: tuple(edges) for key, edges in unlocks.items()},
        )

    def _sort_topologically(self) -> tuple[tuple[str, ...], dict[str, int]]:
        """Kahn's algorithm: order the nodes so prerequisites come first, and record each node's dependency depth."""
        open_prereqs = {key: len(self.prereqs_of[key]) for key in self.nodes}
        layer = {key: 0 for key in self.nodes}
        queue: deque[str] = deque(key for key, count in open_prereqs.items() if count == 0)
        order: list[str] = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent, _level in self.unlocks_of[node]:
                layer[dependent] = max(layer[dependent], layer[node] + 1)
                open_prereqs[dependent] -= 1
                if open_prereqs[dependent] == 0:
                    queue.append(dependent)
        if len(order) != len(self.nodes):
            raise ValueError(f"Cycle detected in technology requirement graph: {sorted(set(self.nodes) - set(order))}")
        return tuple(order), layer

    def _close_transitively(self) -> tuple[dict[str, frozenset[str]], dict[str, frozenset[str]]]:
        """Collect each node's direct and indirect prerequisites (`ancestors_of`) and dependents (`descendants_of`)."""
        ancestors: dict[str, set[str]] = {key: set() for key in self.nodes}
        for node in self.topological_order:
            for prerequisite, _level in self.prereqs_of[node]:
                ancestors[node].add(prerequisite)
                ancestors[node].update(ancestors[prerequisite])
        descendants: dict[str, set[str]] = {key: set() for key in self.nodes}
        for node in reversed(self.topological_order):
            for dependent, _level in self.unlocks_of[node]:
                descendants[node].add(dependent)
                descendants[node].update(descendants[dependent])
        return (
            {key: frozenset(found) for key, found in ancestors.items()},
            {key: frozenset(found) for key, found in descendants.items()},
        )

    def current_level(self, key: str, buildings: dict[str, int], researches: dict[str, int]) -> int:
        """Level of `key` in the given state (ships and defenses have no level, so they return 0)."""
        category = self.nodes[key]["category"]
        if category == "buildings":
            return max(0, int(buildings.get(key, 0) or 0))
        if category == "research":
            return max(0, int(researches.get(key, 0) or 0))
        return 0

    def direct_missing(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
    ) -> list[dict[str, Any]]:
        """Direct prerequisites of `target_key` that the given state does not meet yet."""
        missing: list[dict[str, Any]] = []
        for source, required_level in self.prereqs_of[target_key]:
            level = self.current_level(source, buildings, researches)
            if level < required_level:
                missing.append(
                    {
                        "key": source,
                        "name": self.nodes[source]["name"],
                        "category": self.nodes[source]["category"],
                        "required_level": required_level,
                        "current_level": level,
                        "missing_levels": required_level - level,
                    }
                )
        return missing

    def is_unlocked(self, target_key: str, buildings: dict[str, int], researches: dict[str, int]) -> bool:
        """True when the given state meets every direct prerequisite of `target_key`."""
        return not self.direct_missing(target_key, buildings, researches)

    def transitive_requirements(self, target_key: str, *, include_target_level: int = 0) -> dict[str, int]:
        """Minimum level of every direct and indirect prerequisite of `target_key`, in topological order.

        When several requirement paths reach the same entity, the highest required level wins.
        `include_target_level` also demands that level of the target itself (buildings and researches only).
        """
        if target_key not in self.nodes:
            raise KeyError(f"Unknown node key: {target_key!r}")
        required: dict[str, int] = {}
        if include_target_level > 0 and self.nodes[target_key]["category"] in ("buildings", "research"):
            required[target_key] = int(include_target_level)
        for source, level in self.prereqs_of[target_key]:
            required[source] = max(required.get(source, 0), int(level))
        # Visiting nodes from the last in topological order to the first sees every dependent before its prerequisites.
        for node in reversed(self.topological_order):
            if node not in required:
                continue
            for source, level in self.prereqs_of[node]:
                if level > required.get(source, 0):
                    required[source] = int(level)
        return {key: required[key] for key in sorted(required, key=self._topological_rank.__getitem__)}

    def _level_cost_and_base_time(self, key: str, level: int) -> tuple[dict[str, Any], float]:
        """Cost and base time of taking building or research `key` to `level`."""
        node = self.nodes[key]
        base_time = float(self._base_times[key].get(level, 0.0))
        if node["category"] == "buildings":
            return game_data.building_cost(int(node["entity_id"]), level), base_time
        levels = game_data.RESEARCHES[node["slug"]].get("levels") or {}
        return full_cost((levels.get(level) or {}).get("cost")), base_time

    def plan_unlock_sequence(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
        *,
        speedup_factor: float,
        include_target_level: int = 0,
    ) -> pd.DataFrame:
        """Ordered upgrades that take the given state to the point where `target_key` is unlocked.

        One row per building or research that must be raised, in an order that respects every prerequisite.
        Times use the simulated state, so a Development center, Research lab or Microsystem accelerator
        raised by an earlier step shortens the later ones.
        """
        simulated_buildings = {str(key): max(0, int(level or 0)) for key, level in buildings.items()}
        simulated_researches = {str(key): max(0, int(level or 0)) for key, level in researches.items()}
        speed = max(0.1, float(speedup_factor))

        rows: list[dict[str, Any]] = []
        total_spice_eq = 0.0
        total_time = 0.0
        requirements = self.transitive_requirements(target_key, include_target_level=include_target_level)
        for key, required_level in requirements.items():
            node = self.nodes[key]
            current = self.current_level(key, simulated_buildings, simulated_researches)
            if current >= required_level:
                continue
            step_cost = {resource: 0 for resource in RESOURCE_KEYS}
            step_time = 0.0
            for level in range(current + 1, required_level + 1):
                cost, base_time = self._level_cost_and_base_time(key, level)
                for resource in RESOURCE_KEYS:
                    step_cost[resource] += int(cost.get(resource, 0))
                if node["category"] == "buildings":
                    divisor = building_time_divisor(
                        simulated_buildings.get("b7", 0), simulated_buildings.get("b17", 0), speed
                    )
                    simulated_buildings[key] = level
                else:
                    divisor = research_time_divisor(simulated_buildings.get("b8", 0), speed)
                    simulated_researches[key] = level
                step_time += base_time / divisor

            step_spice_eq = float(game_data.spice_equivalent(step_cost))
            total_spice_eq += step_spice_eq
            total_time += step_time
            rows.append(
                {
                    "step": len(rows) + 1,
                    "key": key,
                    "name": node["name"],
                    "category": node["category"],
                    "topological_layer": self.topological_layer[key],
                    "from_level": current,
                    "to_level": required_level,
                    "levels_added": required_level - current,
                    "pig_iron": step_cost["pig_iron"],
                    "metal": step_cost["metal"],
                    "kryptonite": step_cost["kryptonite"],
                    "spice": step_cost["spice"],
                    "step_cost_spice_eq": round(step_spice_eq, 2),
                    "step_effective_time_s": round(step_time, 2),
                    "step_effective_time_text": fmt_duration(step_time),
                    "cum_cost_spice_eq": round(total_spice_eq, 2),
                    "cum_effective_time_s": round(total_time, 2),
                    "cum_effective_time_text": fmt_duration(total_time),
                }
            )
        return pd.DataFrame(rows, columns=list(self.PLAN_COLUMNS))

### 7.2 Graph Tables

Builds the graph (`TECH_GRAPH`) and two tables that describe it for the reference colony:

- `tech_graph_nodes_df` — one row per entity in topological order: layer, number of direct requirements (`in_degree`) and of direct dependents (`out_degree`), number of indirect ones, current level, and whether it is unlocked now;
- `tech_graph_edges_df` — one row per requirement: source, required level, target, the source's current level, whether the requirement is met, and the levels still missing.

In [69]:
TECH_GRAPH = TechnologyRequirementGraph()

tech_graph_nodes_df = pd.DataFrame(
    [
        {
            "key": key,
            "category": TECH_GRAPH.nodes[key]["category"],
            "slug": TECH_GRAPH.nodes[key]["slug"],
            "name": TECH_GRAPH.nodes[key]["name"],
            "topological_layer": TECH_GRAPH.topological_layer[key],
            "in_degree": len(TECH_GRAPH.prereqs_of[key]),
            "out_degree": len(TECH_GRAPH.unlocks_of[key]),
            "transitive_prereq_count": len(TECH_GRAPH.ancestors_of[key]),
            "transitive_unlock_count": len(TECH_GRAPH.descendants_of[key]),
            "is_root_node": len(TECH_GRAPH.prereqs_of[key]) == 0,
            "current_level": TECH_GRAPH.current_level(key, colony_buildings, colony_researches),
            "unlocked_now": TECH_GRAPH.is_unlocked(key, colony_buildings, colony_researches),
        }
        for key in TECH_GRAPH.topological_order
    ]
)

tech_graph_edges_df = pd.DataFrame(
    [
        {
            "source_key": source,
            "source_name": TECH_GRAPH.nodes[source]["name"],
            "source_category": TECH_GRAPH.nodes[source]["category"],
            "required_level": required_level,
            "target_key": target,
            "target_name": TECH_GRAPH.nodes[target]["name"],
            "target_category": TECH_GRAPH.nodes[target]["category"],
            "target_layer": TECH_GRAPH.topological_layer[target],
            "source_current_level": TECH_GRAPH.current_level(source, colony_buildings, colony_researches),
            "satisfied_now": TECH_GRAPH.current_level(source, colony_buildings, colony_researches) >= required_level,
            "levels_missing": max(
                0, required_level - TECH_GRAPH.current_level(source, colony_buildings, colony_researches)
            ),
        }
        for target in TECH_GRAPH.topological_order
        for source, required_level in TECH_GRAPH.prereqs_of[target]
    ]
)

print(
    f"TECH_GRAPH: {len(TECH_GRAPH.nodes)} nodes, {len(tech_graph_edges_df)} requirement edges, "
    f"deepest layer {max(TECH_GRAPH.topological_layer.values())}\n"
)
print("Entities that unlock the most others:")
print(
    tech_graph_nodes_df.sort_values(["transitive_unlock_count", "out_degree"], ascending=[False, False])
    .head(12)[
        [
            "key",
            "name",
            "category",
            "topological_layer",
            "in_degree",
            "out_degree",
            "transitive_unlock_count",
            "current_level",
            "unlocked_now",
        ]
    ]
    .to_string(index=False)
)

TECH_GRAPH: 62 nodes, 101 requirement edges, deepest layer 5

Entities that unlock the most others:
key                name  category  topological_layer  in_degree  out_degree  transitive_unlock_count  current_level  unlocked_now
 b7  Development center buildings                  0          0           4                       51              9          True
 b8        Research lab buildings                  1          1          12                       49              3          True
 r2   Energy technology  research                  2          1          13                       40              4          True
 b9     Weapons factory buildings                  3          2          28                       28              4          True
 r3     Engine research  research                  2          1           3                       18              6          True
 r9   Shield technology  research                  3          2           4                        7              0     

### 7.3 Unlock Plans

`plan_unlock_sequence` lists the upgrades still missing before a target is unlocked, in an order that respects every prerequisite, with the cost and time of each step.

- When several requirement paths reach the same entity, the highest level wins and is counted once. The Phoenix (`s7`) needs the Weapons factory `b9` at level 7, which needs the Development center `b7` at level 3. It also needs Defense systems `r10` at level 5, which needs the Research lab `b8` at level 5, which needs `b7` at level 2. The plan raises `b7` to level 3, once.
- Times follow the simulated state: after a step raises the Development center, the Research lab or the Microsystem accelerator, the later steps are faster.

Tables:

- `unlock_targets_summary_df` — for every entity: whether it is unlocked, the upgrades and levels still missing, their total cost and time, and the ordered chain of upgrades;
- `phoenix_unlock_plan_df` — the plan for the Phoenix from the reference colony;
- `cold_phoenix_unlock_plan_df` — the same plan from an empty account;
- `TECH_GRAPH_TABLES` — registry of the five tables of Section 7.

In [70]:
def plan_unlock_from_colony(target_key: str) -> pd.DataFrame:
    """Unlock plan for `target_key` starting from the reference colony's current buildings and researches."""
    return TECH_GRAPH.plan_unlock_sequence(
        target_key,
        colony_buildings,
        colony_researches,
        speedup_factor=GAME_PARAM_CONTEXT["speedup_factor"],
    )


def unlock_summary_row(target_key: str) -> dict[str, Any]:
    """One `unlock_targets_summary_df` row: what is still missing before `target_key` is unlocked."""
    node = TECH_GRAPH.nodes[target_key]
    plan = plan_unlock_from_colony(target_key)
    total_time = float(plan["step_effective_time_s"].sum()) if not plan.empty else 0.0
    return {
        "key": target_key,
        "name": node["name"],
        "category": node["category"],
        "topological_layer": TECH_GRAPH.topological_layer[target_key],
        "unlocked_now": bool(plan.empty),
        "missing_upgrade_nodes": int(len(plan)),
        "total_levels_to_add": int(plan["levels_added"].sum()) if not plan.empty else 0,
        "unlock_cost_spice_eq": round(float(plan["step_cost_spice_eq"].sum()) if not plan.empty else 0.0, 2),
        "unlock_effective_time_s": round(total_time, 2),
        "unlock_effective_time_text": fmt_duration(total_time) if total_time > 0.0 else "Ready",
        "ordered_prerequisite_chain": (
            " -> ".join(
                f"{step['key']} L{int(step['from_level'])}->{int(step['to_level'])}"
                for step in plan.to_dict("records")
            )
            if not plan.empty
            else "Unlocked"
        ),
    }


unlock_targets_summary_df = pd.DataFrame([unlock_summary_row(key) for key in TECH_GRAPH.topological_order])
phoenix_unlock_plan_df = plan_unlock_from_colony("s7")
cold_phoenix_unlock_plan_df = TECH_GRAPH.plan_unlock_sequence(
    "s7", {}, {}, speedup_factor=GAME_PARAM_CONTEXT["speedup_factor"]
)

TECH_GRAPH_TABLES: dict[str, pd.DataFrame] = {
    "tech_graph_nodes": tech_graph_nodes_df,
    "tech_graph_edges": tech_graph_edges_df,
    "unlock_targets_summary": unlock_targets_summary_df,
    "phoenix_unlock_plan": phoenix_unlock_plan_df,
    "cold_phoenix_unlock_plan": cold_phoenix_unlock_plan_df,
}

print(
    f"{int(unlock_targets_summary_df['unlocked_now'].sum())} entities are unlocked, "
    f"{int((~unlock_targets_summary_df['unlocked_now']).sum())} are locked\n"
)
print("Cheapest locked entities to unlock:")
print(
    unlock_targets_summary_df[~unlock_targets_summary_df["unlocked_now"]]
    .sort_values(["unlock_cost_spice_eq", "topological_layer"])
    .head(15)[
        [
            "key",
            "name",
            "category",
            "topological_layer",
            "total_levels_to_add",
            "unlock_cost_spice_eq",
            "unlock_effective_time_text",
            "ordered_prerequisite_chain",
        ]
    ]
    .to_string(index=False)
)
print("\nUnlock plan for the Phoenix (s7) from the reference colony:")
if phoenix_unlock_plan_df.empty:
    print("already unlocked")
else:
    print(
        phoenix_unlock_plan_df[
            [
                "step",
                "key",
                "name",
                "from_level",
                "to_level",
                "step_cost_spice_eq",
                "step_effective_time_text",
                "cum_cost_spice_eq",
                "cum_effective_time_text",
            ]
        ].to_string(index=False)
    )

30 entities are unlocked, 32 are locked

Cheapest locked entities to unlock:
key                  name  category  topological_layer  total_levels_to_add  unlock_cost_spice_eq unlock_effective_time_text                    ordered_prerequisite_chain
 s5               Warship     ships                  4                    2               2500.00                     1m 40s                                      r8 L0->2
 d8 Light graviton cannon   defense                  4                    2               3260.00                     2m 19s                                     r13 L0->2
 r9     Shield technology  research                  3                    1               4960.00                        27s                                      b8 L3->4
d10     Small shield dome   defense                  4                    3               7260.00                     2m 16s                          b8 L3->4 -> r9 L0->2
b20         Tunnel system buildings                  4              

## 8. Planet Sizes & States

The size (used and total fields) and the state (resources, storage, energy, buildings, ships, defenses) of every planet of the roster that Section 6 loaded: the planets owned now, plus a generic planet for every colony that can still be founded.

### 8.1 Owned Planets

`live_planets_df` has one summary row per owned planet of `live_planet_states`: size, resources with their Spice equivalent, storage capacities, energy balance, and the totals of building levels, ships and defenses.

In [71]:
def planet_state_to_row(slot_index: int, planet: PlanetState, *, is_empty_slot: bool) -> dict[str, Any]:
    """One row of the planet tables: size, resources, storage, energy and totals of a planet or empty slot."""
    fields_max = int(planet.fields_max)
    fields_used = int(planet.fields_used)
    resources = {resource: int(getattr(planet.resources, resource)) for resource in RESOURCE_KEYS}
    return {
        "slot_index": slot_index,
        "planet_id": int(planet.planet_id),
        "status": "empty_colony_slot" if is_empty_slot else "colonized",
        "is_empty_slot": bool(is_empty_slot),
        "name": planet.name,
        "coords": planet.coords,
        "planet_type": planet.planet_type,
        "temperature_c": int(planet.temperature),
        "fields_used": fields_used,
        "fields_max": fields_max,
        "fields_remaining": int(planet.fields_remaining),
        "field_utilization_pct": round((100.0 * float(fields_used) / float(fields_max)) if fields_max > 0 else 0.0, 2),
        **resources,
        "resources_spice_eq": round(float(game_data.spice_equivalent(resources)), 2),
        **{f"storage_{resource}": int(getattr(planet.storage, resource)) for resource in RESOURCE_KEYS},
        "energy_produced": int(planet.energy.produced),
        "energy_used": int(planet.energy.used),
        "energy_net": int(planet.energy.produced) - int(planet.energy.used),
        "total_building_levels": int(sum(planet.buildings.values())),
        "total_ships": int(sum(planet.ships.values())),
        "total_defenses": int(sum(planet.defense.values())),
    }


live_planets_df = pd.DataFrame(
    [
        planet_state_to_row(index, planet, is_empty_slot=False)
        for index, planet in enumerate(live_planet_states, start=1)
    ]
)

print(f"{len(live_planet_states)} owned planet(s); the empire holds up to {MAX_EMPIRE_PLANETS}")
if not live_planets_df.empty:
    print(
        live_planets_df[
            [
                "planet_id",
                "name",
                "coords",
                "planet_type",
                "temperature_c",
                "fields_used",
                "fields_max",
                "fields_remaining",
                "field_utilization_pct",
                "resources_spice_eq",
                "energy_net",
                "total_building_levels",
                "total_ships",
            ]
        ].to_string(index=False)
    )

1 owned planet(s); the empire holds up to 11
 planet_id   name    coords planet_type  temperature_c  fields_used  fields_max  fields_remaining  field_utilization_pct  resources_spice_eq  energy_net  total_building_levels  total_ships
     12476 Colony 02:380:12      normal             31          137         230                93                  59.57           668897.67         965                    137           75


### 8.2 Full Empire Roster with Empty Colony Slots

The same tables for the whole roster in `empire_planet_states`: the owned planets followed by the generic planets, each an empty colony slot with `EMPTY_COLONY_SIZE_FIELDS` free fields.

- `empire_planets_df` — the summary columns of `live_planets_df` for every slot, with `status` and `is_empty_slot`;
- `empire_planet_buildings_df` — the level of every building on every slot;
- `planet_sizes_arr`, `planet_fields_used_arr`, `planet_fields_remaining_arr` — total, used and free fields per slot, as `numpy` arrays;
- `EMPIRE_PLANET_STATE_SUMMARY` — the totals; `EMPIRE_PLANET_TABLES` — registry of the three planet tables.

In [72]:
def build_planet_tables(planets: list[PlanetState], owned_count: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Summary table and building-level table of a roster whose first `owned_count` planets are owned."""
    building_keys = [f"b{building_id}" for building_id in sorted(game_data.BUILDING_BY_ID)]
    planet_rows: list[dict[str, Any]] = []
    building_rows: list[dict[str, Any]] = []
    for slot_index, planet in enumerate(planets, start=1):
        is_empty_slot = slot_index > owned_count
        planet_rows.append(planet_state_to_row(slot_index, planet, is_empty_slot=is_empty_slot))
        building_rows.append(
            {
                "slot_index": slot_index,
                "planet_id": int(planet.planet_id),
                "coords": planet.coords,
                "status": "empty_colony_slot" if is_empty_slot else "colonized",
                "fields_used": int(planet.fields_used),
                "fields_max": int(planet.fields_max),
                **{key: max(0, int(planet.buildings.get(key, 0) or 0)) for key in building_keys},
            }
        )
    return pd.DataFrame(planet_rows), pd.DataFrame(building_rows)


empire_planets_df, empire_planet_buildings_df = build_planet_tables(empire_planet_states, len(live_planet_states))

planet_sizes_arr: np.ndarray = empire_planets_df["fields_max"].to_numpy(dtype=np.int64)
planet_fields_used_arr: np.ndarray = empire_planets_df["fields_used"].to_numpy(dtype=np.int64)
planet_fields_remaining_arr: np.ndarray = empire_planets_df["fields_remaining"].to_numpy(dtype=np.int64)

EMPIRE_PLANET_STATE_SUMMARY: dict[str, Any] = {
    "max_empire_planets": int(MAX_EMPIRE_PLANETS),
    "empty_colony_size_fields": int(EMPTY_COLONY_SIZE_FIELDS),
    "colonized_planets": int((~empire_planets_df["is_empty_slot"]).sum()),
    "empty_colony_slots": int(empire_planets_df["is_empty_slot"].sum()),
    "total_empire_slots": int(len(empire_planets_df)),
    "total_fields_capacity": int(np.sum(planet_sizes_arr)),
    "total_fields_used": int(np.sum(planet_fields_used_arr)),
    "total_fields_remaining": int(np.sum(planet_fields_remaining_arr)),
    "mean_planet_size_fields": float(np.mean(planet_sizes_arr)) if planet_sizes_arr.size > 0 else 0.0,
}

EMPIRE_PLANET_TABLES: dict[str, pd.DataFrame] = {
    "live_planets": live_planets_df,
    "empire_planets": empire_planets_df,
    "empire_planet_buildings": empire_planet_buildings_df,
}

print(
    f"{EMPIRE_PLANET_STATE_SUMMARY['colonized_planets']} owned + "
    f"{EMPIRE_PLANET_STATE_SUMMARY['empty_colony_slots']} empty {EMPTY_COLONY_SIZE_FIELDS}-field slots = "
    f"{EMPIRE_PLANET_STATE_SUMMARY['total_empire_slots']} planets; "
    f"fields: {EMPIRE_PLANET_STATE_SUMMARY['total_fields_used']:,} used of "
    f"{EMPIRE_PLANET_STATE_SUMMARY['total_fields_capacity']:,}\n"
)
print(
    empire_planets_df[
        [
            "slot_index",
            "planet_id",
            "status",
            "name",
            "coords",
            "temperature_c",
            "fields_used",
            "fields_max",
            "fields_remaining",
            "resources_spice_eq",
            "energy_net",
            "total_building_levels",
        ]
    ].to_string(index=False)
)
print("\nBuilding levels per planet (b1-b14):")
print(
    empire_planet_buildings_df[
        ["slot_index", "coords", "status", "fields_used", "fields_max", *(f"b{number}" for number in range(1, 15))]
    ].to_string(index=False)
)

1 owned + 10 empty 300-field slots = 11 planets; fields: 137 used of 3,230

 slot_index  planet_id            status                 name    coords  temperature_c  fields_used  fields_max  fields_remaining  resources_spice_eq  energy_net  total_building_levels
          1      12476         colonized               Colony 02:380:12             31          137         230                93           668897.67         965                    137
          2         -1 empty_colony_slot  Empty Colony Slot 1  EMPTY:01              0            0         300               300                0.00           0                      0
          3         -2 empty_colony_slot  Empty Colony Slot 2  EMPTY:02              0            0         300               300                0.00           0                      0
          4         -3 empty_colony_slot  Empty Colony Slot 3  EMPTY:03              0            0         300               300                0.00           0                      0

## 9. Maximum Research Levels in the Universe

The highest level of every research that anyone in the universe is known to have. The configuration optimizer, `config_optimizer.ipynb`, plans with these levels as its reference.

The bot keeps them in the `universe_research_max` table of its database. The notebook reads that ledger from the temporary database copy and combines it with the sources it can check itself:

- **Server records** (`server_record_level`) — the level that the game's statistics pages list for a research, once the bot has synchronized them into the ledger;
- **Spy reports** (`spied_max_level`) — the highest level found in the archived reports of `spy_reports`;
- **Your own research** (`own_level`) — the levels of the newest planet snapshot.

`universe_max_level` is the highest of these and of the maximum that the ledger stored earlier; the ledger never lowers a level. `max_source` names the source that holds the maximum (`server_record`, `spy_report`, `own_level`, `ledger` or `none`), and `holder` the player, when one is known. A level of 0 means that no source reported the research.

Only the bot fills the ledger; this notebook reads it. Until the bot has synchronized the server records, or spied a player whose report lists researches, every maximum is your own level.

- `universe_research_sources` — the raw inputs: the ledger rows, the highest spied level per research, and the number of spy reports that list researches;
- `universe_research_df` — one row per research: the level from each source, `universe_max_level`, `max_source`, `holder`, the distance from your own level (`gap_to_max`, `own_percent_of_max`), the last level of the catalog, and the time of the ledger's last update. A research that the ledger tracks but the catalog does not list comes last;
- `UNIVERSE_RESEARCH_MAX_LEVELS` — `{research key: universe_max_level}`;
- `UNIVERSE_RESEARCH_SUMMARY` — the number of ledger rows, of spy reports with researches, and of researches known from each source or above your own level.

In [73]:
SQL_UNIVERSE_RESEARCH_LEDGER = """
SELECT
    research_key,
    research_id,
    slug,
    name,
    own_level,
    spied_max_level,
    server_record_level,
    universe_max_level,
    holder,
    holder_coords,
    updated_ts
FROM universe_research_max
ORDER BY research_id ASC, slug ASC
"""

SQL_SPY_REPORT_RESEARCHES = """
SELECT target_coords, owner, researches_json
FROM spy_reports
WHERE researches_json != '' AND researches_json != '{}'
ORDER BY ts ASC, id ASC
"""


def research_key_aliases() -> dict[str, str]:
    """Key, slug and name of every catalog research in lower case, each mapped to the key (`r1`, `r2`, ...)."""
    aliases: dict[str, str] = {}
    for key, info in research_catalog():
        for alias in (key, str(info["slug"]), str(info["name"])):
            aliases[alias.lower()] = key
    return aliases


def load_universe_research_sources(db_path: Path) -> dict[str, Any]:
    """Read what the bot recorded about research levels in the universe from a temporary copy of `db_path`.

    `ledger_rows` are the rows of the bot's `universe_research_max` table. `spied_levels` holds, per research,
    the highest level found in the archived spy reports and the planet it was seen on. Both stay empty without
    a database.
    """
    sources: dict[str, Any] = {"ledger_rows": [], "spied_levels": {}, "spy_reports_with_research": 0}
    if not has_database(db_path):
        return sources
    aliases = research_key_aliases()
    spied_levels: dict[str, dict[str, Any]] = sources["spied_levels"]
    with open_readonly_db_copy(db_path) as conn:
        sources["ledger_rows"] = [dict(row) for row in conn.execute(SQL_UNIVERSE_RESEARCH_LEDGER)]
        for report in conn.execute(SQL_SPY_REPORT_RESEARCHES):
            levels = {
                aliases[name.strip().lower()]: level
                for name, level in parse_level_map(report["researches_json"]).items()
                if level > 0 and name.strip().lower() in aliases
            }
            if levels:
                sources["spy_reports_with_research"] += 1
            for key, level in levels.items():
                # Reports are read oldest first, so the newest sighting of the highest level is kept.
                if level >= spied_levels.get(key, {}).get("level", 0):
                    spied_levels[key] = {
                        "level": level,
                        "holder": str(report["owner"] or ""),
                        "holder_coords": str(report["target_coords"] or ""),
                    }
    return sources


def universe_research_row(
    key: str,
    info: dict[str, Any] | None,
    ledger_row: dict[str, Any],
    spied: dict[str, Any],
    own_level: int,
) -> dict[str, Any]:
    """One `universe_research_df` row: the levels each source reports for a research and the maximum of them."""
    levels = (info or {}).get("levels") or {}
    own = max(int(own_level), int(ledger_row.get("own_level") or 0))
    spied_level = max(int(spied.get("level") or 0), int(ledger_row.get("spied_max_level") or 0))
    server_record = int(ledger_row.get("server_record_level") or 0)
    ledger_max = int(ledger_row.get("universe_max_level") or 0)
    universe_max = max(own, spied_level, server_record, ledger_max)

    if universe_max == 0:
        max_source = "none"
    elif server_record == universe_max:
        max_source = "server_record"
    elif spied_level == universe_max:
        max_source = "spy_report"
    elif own == universe_max:
        max_source = "own_level"
    else:  # the ledger never lowers a maximum, so it can remember more than today's sources show
        max_source = "ledger"
    if max_source == "spy_report" and int(spied.get("level") or 0) == universe_max:
        holder, holder_coords = str(spied["holder"]), str(spied["holder_coords"])
    elif max_source != "own_level" and ledger_max == universe_max > 0:
        holder, holder_coords = str(ledger_row.get("holder") or ""), str(ledger_row.get("holder_coords") or "")
    else:
        holder, holder_coords = "", ""

    updated_ts = float(ledger_row.get("updated_ts") or 0.0)
    return {
        "research_id": int(key[1:]) if info is not None else int(ledger_row.get("research_id") or 0),
        "key": key,
        "slug": str(info["slug"]) if info is not None else str(ledger_row.get("slug") or key),
        "name": str(info["name"]) if info is not None else str(ledger_row.get("name") or key),
        "own_level": own,
        "spied_max_level": spied_level,
        "server_record_level": server_record,
        "universe_max_level": universe_max,
        "max_source": max_source,
        "holder": holder,
        "holder_coords": holder_coords,
        "gap_to_max": universe_max - own,
        "own_percent_of_max": round(100.0 * float(own) / float(universe_max), 1) if universe_max > 0 else 0.0,
        "catalog_max_level": int((info or {}).get("max_table_level") or len(levels)),
        "ledger_updated_utc": (
            datetime.datetime.fromtimestamp(updated_ts, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            if updated_ts > 0.0
            else ""
        ),
    }


def build_universe_research_table(sources: dict[str, Any], own_levels: dict[str, int]) -> pd.DataFrame:
    """One row per research of the catalog, followed by any other research the bot's ledger tracks."""
    catalog = dict(research_catalog())
    ledger = {str(row["research_key"]): row for row in sources["ledger_rows"]}
    spied_levels = sources["spied_levels"]
    return pd.DataFrame(
        [
            universe_research_row(
                key,
                catalog.get(key),
                ledger.get(key, {}),
                spied_levels.get(key, {}),
                max(0, int(own_levels.get(key, 0) or 0)),
            )
            for key in (*catalog, *(key for key in ledger if key not in catalog))
        ]
    )


universe_research_sources: dict[str, Any] = load_universe_research_sources(LIVE_DB_PATH)
universe_research_df = build_universe_research_table(universe_research_sources, colony_researches)

# The highest known level of every research in the universe: the reference levels of the configuration optimizer.
UNIVERSE_RESEARCH_MAX_LEVELS: dict[str, int] = {
    str(key): int(level) for key, level in zip(universe_research_df["key"], universe_research_df["universe_max_level"])
}
UNIVERSE_RESEARCH_SUMMARY: dict[str, Any] = {
    "researches": int(len(universe_research_df)),
    "ledger_rows": len(universe_research_sources["ledger_rows"]),
    "spy_reports_with_research": int(universe_research_sources["spy_reports_with_research"]),
    "researches_with_server_record": int((universe_research_df["server_record_level"] > 0).sum()),
    "researches_seen_in_spy_reports": int((universe_research_df["spied_max_level"] > 0).sum()),
    "researches_above_own_level": int((universe_research_df["gap_to_max"] > 0).sum()),
    "highest_level": int(universe_research_df["universe_max_level"].max()),
}

print(
    f"{UNIVERSE_RESEARCH_SUMMARY['ledger_rows']} ledger row(s), "
    f"{UNIVERSE_RESEARCH_SUMMARY['spy_reports_with_research']} spy report(s) with research levels, "
    f"{UNIVERSE_RESEARCH_SUMMARY['researches_with_server_record']} server record(s); "
    f"{UNIVERSE_RESEARCH_SUMMARY['researches_above_own_level']} research(es) known above your own level\n"
)
if UNIVERSE_RESEARCH_SUMMARY["researches_above_own_level"] == 0:
    print("No other player's research is recorded yet, so every universe maximum is your own level.\n")
print(
    universe_research_df[
        [
            "key",
            "name",
            "own_level",
            "spied_max_level",
            "server_record_level",
            "universe_max_level",
            "max_source",
            "holder",
            "gap_to_max",
            "own_percent_of_max",
            "catalog_max_level",
        ]
    ].to_string(index=False)
)

16 ledger row(s), 0 spy report(s) with research levels, 13 server record(s); 13 research(es) known above your own level

                key                name  own_level  spied_max_level  server_record_level  universe_max_level    max_source   holder  gap_to_max  own_percent_of_max  catalog_max_level
                 r1   Mining technology          9                0                   30                  30 server_record ventkx86          21                30.0                 40
                 r2   Energy technology          4                0                   28                  28 server_record ventkx86          24                14.3                 40
                 r3     Engine research          6                0                   20                  20 server_record ventkx86          14                30.0                 35
                 r4       Rocket engine          8                0                   26                  26 server_record ventkx86          18    

## 10. Print Every Generated Data Set (One Cell per Section)

Each cell below prints, in full, everything the matching cell of Sections 1 to 9 generated: every row and column of each `pandas` table, every element of each `numpy` array, and every dictionary and scalar. Nothing is shortened with `...`, large numbers are written out instead of switching to scientific notation, and every printout starts with the date and time at which it was printed.

- Cells that only install packages, import modules, or define functions, classes or SQL text generate no data, so they have no print cell.
- The `SQL_*` query texts and the `*_COLUMNS` tuples are constants of the source, not generated data, and are not repeated.
- All printed text passes through `redact_sensitive`, so a session token can never appear in a printout.

### 10.1 Section 1.1 Data — Workspace Paths & Shared Constants

`REPO_ROOT`, `ACTIVE_DATA_ROOT`, `LIVE_DB_PATH`, `METADATA_DIR` and `RESOURCE_KEYS`.

In [74]:
print_section_data(
    "Section 1.1 data: workspace paths and shared constants",
    [
        ("REPO_ROOT", REPO_ROOT),
        ("ACTIVE_DATA_ROOT", ACTIVE_DATA_ROOT),
        ("LIVE_DB_PATH", LIVE_DB_PATH),
        ("METADATA_DIR", METADATA_DIR),
        ("RESOURCE_KEYS", RESOURCE_KEYS),
    ],
)

Section 1.1 data: workspace paths and shared constants
Printed: 2026-11-17 09:59:32 -0300 | 5 item(s)
[REPO_ROOT] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer')
----------------------------------------------------------------------------------------------------
[ACTIVE_DATA_ROOT] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer')
----------------------------------------------------------------------------------------------------
[LIVE_DB_PATH] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/state/spaceinvasion.db')
----------------------------------------------------------------------------------------------------
[METADATA_DIR] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/notebooks/metadata')
--------------------------------------------------------------------

### 10.2 Section 1.6 Data — Static Metadata

The content of the JSON files as loaded: `BUILDING_METADATA`, `TECHNOLOGY_METADATA`, `TECHNOLOGY_COST_SCALE_FACTOR`, `SHIP_METADATA`, `DEFENSE_METADATA`, `RESOURCE_METADATA` and `PLANNER_SETTINGS`.

In [75]:
print_section_data(
    "Section 1.6 data: static metadata",
    [
        ("BUILDING_METADATA", BUILDING_METADATA),
        ("TECHNOLOGY_METADATA", TECHNOLOGY_METADATA),
        ("TECHNOLOGY_COST_SCALE_FACTOR", TECHNOLOGY_COST_SCALE_FACTOR),
        ("SHIP_METADATA", SHIP_METADATA),
        ("DEFENSE_METADATA", DEFENSE_METADATA),
        ("RESOURCE_METADATA", RESOURCE_METADATA),
        ("PLANNER_SETTINGS", PLANNER_SETTINGS),
    ],
)

Section 1.6 data: static metadata
Printed: 2026-11-17 09:59:32 -0300 | 7 item(s)
[BUILDING_METADATA] dict, len=21
{
  "b1": {"name": "Pig iron mine", "category": "resource_mine", "cost_scale_factor": 1.51, "primary_effect": "Mines Pig iron (+2%/lvl r1); consumes energy"},
  "b2": {"name": "Blast furnace", "category": "smelter", "cost_scale_factor": 1.51, "primary_effect": "Smelts Pig iron -> Metal at 2:1 loss (NOT boosted by r1); consumes energy"},
  "b3": {"name": "Kryptonite mine", "category": "resource_mine", "cost_scale_factor": 1.51, "primary_effect": "Mines Kryptonite (+2%/lvl r1); consumes energy"},
  "b4": {"name": "Spice mine", "category": "resource_mine", "cost_scale_factor": 1.51, "primary_effect": "Mines Spice on planet-temperature curve (+2%/lvl r1); consumes energy"},
  "b5": {"name": "Fusion plant", "category": "power_plant", "cost_scale_factor": 1.51, "primary_effect": "Fuel-free planetary power (+1%/lvl r2); 1 field/level"},
  "b6": {"name": "Thermosolar plant", "categ

### 10.3 Section 2.3 Data — Attack Telemetry

`telemetry_source`, `origin_planet_coords`, `fleet_slot_capacity`, `colony_buildings`, `colony_researches`, the other entries of `telemetry_bundle`, `attacks_df`, `recycles_df` and `attack_dispatch_ts_arr`. The three row lists of `telemetry_bundle` are not printed separately: they are the rows of the two tables and the values of the array.

In [76]:
print_section_data(
    "Section 2.3 data: attack telemetry",
    [
        ("telemetry_source", telemetry_source),
        ("origin_planet_coords", origin_planet_coords),
        ("fleet_slot_capacity", fleet_slot_capacity),
        ("colony_buildings", colony_buildings),
        ("colony_researches", colony_researches),
        (
            "telemetry_bundle (every other key; its three row lists are the tables and array below)",
            {
                key: value
                for key, value in telemetry_bundle.items()
                if key
                not in ("records", "recycle_rows", "dispatch_timestamps", "colony_buildings", "colony_researches")
            },
        ),
        ("attacks_df", attacks_df),
        ("recycles_df", recycles_df),
        ("attack_dispatch_ts_arr", attack_dispatch_ts_arr),
    ],
)

Section 2.3 data: attack telemetry
Printed: 2026-11-17 09:59:32 -0300 | 9 item(s)
[telemetry_source] str = 'sqlite:mer_joined(battle_reports+raid_history+spy_targets)'
----------------------------------------------------------------------------------------------------
[origin_planet_coords] str = '02:380:12'
----------------------------------------------------------------------------------------------------
[fleet_slot_capacity] int = 7
----------------------------------------------------------------------------------------------------
[colony_buildings] dict, len=19
{
  "b1": 21,
  "b10": 6,
  "b11": 5,
  "b12": 3,
  "b13": 4,
  "b14": 3,
  "b15": 0,
  "b16": 0,
  "b17": 0,
  "b18": 0,
  "b2": 20,
  "b21": 0,
  "b3": 21,
  "b4": 15,
  "b5": 23,
  "b6": 0,
  "b7": 9,
  "b8": 3,
  "b9": 4
}
----------------------------------------------------------------------------------------------------
[colony_researches] dict, len=15
{
  "r1": 9,
  "r10": 0,
  "r11": 0,
  "r12": 0,
  "r13": 0,
  "r

### 10.4 Section 3 Data — Attack Resource Means & Standard Deviations

`METRIC_KEYS`, `attack_resource_stats_df`, the five dictionaries of means and standard deviations per resource, the mean and standard deviation of the recycler harvests, and `winning_attacks_df`.

In [77]:
print_section_data(
    "Section 3 data: attack resource means and standard deviations",
    [
        ("METRIC_KEYS", METRIC_KEYS),
        ("attack_resource_stats_df", attack_resource_stats_df),
        ("attack_resource_mean", attack_resource_mean),
        ("attack_resource_std", attack_resource_std),
        ("attack_resource_pop_std", attack_resource_pop_std),
        ("winning_attack_resource_mean", winning_attack_resource_mean),
        ("winning_attack_resource_std", winning_attack_resource_std),
        ("recycle_spice_eq_mean", recycle_spice_eq_mean),
        ("recycle_spice_eq_std", recycle_spice_eq_std),
        ("winning_attacks_df", winning_attacks_df),
    ],
)

Section 3 data: attack resource means and standard deviations
Printed: 2026-11-17 09:59:32 -0300 | 10 item(s)
[METRIC_KEYS] tuple, len=7
["pig_iron", "metal", "kryptonite", "spice", "spice_eq", "net_spice_eq", "debris_spice_eq"]
----------------------------------------------------------------------------------------------------
[attack_resource_stats_df] pandas.DataFrame, shape=(7, 9)
         metric  samples_all     mean_all sample_std_all  pop_std_all  samples_wins   mean_wins sample_std_wins pop_std_wins
       pig_iron            7  4065.857143    4575.737880  4236.310100             6 4743.500000     4611.677992  4209.866773
          metal            7  5377.285714    5413.949443  5012.343213             6 6273.500000     5331.616819  4867.077999
     kryptonite            7  5705.142857    5930.124547  5490.228500             6 6656.000000     5882.598609  5370.053259
          spice            7  2328.000000    2527.637237  2340.137359             6 2716.000000     2530.251055 

### 10.5 Section 4 Data — Attack Flight & Cadence Times

`attack_time_stats_df`, the nine unrounded means and standard deviations, and the arrays `one_way_times_arr`, `round_trip_times_arr` and `inter_attack_intervals_arr`.

In [78]:
print_section_data(
    "Section 4 data: attack flight and cadence times",
    [
        ("attack_time_stats_df", attack_time_stats_df),
        (
            "exact time statistics in seconds",
            {
                "attack_one_way_time_mean_sec": attack_one_way_time_mean_sec,
                "attack_one_way_time_std_sec": attack_one_way_time_std_sec,
                "attack_one_way_time_pop_std_sec": attack_one_way_time_pop_std_sec,
                "attack_time_mean_sec": attack_time_mean_sec,
                "attack_time_std_sec": attack_time_std_sec,
                "attack_time_pop_std_sec": attack_time_pop_std_sec,
                "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
                "inter_attack_time_std_sec": inter_attack_time_std_sec,
                "inter_attack_time_pop_std_sec": inter_attack_time_pop_std_sec,
            },
        ),
        ("one_way_times_arr", one_way_times_arr),
        ("round_trip_times_arr", round_trip_times_arr),
        ("inter_attack_intervals_arr", inter_attack_intervals_arr),
    ],
)

Section 4 data: attack flight and cadence times
Printed: 2026-11-17 09:59:32 -0300 | 5 item(s)
[attack_time_stats_df] pandas.DataFrame, shape=(3, 8)
                             time_metric  samples mean_sec mean_text sample_std_sec sample_std_text pop_std_sec median_sec
One-Way Flight Time (Dispatch -> Impact)        7   151.47    2m 31s         113.25          1m 53s      104.85     115.58
Round-Trip Attack Time (Slot Occupation)        7   302.93    5m 03s         226.49          3m 46s      209.69     231.17
          Inter-Attack Dispatch Interval        6 13249.65    3h 40m       21626.47          6h 00m    19742.18    3354.80
----------------------------------------------------------------------------------------------------
[exact time statistics in seconds] dict, len=9
{
  "attack_one_way_time_mean_sec": 151.4660808500405,
  "attack_one_way_time_std_sec": 113.24717030740591,
  "attack_one_way_time_pop_std_sec": 104.84650651296167,
  "attack_time_mean_sec": 302.932161700081,
  

### 10.6 Section 5 Data — Attack Success Rate & Simulation Parameters

`attack_summary_stats_df`, the success-rate and expected-yield scalars, `attack_class_stats_df` with `per_class_success_rate`, `attack_target_stats_df` with `per_target_success_rate`, `ATTACK_SIM_PARAMS`, and the `ATTACK_STATS_TABLES` registry.

In [79]:
print_section_data(
    "Section 5 data: attack success rate and simulation parameters",
    [
        ("attack_summary_stats_df", attack_summary_stats_df),
        (
            "success-rate and expected-yield scalars",
            {
                "attack_attempts": attack_attempts,
                "attack_wins": attack_wins,
                "attack_losses": attack_losses,
                "attack_success_rate": attack_success_rate,
                "attack_success_rate_std_err": attack_success_rate_std_err,
                "attack_success_rate_smoothed": attack_success_rate_smoothed,
                "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
                "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
                "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
                "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
            },
        ),
        ("attack_class_stats_df", attack_class_stats_df),
        ("per_class_success_rate", per_class_success_rate),
        ("attack_target_stats_df", attack_target_stats_df),
        ("per_target_success_rate", per_target_success_rate),
        ("ATTACK_SIM_PARAMS", ATTACK_SIM_PARAMS),
        ("ATTACK_STATS_TABLES", ATTACK_STATS_TABLES),
    ],
)

Section 5 data: attack success rate and simulation parameters
Printed: 2026-11-17 09:59:32 -0300 | 8 item(s)
[attack_summary_stats_df] pandas.DataFrame, shape=(7, 3)
                            metric         value                                 display
                     Total Attacks      7.000000                   7 (6 wins / 1 losses)
            Empirical Success Rate      0.857143                  85.71% (+/- 13.23% SE)
     Laplace-Smoothed Success Rate      0.777778                                  77.78%
 Max Sorties per Fleet Slot / Hour     11.883800 11.88 raids/hr (mean round-trip 5m 03s)
Gross Loot per Slot-Hour (SpiceEq)  83879.430000                     83,879.4 SpiceEq/hr
  Net Loot per Slot-Hour (SpiceEq) -30091.770000                    -30,091.8 SpiceEq/hr
   Full Capacity Ceiling (7 Slots) 587156.010000              587,156.0 gross SpiceEq/hr
----------------------------------------------------------------------------------------------------
[success-rate and exp

### 10.7 Section 6 Data — Colonies, Generic Planets & Contexts

The table level caps and empire settings, `GAME_PARAM_CONTEXT`, `GENERIC_PLANET_CONTEXT`, `planet_contexts_df`, `PLANET_PARAM_CONTEXTS`, `merchant_rates_df`, and the typed `live_planet_states` and `empire_planet_states`: one JSON line per planet with its resources, storage, energy, buildings, ships, defenses and economy rates.

In [80]:
print_section_data(
    "Section 6 data: colonies, generic planets and contexts",
    [
        (
            "table level caps and empire settings",
            {
                "MAX_BUILDING_TABLE_LEVEL": MAX_BUILDING_TABLE_LEVEL,
                "MAX_RESEARCH_TABLE_LEVEL": MAX_RESEARCH_TABLE_LEVEL,
                "MAX_EMPIRE_PLANETS": MAX_EMPIRE_PLANETS,
                "EMPTY_COLONY_SIZE_FIELDS": EMPTY_COLONY_SIZE_FIELDS,
                "EMPTY_COLONY_DEFAULT_TEMP_C": EMPTY_COLONY_DEFAULT_TEMP_C,
            },
        ),
        ("GAME_PARAM_CONTEXT", GAME_PARAM_CONTEXT),
        ("GENERIC_PLANET_CONTEXT", GENERIC_PLANET_CONTEXT),
        ("planet_contexts_df", planet_contexts_df),
        ("PLANET_PARAM_CONTEXTS", PLANET_PARAM_CONTEXTS),
        ("merchant_rates_df", merchant_rates_df),
        ("live_planet_states", live_planet_states),
        ("empire_planet_states", empire_planet_states),
    ],
)

Section 6 data: colonies, generic planets and contexts
Printed: 2026-11-17 09:59:32 -0300 | 8 item(s)
[table level caps and empire settings] dict, len=5
{
  "MAX_BUILDING_TABLE_LEVEL": 50,
  "MAX_RESEARCH_TABLE_LEVEL": 30,
  "MAX_EMPIRE_PLANETS": 11,
  "EMPTY_COLONY_SIZE_FIELDS": 300,
  "EMPTY_COLONY_DEFAULT_TEMP_C": 0
}
----------------------------------------------------------------------------------------------------
[GAME_PARAM_CONTEXT] dict, len=11
{
  "wiki_synced_at_utc": "2026-11-15T02:03:10Z",
  "speedup_factor": 20.0,
  "r1_mining_tech": 9,
  "r2_energy_tech": 4,
  "r4_rocket_engine": 8,
  "r5_diffusion_engine": 7,
  "r6_warp_engine": 0,
  "r8_weapons_tech": 0,
  "r9_shield_tech": 0,
  "r10_defense_systems": 0,
  "points_per_resource_divisor": 1000.0
}
----------------------------------------------------------------------------------------------------
[GENERIC_PLANET_CONTEXT] dict, len=22
{
  "wiki_synced_at_utc": "2026-11-15T02:03:10Z",
  "speedup_factor": 20.0,
  "r1_mining

### 10.8 Section 6.1 Data — Buildings

`buildings_catalog_df` (one row per building), `planet_buildings_df` (one row per planet and building) and `buildings_levels_df` (every level of every building, all columns).

**Precision of high-level costs.** `game_data` copies its costs from the wiki, which rounds every cost of `1,000,000,000` or more. Compared with exact doubling for the buildings whose cost doubles per level: every cost below `10^9` is exact, larger costs are up to 0.42 % off, and from `10^15` on only 5 significant digits remain (hence values such as `281470000000000032768`).

In [81]:
print_section_data(
    "Section 6.1 data: buildings",
    [
        ("buildings_catalog_df", buildings_catalog_df),
        ("planet_buildings_df", planet_buildings_df),
        ("buildings_levels_df", buildings_levels_df),
    ],
)

Section 6.1 data: buildings
Printed: 2026-11-17 09:59:32 -0300 | 3 item(s)
[buildings_catalog_df] pandas.DataFrame, shape=(21, 17)
 building_id key                    slug                    name          category  asteroid_ok  max_table_level cost_scale_factor  l1_pig_iron  l1_metal  l1_kryptonite  l1_spice l1_cost_spice_eq l1_base_time_s l1_effective_time_s                                            prerequisites                                                                                           primary_effect
           1  b1           pig-iron-mine           Pig iron mine     resource_mine        False              100              1.51           78         0             25         0            21.33             93                4.65                                                     None                                                             Mines Pig iron (+2%/lvl r1); consumes energy
           2  b2           blast-furnace           Blast furnace           smelter 

### 10.9 Section 6.2 Data — Technologies

`technologies_catalog_df` (one row per research), `planet_technologies_df` (one row per planet and research) and `technologies_levels_df` (every level of every research).

The wiki rounding described in 10.8 applies here too: research costs of `1,000,000,000` or more are up to 0.39 % off exact doubling, and costs below `10^9` are exact.

In [82]:
print_section_data(
    "Section 6.2 data: technologies",
    [
        ("technologies_catalog_df", technologies_catalog_df),
        ("planet_technologies_df", planet_technologies_df),
        ("technologies_levels_df", technologies_levels_df),
    ],
)

Section 6.2 data: technologies
Printed: 2026-11-17 09:59:32 -0300 | 3 item(s)
[technologies_catalog_df] pandas.DataFrame, shape=(15, 20)
 research_id key                slug                name           domain  current_level  max_table_level cost_scale_factor  l1_pig_iron  l1_metal  l1_kryptonite  l1_spice l1_cost_spice_eq l1_base_time_s l1_effective_time_s                  effect_metric per_level_bonus_pct                                                                                          applies_to                                       prerequisites                                                                   strategic_effect
           1  r1   mining-technology   Mining technology          economy              9               40                 2         1000         0            500         0           333.33           2083              104.15              mine_output_bonus                   2                      b1 Pig iron, b3 Kryptonite, b4 Spice across all planets (

### 10.10 Section 6.3 Data — Spaceships

`ships_df` (every ship, all columns) and `planet_ships_df` (one row per planet and ship).

In [83]:
print_section_data(
    "Section 6.3 data: spaceships",
    [
        ("ships_df", ships_df),
        ("planet_ships_df", planet_ships_df),
    ],
)

Section 6.3 data: spaceships
Printed: 2026-11-17 09:59:32 -0300 | 2 item(s)
[ships_df] pandas.DataFrame, shape=(15, 34)
 ship_id key              slug              name                    role  pig_iron   metal  kryptonite  spice cost_spice_eq points_value base_build_time_s effective_build_time_s effective_build_time_text units_per_hour_effective  hull_base  shield_base  attack_base hull_effective shield_effective attack_effective           engine engine_tech_key  speed_base speed_effective   cargo  fuel energy_output_base energy_output_effective  fields_used cargo_per_spice_eq attack_per_spice_eq hull_per_spice_eq                                                                prerequisites
       1  s1   small-transport   Small transport         cargo_transport         0    3000        3000      0       2200.00          6.0           1700.00                  85.00                    1m 25s                    42.35       6000           10            2           6000               10   

### 10.11 Section 6.4 Data — Planetary Defenses & Table Registry

`defenses_df` (every defense, all columns), `planet_defenses_df` (one row per planet and defense), `game_params_summary_df`, and the `GAME_PARAMS_TABLES` registry (name and shape of the twelve tables of Section 6).

In [84]:
print_section_data(
    "Section 6.4 data: planetary defenses and table registry",
    [
        ("defenses_df", defenses_df),
        ("planet_defenses_df", planet_defenses_df),
        ("game_params_summary_df", game_params_summary_df),
        ("GAME_PARAMS_TABLES", GAME_PARAMS_TABLES),
    ],
)

Section 6.4 data: planetary defenses and table registry
Printed: 2026-11-17 09:59:32 -0300 | 4 item(s)
[defenses_df] pandas.DataFrame, shape=(11, 25)
 defense_id key                  slug                  name             role  can_dig_in  pig_iron  metal  kryptonite  spice cost_spice_eq points_value base_build_time_s effective_build_time_s effective_build_time_text units_per_hour_effective  hull_base  shield_base  attack_base hull_effective shield_effective attack_effective attack_per_spice_eq hull_plus_shield_per_spice_eq                                                                     prerequisites
          1  d1                mortar                Mortar planetary_turret        True         0   2000           0      0        800.00          2.0           1000.00                  50.00                       50s                    72.00       2000           15           35           2000               15               35              0.0437                          2.52         

### 10.12 Section 7.2 Data — Technology Requirement Graph

The content of `TECH_GRAPH` (node and edge counts, `topological_order`, the `nodes` store, and the `ancestors_of` and `descendants_of` closure of every node) with `tech_graph_nodes_df` and `tech_graph_edges_df`.

In [85]:
print_section_data(
    "Section 7.2 data: technology requirement graph",
    [
        (
            "TECH_GRAPH (TechnologyRequirementGraph summary)",
            {
                "nodes": len(TECH_GRAPH.nodes),
                "edges": sum(len(requirements) for requirements in TECH_GRAPH.prereqs_of.values()),
                "max_topological_layer": max(TECH_GRAPH.topological_layer.values()),
            },
        ),
        ("TECH_GRAPH.topological_order", list(TECH_GRAPH.topological_order)),
        ("TECH_GRAPH.nodes", pd.DataFrame(list(TECH_GRAPH.nodes.values()))),
        ("tech_graph_nodes_df", tech_graph_nodes_df),
        ("tech_graph_edges_df", tech_graph_edges_df),
        (
            "TECH_GRAPH.ancestors_of / TECH_GRAPH.descendants_of (transitive closure per node)",
            {
                node: {
                    "all_prerequisites": [
                        key for key in TECH_GRAPH.topological_order if key in TECH_GRAPH.ancestors_of[node]
                    ],
                    "all_unlocks": [
                        key for key in TECH_GRAPH.topological_order if key in TECH_GRAPH.descendants_of[node]
                    ],
                }
                for node in TECH_GRAPH.topological_order
            },
        ),
    ],
)

Section 7.2 data: technology requirement graph
Printed: 2026-11-17 09:59:32 -0300 | 6 item(s)
[TECH_GRAPH (TechnologyRequirementGraph summary)] dict, len=3
{
  "nodes": 62,
  "edges": 101,
  "max_topological_layer": 5
}
----------------------------------------------------------------------------------------------------
[TECH_GRAPH.topological_order] list, len=62
["b1", "b2", "b3", "b4", "b5", "b7", "b10", "b11", "b12", "b13", "b14", "b8", "b19", "r2", "r3", "r7", "r8", "r10", "r14", "b6", "b9", "b17", "r1", "r9", "r11", "r13", "r4", "r5", "r6", "b20", "s9", "d1", "d5", "b18", "b21", "r12", "d10", "d11", "d2", "d3", "d4", "d7", "d8", "d9", "s1", "s2", "s4", "s5", "s8", "s10", "s3", "s6", "s7", "s11", "r15", "s13", "s14", "s15", "b15", "b16", "d6", "s16"]
----------------------------------------------------------------------------------------------------
[TECH_GRAPH.nodes] pandas.DataFrame, shape=(62, 6)
key  category  entity_id                    slug                    name  max_level


### 10.13 Section 7.3 Data — Unlock Plans

`unlock_targets_summary_df` (remaining cost, time and ordered chain of upgrades of every entity), `phoenix_unlock_plan_df` (from the reference colony), `cold_phoenix_unlock_plan_df` (from an empty account), and the `TECH_GRAPH_TABLES` registry.

In [86]:
print_section_data(
    "Section 7.3 data: unlock plans",
    [
        ("unlock_targets_summary_df", unlock_targets_summary_df),
        ("phoenix_unlock_plan_df", phoenix_unlock_plan_df),
        ("cold_phoenix_unlock_plan_df", cold_phoenix_unlock_plan_df),
        ("TECH_GRAPH_TABLES", TECH_GRAPH_TABLES),
    ],
)

Section 7.3 data: unlock plans
Printed: 2026-11-17 09:59:32 -0300 | 4 item(s)
[unlock_targets_summary_df] pandas.DataFrame, shape=(62, 11)
key                    name  category  topological_layer  unlocked_now  missing_upgrade_nodes  total_levels_to_add unlock_cost_spice_eq unlock_effective_time_s unlock_effective_time_text                                                ordered_prerequisite_chain
 b1           Pig iron mine buildings                  0          True                      0                    0                 0.00                    0.00                      Ready                                                                  Unlocked
 b2           Blast furnace buildings                  0          True                      0                    0                 0.00                    0.00                      Ready                                                                  Unlocked
 b3         Kryptonite mine buildings                  0          True        

### 10.14 Section 8.1 Data — Owned Planets

`live_planets_df` (one row per owned planet, all columns).

In [87]:
print_section_data(
    "Section 8.1 data: owned planets",
    [
        ("live_planets_df", live_planets_df),
    ],
)

Section 8.1 data: owned planets
Printed: 2026-11-17 09:59:32 -0300 | 1 item(s)
[live_planets_df] pandas.DataFrame, shape=(1, 27)
 slot_index  planet_id    status  is_empty_slot   name    coords planet_type  temperature_c  fields_used  fields_max  fields_remaining field_utilization_pct  pig_iron  metal  kryptonite  spice resources_spice_eq  storage_pig_iron  storage_metal  storage_kryptonite  storage_spice  energy_produced  energy_used  energy_net  total_building_levels  total_ships  total_defenses
          1      12476 colonized          False Colony 02:380:12      normal             31          137         230                93                 59.57    195068  40720      433519 475592          668897.67           3200000         800000             1600000         800000             5954         4989         965                    137           75               0


### 10.15 Section 8.2 Data — Full Empire Roster with Empty Colony Slots

`EMPIRE_PLANET_STATE_SUMMARY`, `empire_planets_df`, `empire_planet_buildings_df` (the level of every building per slot), the three field arrays, and the `EMPIRE_PLANET_TABLES` registry.

In [88]:
print_section_data(
    f"Section 8.2 data: full empire roster with empty {EMPTY_COLONY_SIZE_FIELDS}-field slots",
    [
        ("EMPIRE_PLANET_STATE_SUMMARY", EMPIRE_PLANET_STATE_SUMMARY),
        ("empire_planets_df", empire_planets_df),
        ("empire_planet_buildings_df", empire_planet_buildings_df),
        ("planet_sizes_arr", planet_sizes_arr),
        ("planet_fields_used_arr", planet_fields_used_arr),
        ("planet_fields_remaining_arr", planet_fields_remaining_arr),
        ("EMPIRE_PLANET_TABLES", EMPIRE_PLANET_TABLES),
    ],
)

Section 8.2 data: full empire roster with empty 300-field slots
Printed: 2026-11-17 09:59:32 -0300 | 7 item(s)
[EMPIRE_PLANET_STATE_SUMMARY] dict, len=9
{
  "max_empire_planets": 11,
  "empty_colony_size_fields": 300,
  "colonized_planets": 1,
  "empty_colony_slots": 10,
  "total_empire_slots": 11,
  "total_fields_capacity": 3230,
  "total_fields_used": 137,
  "total_fields_remaining": 3093,
  "mean_planet_size_fields": 293.6363636363636
}
----------------------------------------------------------------------------------------------------
[empire_planets_df] pandas.DataFrame, shape=(11, 27)
 slot_index  planet_id            status  is_empty_slot                 name    coords planet_type  temperature_c  fields_used  fields_max  fields_remaining field_utilization_pct  pig_iron  metal  kryptonite  spice resources_spice_eq  storage_pig_iron  storage_metal  storage_kryptonite  storage_spice  energy_produced  energy_used  energy_net  total_building_levels  total_ships  total_defenses
      

### 10.16 Section 9 Data — Maximum Research Levels in the Universe

`universe_research_sources` (ledger rows, highest spied levels, number of spy reports with researches), `universe_research_df`, `UNIVERSE_RESEARCH_MAX_LEVELS` and `UNIVERSE_RESEARCH_SUMMARY`.

In [89]:
print_section_data(
    "Section 9 data: maximum research levels in the universe",
    [
        ("universe_research_sources", universe_research_sources),
        ("universe_research_df", universe_research_df),
        ("UNIVERSE_RESEARCH_MAX_LEVELS", UNIVERSE_RESEARCH_MAX_LEVELS),
        ("UNIVERSE_RESEARCH_SUMMARY", UNIVERSE_RESEARCH_SUMMARY),
    ],
)

Section 9 data: maximum research levels in the universe
Printed: 2026-11-17 09:59:32 -0300 | 4 item(s)
[universe_research_sources] dict, len=3
{
  "ledger_rows": [{"research_key": "r1", "research_id": 1, "slug": "mining-technology", "name": "Mining technology", "own_level": 9, "spied_max_level": 0, "server_record_level": 30, "universe_max_level": 30, "holder": "ventkx86", "holder_coords": "", "updated_ts": 1794900668.659}, {"research_key": "r2", "research_id": 2, "slug": "energy-technology", "name": "Energy technology", "own_level": 4, "spied_max_level": 0, "server_record_level": 28, "universe_max_level": 28, "holder": "ventkx86", "holder_coords": "", "updated_ts": 1794900668.659}, {"research_key": "r3", "research_id": 3, "slug": "engine-research", "name": "Engine research", "own_level": 6, "spied_max_level": 0, "server_record_level": 20, "universe_max_level": 20, "holder": "ventkx86", "holder_coords": "", "updated_ts": 1794900668.659}, {"research_key": "r4", "research_id": 4, "slug": 

## 11. Best Discounted Weekly Path Toward the Saved Configuration

`config_optimizer.ipynb` says which building levels the empire should end with, which can take centuries when level caps are high. Rather than optimizing only the final completion date, this section maximizes **weekly-discounted resource production** over the next `PATH_REWARD_WEEKS` weeks (`w = 1..W`, week `w` weighted by `PATH_WEEKLY_DISCOUNT ** (w - 1)`), so early-week snowballing counts more than late-game builds. It reads the target configuration from `config_optimizer.ipynb`, so run that notebook first and whenever a colony is founded or lost.

**The model.** A **step** is one building level to build or tear down, one research level, the missing minimum-fleet ships on one planet, or the founding of one colony. Section 11.4 simulates the empire across the `W`-week window under these rules:

- Resources are pooled, as in the configuration model. A step is paid when it starts and counts when it ends.
- A planet builds or tears down one level at a time, the account researches one level at a time, and each shipyard works on one order at a time.
- A step needs its prerequisites, a free field, and warehouses on the paying planet that hold each resource of its cost. Research is paid on the planet with the Research lab, Colony ships on the central planet.
- Building times follow the game: the base time divided by `(Development center + 1) * 2^Microsystem accelerator * universe speed`, with the levels that stand when the step starts.
- Each planet produces as in the configuration model (power ratio, smelting ratio), at the research levels of the moment. Weekly production is measured in **Spice-equivalent** (`game_data.spice_equivalent`) on the empire's founded planets. The **Blast furnaces run only while the next step lacks Metal**; otherwise the Pig iron is kept for building.
- **Attacks** add an income driven by **Fleet base (`b10`, Fleet Command Center)** slots (`1 + b10` simultaneous flights per founded planet, or `1 + PATH_FLEET_COMMAND_CENTER_LEVEL` when overridden) multiplied by `3600 / median_round_trip_sec` and the median attack loot (Section 11.3). Upgrading `Fleet base` (`b10`) or founding colonies therefore grows hourly raid income along the path.
- **Spare percentage** (`PATH_RESOURCE_SPARE_PERCENT`): deducts a per-resource percentage from all hourly income (planet output and attack loot) for ship construction or raid losses; both buildable cashflow and the weekly reward count the net kept fraction `1 - spare%`.
- A level that the configuration no longer has is torn down when its field is needed, or once builds finish.

**The search.** A **strategy** ranks the next steps using Net Present Value (NPV) over the remaining discounted weeks (Section 11.5). Three named strategies are simulated, and an 8-gene genetic algorithm tunes the policy to maximize the discounted weekly SpiceEq reward (Section 11.6). Section 11.4 also computes a **reward ceiling** (upper bound on discounted SpiceEq) and the full-configuration **time lower bound**.

**The result** is the best `W`-week path, printed in Section 11.7 and saved by Section 11.8 as `full_planner_path.txt` — listing the max level reached of each factor in its header followed by the actions started within the `W` weeks.

**What the model leaves out:** hauling resources between planets, Merchant trades, the 5 % production of a full warehouse, the energy that a furnace frees while it is off, steps that are already under construction today, and the change of the universe speed as the account grows.

### 11.1 Meta Parameters

Everything you can tune in this section is in this cell, and nothing else: it only sets values. Section 11.2 prints them.

- `PATH_TARGET_FILE_NAME` — the file in `METADATA_DIR` that holds the configuration toward which the path builds.
- `PATH_REWARD_WEEKS`, `PATH_WEEKLY_DISCOUNT` — the `W`-week horizon (`168 * W` hours) and the weekly discount factor `gamma` (`gamma ** (w - 1)` for week `w`).
- `PATH_RESOURCE_SPARE_PERCENT` — percentage of each resource (`0 <= pct < 100`) set aside for building ships or lost to other players' attacks.
- `PATH_FLEET_COMMAND_CENTER_LEVEL`, `PATH_ATTACKS_PER_DAY` — `None` uses each founded planet's live/simulated `Fleet base` (`b10`) level (`1 + b10` slots x `3600 / median_round_trip_sec`); an integer sets a fixed `Fleet base` level per founded planet, and `PATH_ATTACKS_PER_DAY` overrides total attacks/day (`0` disables raids).
- `PATH_COLONY_FOUNDING_HOURS`, `PATH_DEMOLITION_TIME_FACTOR` — how long a colony takes after its Colony ship is paid, and the time of tearing a level down as a share of the time of building it.
- `PATH_SCARCITY_FOCUS`, `PATH_CRITICAL_QUEUE_SHARE`, `PATH_PATIENCE`, `PATH_SPEED_WEIGHT`, `PATH_REVALUE_EVERY_STEPS` — the settings of the discounted payback rule, explained in Section 11.5.
- `PATH_SEARCH_TOLERANCE` and the `PATH_GA_*` values — when the genetic algorithm runs, and its population, generations, tournament size, crossover and mutation rates, elite count and seed.
- `PATH_PROGRESS_DAYS`, `PATH_OUTPUT_SHARES` — the rows of the two progress tables of Section 11.7.
- `PATH_ACTIONS_FILE_NAME` — the text file that Section 11.8 writes into `METADATA_DIR`.

In [ ]:
# The configuration toward which the path builds: written into METADATA_DIR by config_optimizer.ipynb.
PATH_TARGET_FILE_NAME: str = "config_optimizer_result.json"

# Horizon W (in weeks) and weekly discount gamma: week w in 1..W is weighted by gamma ** (w - 1).
# Only actions starting within these PATH_REWARD_WEEKS weeks are planned and written to the text file.
PATH_REWARD_WEEKS: int = 26
PATH_WEEKLY_DISCOUNT: float = 0.90

# Percentage of each resource's hourly income reserved for building ships or lost to enemy player attacks.
# Both buildable income and the weekly production reward count the net kept share (100% - spare%).
PATH_RESOURCE_SPARE_PERCENT: dict[str, float] = {
    "pig_iron": 10.0,
    "metal": 10.0,
    "kryptonite": 10.0,
    "spice": 10.0,
}

# Attacks: each founded planet provides `1 + b10` fleet slots (Fleet base / Fleet Command Center), launching
# `3600 / median_round_trip_sec` median-loot attacks per hour per slot. `None` uses each planet's dynamic `b10`
# level along the path; an integer >= 0 sets a fixed level per founded planet; `PATH_ATTACKS_PER_DAY` overrides pace.
PATH_FLEET_COMMAND_CENTER_LEVEL: int | None = None
PATH_ATTACKS_PER_DAY: float | None = None

# A colony costs one Colony ship and is ready this many hours after the ship is paid for.
PATH_COLONY_FOUNDING_HOURS: float = 0.0
# Tearing a level down costs nothing and takes this share of the time that building the level takes.
PATH_DEMOLITION_TIME_FACTOR: float = 1.0

# The discounted payback rule that picks the next step (Section 11.5), before the genetic algorithm tunes it.
PATH_SCARCITY_FOCUS: float = 1.0
PATH_CRITICAL_QUEUE_SHARE: float = 0.9
PATH_PATIENCE: float = 0.25
PATH_SPEED_WEIGHT: float = 1.5
PATH_REVALUE_EVERY_STEPS: int = 25

# Genetic algorithm that tunes the rule (Section 11.6). Skipped when the best named rule is already within
# PATH_SEARCH_TOLERANCE of the reward ceiling.
PATH_SEARCH_TOLERANCE: float = 0.01
PATH_GA_POPULATION_SIZE: int = 10
PATH_GA_GENERATIONS: int = 6
PATH_GA_TOURNAMENT_SIZE: int = 3
PATH_GA_CROSSOVER_RATE: float = 0.9
PATH_GA_MUTATION_RATE: float = 0.3
PATH_GA_ELITE_COUNT: int = 2
PATH_GA_RANDOM_SEED: int = 20261117

# Report (Section 11.7): the days after which progress is reported, and the shares of the configuration's output.
PATH_PROGRESS_DAYS: tuple[float, ...] = (1.0, 7.0, 14.0, 30.0, 60.0, 90.0, 182.0)
PATH_OUTPUT_SHARES: tuple[float, ...] = (0.1, 0.25, 0.5, 0.75, 0.9, 1.0)

# The text file with the actions of the W-week path, in the order to take them (Section 11.8). Goes to METADATA_DIR.
PATH_ACTIONS_FILE_NAME: str = "full_planner_path.txt"

### 11.2 The Configuration to Reach and the Steps to It

`load_path_problem` reads the saved configuration, compares it with the roster of Section 6 and lists every step between today and the configuration.

- `load_path_target` reads the file. It stops with an error that names `config_optimizer.ipynb` when the file is missing or has another `schema_version`.
- `check_path_target` compares each planet of the configuration with the roster slot it was planned for. A configuration that does not cover the normal planets of the roster, or does not fit the fields of a planet, stops the section. Smaller differences, such as a colony that was founded since, are reported in `same_planet`.
- `build_path_problem` lists the steps. A building below its planned level gets one step per level; a building above it gets one step per level to tear down; a building that the configuration does not list is torn down completely. Research goes from your level to the planned level, the fleet from the ships stationed to the minimum fleet, and every empty slot gets a step that founds its colony. The function stops with an error when the configuration cannot be reached at all: a prerequisite that it lacks, or a cost that its warehouses cannot hold.

The levels of today are those of the roster, not those in the file, so the path starts from where the empire stands now. The stock is the sum over all your planets.

- `WAREHOUSE_KEYS`, `PATH_STEP_KINDS` — the warehouse of each resource, and the five kinds of step.
- `PATH_TARGET_PATH`, `PATH_TARGET_SUMMARY` — the file, when it was saved, and the totals of the problem: planets to found, steps per kind, cost, stock.
- `path_target_checks_df` — one row per planet: the planet of the roster next to the planet in the file, the fields that the configuration fills, and `same_planet`.
- `PATH_PROBLEM` — the problem as the simulator reads it: the planets with their levels of today and of the configuration, the research levels, the stock, and the list of steps.
- `path_steps_summary_df` — number and cost of the steps per planet and kind.

`fmt_path_time` writes hours as text, in years once they exceed two years.

In [ ]:
# The warehouse of each resource, in `RESOURCE_KEYS` order.
WAREHOUSE_KEYS: tuple[str, ...] = tuple(
    str(RESOURCE_METADATA[resource]["warehouse_building_key"]) for resource in RESOURCE_KEYS
)
PATH_STEP_KINDS: tuple[str, ...] = ("build", "demolish", "research", "ships", "colonize")


def warehouse_level_for(amount: float) -> int:
    """Lowest warehouse level whose capacity holds `amount` units of a resource."""
    level = 0
    while warehouse_capacity(level) < amount:
        level += 1
    return level


def load_path_target(path: Path) -> dict[str, Any]:
    """The configuration that `config_optimizer.ipynb` saved, or an error that says how to get one."""
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} does not exist: run notebooks/config_optimizer.ipynb first, its last cell writes this file"
        )
    target = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(target, dict) or target.get("schema_version") != 1:
        found = target.get("schema_version") if isinstance(target, dict) else type(target).__name__
        raise ValueError(
            f"{path.name} has schema_version {found!r}, but this section reads version 1: "
            "run notebooks/config_optimizer.ipynb again"
        )
    return target


def check_path_target(
    target: dict[str, Any], planets: list[PlanetState], contexts: list[dict[str, Any]]
) -> pd.DataFrame:
    """Compare every planet of the saved configuration with the roster slot it was planned for.

    One row per planet of the configuration. The search needs a configuration that covers every normal planet
    of the roster and fits the fields of each: otherwise a `ValueError` asks for a new run of
    `config_optimizer.ipynb`. Other differences (a colony that was founded since, another temperature) are
    reported in `same_planet` and do not stop the search.
    """
    stale = f"run notebooks/config_optimizer.ipynb again, {PATH_TARGET_FILE_NAME} was saved for another roster"
    roster_slots = {
        context["slot_index"] for planet, context in zip(planets, contexts) if planet.planet_type == "normal"
    }
    target_slots = {int(entry["slot_index"]) for entry in target["planets"]}
    if roster_slots != target_slots:
        raise ValueError(
            f"The saved configuration plans the roster slots {sorted(target_slots)}, "
            f"but the normal planets of the roster are in the slots {sorted(roster_slots)}: {stale}"
        )
    rows: list[dict[str, Any]] = []
    for entry in target["planets"]:
        slot_index = int(entry["slot_index"])
        planet, context = planets[slot_index - 1], contexts[slot_index - 1]
        target_fields = sum(int(building["target_level"]) for building in entry["buildings"])
        if target_fields > int(planet.fields_max):
            raise ValueError(
                f"The saved configuration puts {target_fields} levels on {planet.coords} (slot {slot_index}), "
                f"which has {planet.fields_max} fields: {stale}"
            )
        rows.append(
            {
                **planet_identity(context),
                "saved_coords": str(entry["coords"]),
                "saved_planet_status": str(entry["planet_status"]),
                "temperature_c": int(planet.temperature),
                "saved_temperature_c": int(entry["temperature_c"]),
                "fields_max": int(planet.fields_max),
                "saved_fields_max": int(entry["fields_max"]),
                "target_fields": target_fields,
                "same_planet": (
                    str(entry["coords"]) == planet.coords
                    and str(entry["planet_status"]) == context["planet_status"]
                    and int(entry["temperature_c"]) == int(planet.temperature)
                    and int(entry["fields_max"]) == int(planet.fields_max)
                ),
            }
        )
    return pd.DataFrame(rows)


def path_requirements(key: str, building_owner: int) -> list[tuple[int, str, int]]:
    """Direct prerequisites of `key` as `(owner, key, level)`: `building_owner` for a building, -1 for a research."""
    return [
        (building_owner if TECH_GRAPH.nodes[source]["category"] == "buildings" else -1, source, int(level))
        for source, level in TECH_GRAPH.prereqs_of[key]
    ]


def build_path_problem(
    target: dict[str, Any],
    planets: list[PlanetState],
    contexts: list[dict[str, Any]],
    owned_planets: list[PlanetState],
    research_levels: dict[str, int],
    *,
    colony_founding_hours: float,
    demolition_time_factor: float,
) -> dict[str, Any]:
    """Everything between today and the saved configuration, as a list of steps.

    A step is one level to build or to tear down, one research level, the missing ships of one type on one
    planet, or the founding of one colony. `steps[i]` holds what the step costs, the hours it takes before
    the accelerators and the universe speed are applied (`base_hours`), the queue it occupies, the planet
    whose warehouses must hold its cost (`pay_planet`), and what must stand before it can start
    (`requires`). Steps of one building or research form a chain and are listed in the order of their levels.

    Raises `ValueError` when the configuration cannot be reached at all: a prerequisite that it does not
    contain, or a cost that the warehouses of the configuration cannot hold.
    """
    speedup = float(contexts[0]["speedup_factor"])
    min_fleet = {str(ship): int(count) for ship, count in target["meta_parameters"]["MIN_FLEET_PER_PLANET"].items()}
    research_now = {key: max(0, int(research_levels.get(key, 0) or 0)) for key, _ in research_catalog()}
    research_target = {
        key: max(research_now[key], int(target["planned_research_levels"].get(key, 0) or 0))
        for key in research_now
    }

    problem_planets: list[dict[str, Any]] = []
    for index, entry in enumerate(target["planets"]):
        slot_index = int(entry["slot_index"])
        planet, context = planets[slot_index - 1], contexts[slot_index - 1]
        levels = {str(key): int(level) for key, level in planet.buildings.items() if int(level or 0) > 0}
        target_levels = {
            str(building["key"]): int(building["target_level"])
            for building in entry["buildings"]
            if int(building["target_level"]) > 0
        }
        stationed = {str(ship): max(0, int(count or 0)) for ship, count in planet.ships.items()}
        problem_planets.append(
            {
                "index": index,
                **planet_identity(context),
                "temperature_c": int(planet.temperature),
                "fields_max": int(planet.fields_max),
                "founded": context["planet_status"] == "colonized",
                "is_central_planet": bool(entry["is_central_planet"]),
                "pays_for_research": bool(entry["pays_for_research"]),
                "levels": levels,
                "target_levels": target_levels,
                "solar_cells": stationed.get("s9", 0),
                "saved_production": dict(entry.get("production") or {}),
                "missing_ships": {
                    ship: count - stationed.get(ship, 0)
                    for ship, count in min_fleet.items()
                    if count > stationed.get(ship, 0)
                },
            }
        )
    lab_planet = next((planet["index"] for planet in problem_planets if planet["pays_for_research"]), -1)
    central_planet = next(planet["index"] for planet in problem_planets if planet["is_central_planet"])

    base_times: dict[str, dict[int, float]] = {}

    def base_hours(key: str, level: int) -> float:
        """Catalog time of a building or research level in hours, before accelerators and universe speed."""
        if key not in base_times:
            info = game_data.BUILDING_BY_ID[int(key[1:])] if key.startswith("b") else game_data.RESEARCH_BY_ID[key]
            levels_map = info.get("levels") or {}
            top_level = int(info.get("max_table_level") or len(levels_map))
            base_times[key] = build_monotonic_base_times(levels_map, top_level)
        return float(base_times[key][level]) / 3600.0

    def cost_tuple(raw_cost: Any, count: int = 1) -> tuple[float, ...]:
        cost = full_cost(raw_cost)
        return tuple(float(count) * float(cost[resource]) for resource in RESOURCE_KEYS)

    steps: list[dict[str, Any]] = []

    def add_step(**step: Any) -> None:
        step["id"] = len(steps)
        step["cost_spice_eq"] = float(game_data.spice_equivalent(dict(zip(RESOURCE_KEYS, step["cost"]))))
        step["warehouse_levels"] = tuple(warehouse_level_for(amount) for amount in step["cost"])
        steps.append(step)

    planet_count = len(problem_planets)
    for planet in problem_planets:
        index = planet["index"]
        for building_id in sorted(game_data.BUILDING_BY_ID):
            key = f"b{building_id}"
            name = str(game_data.BUILDING_BY_ID[building_id]["name"])
            built, wanted = planet["levels"].get(key, 0), planet["target_levels"].get(key, 0)
            for level in range(built + 1, wanted + 1):
                add_step(
                    kind="build", planet=index, key=key, name=name, level_from=level - 1, level_to=level,
                    cost=cost_tuple(game_data.building_cost(building_id, level)),
                    base_hours=base_hours(key, level), queue=index, pay_planet=index,
                    requires=path_requirements(key, index), chain=("build", index, key),
                )
            for level in range(built, wanted, -1):
                add_step(
                    kind="demolish", planet=index, key=key, name=name, level_from=level, level_to=level - 1,
                    cost=(0.0, 0.0, 0.0, 0.0), base_hours=float(demolition_time_factor) * base_hours(key, level),
                    queue=index, pay_planet=index, requires=[], chain=("demolish", index, key),
                )
    for key, info in research_catalog():
        for level in range(research_now[key] + 1, research_target[key] + 1):
            add_step(
                kind="research", planet=-1, key=key, name=str(info["name"]), level_from=level - 1, level_to=level,
                cost=cost_tuple(((info.get("levels") or {}).get(level) or {}).get("cost")),
                base_hours=base_hours(key, level), queue=planet_count, pay_planet=lab_planet,
                requires=path_requirements(key, lab_planet), chain=("research", -1, key),
            )
    for planet in problem_planets:
        index = planet["index"]
        for ship_key in unit_keys(SHIP_SLUG_BY_KEY):
            missing = planet["missing_ships"].get(ship_key, 0)
            if missing > 0:
                ship = game_data.SHIPS[SHIP_SLUG_BY_KEY[ship_key]]
                add_step(
                    kind="ships", planet=index, key=ship_key, name=str(ship["name"]), level_from=0, level_to=missing,
                    cost=cost_tuple(ship.get("base_cost"), missing),
                    base_hours=missing * float(ship["base_build_time_s"]) / 3600.0,
                    queue=planet_count + 1 + index, pay_planet=index,
                    requires=path_requirements(ship_key, index), chain=("ships", index, ship_key),
                )
    colony_ship = game_data.SHIPS[SHIP_SLUG_BY_KEY["s8"]]
    for planet in problem_planets:
        if not planet["founded"]:
            add_step(
                kind="colonize", planet=planet["index"], key="s8", name=str(colony_ship["name"]),
                level_from=0, level_to=1, cost=cost_tuple(colony_ship.get("base_cost")),
                base_hours=float(colony_founding_hours), queue=-1, pay_planet=central_planet,
                requires=[], chain=("colonize", planet["index"], "s8"),
            )

    # The configuration must be reachable: every prerequisite in it, every cost within its warehouses.
    for step in steps:
        label = f"{step['name']} {step['level_to']}" + (
            f" on {problem_planets[step['planet']]['coords']}" if step["planet"] >= 0 else ""
        )
        for owner, key, level in step["requires"]:
            if owner < 0:
                final_level = research_target.get(key, 0)
            else:
                final_level = problem_planets[owner]["target_levels"].get(key, 0)
            if final_level < level:
                where = "the planned research" if owner < 0 else f"the plan of {problem_planets[owner]['coords']}"
                raise ValueError(
                    f"{label} needs {TECH_GRAPH.nodes[key]['name']} {level}, but {where} ends at level "
                    f"{final_level}: run notebooks/config_optimizer.ipynb again"
                )
        if step["pay_planet"] < 0:
            raise ValueError(f"{label} is planned, but no planet of the configuration pays for research")
        pay_levels = problem_planets[step["pay_planet"]]["target_levels"]
        for resource, warehouse_key, needed in zip(RESOURCE_KEYS, WAREHOUSE_KEYS, step["warehouse_levels"]):
            if needed > pay_levels.get(warehouse_key, 0):
                raise ValueError(
                    f"{label} costs more {resource} than the {TECH_GRAPH.nodes[warehouse_key]['name']} of "
                    f"{problem_planets[step['pay_planet']]['coords']} holds at its planned level "
                    f"{pay_levels.get(warehouse_key, 0)} (level {needed} needed): "
                    "run notebooks/config_optimizer.ipynb again"
                )

    stock = [0.0, 0.0, 0.0, 0.0]
    for planet in owned_planets:
        for position, resource in enumerate(RESOURCE_KEYS):
            stock[position] += float(getattr(planet.resources, resource))
    return {
        "speedup_factor": speedup,
        "planets": problem_planets,
        "research_levels": research_now,
        "research_target": research_target,
        "lab_planet": lab_planet,
        "central_planet": central_planet,
        "start_stock": tuple(stock),
        "steps": steps,
    }


def fmt_path_time(hours: float) -> str:
    """Hours as text: `fmt_duration` below two years, then years of 365 days."""
    if not math.isfinite(hours):
        return "never"
    if hours >= 2.0 * 8760.0:
        return f"{hours / 8760.0:,.1f} years"
    return fmt_duration(hours * 3600.0)


def summarize_path_steps(problem: dict[str, Any]) -> pd.DataFrame:
    """Number and cost of the steps of a path problem, per planet and kind of step."""
    rows: dict[tuple[str, str], dict[str, Any]] = {}
    for step in problem["steps"]:
        planet = problem["planets"][step["planet"]]["coords"] if step["planet"] >= 0 else "account"
        empty = {"planet": planet, "kind": step["kind"], "steps": 0, "cost_spice_eq": 0.0}
        row = rows.setdefault((planet, step["kind"]), {**empty, **{resource: 0.0 for resource in RESOURCE_KEYS}})
        row["steps"] += 1
        for position, resource in enumerate(RESOURCE_KEYS):
            row[resource] += step["cost"][position]
        row["cost_spice_eq"] += step["cost_spice_eq"]
    return pd.DataFrame(list(rows.values()), columns=["planet", "kind", "steps", *RESOURCE_KEYS, "cost_spice_eq"])


def load_path_problem(
    path: Path,
    planets: list[PlanetState],
    contexts: list[dict[str, Any]],
    owned_planets: list[PlanetState],
    research_levels: dict[str, int],
    *,
    colony_founding_hours: float,
    demolition_time_factor: float,
) -> tuple[pd.DataFrame, dict[str, Any], dict[str, Any]]:
    """Read the saved configuration and turn it into a path problem: `(checks, problem, summary)`."""
    target = load_path_target(path)
    checks = check_path_target(target, planets, contexts)
    problem = build_path_problem(
        target,
        planets,
        contexts,
        owned_planets,
        research_levels,
        colony_founding_hours=colony_founding_hours,
        demolition_time_factor=demolition_time_factor,
    )
    steps = problem["steps"]
    summary = {
        "file": path.name,
        "saved_at": str(target.get("generated_at", "")),
        "target_ship": str(target["summary"]["target_ship"]),
        "ships_per_month_of_the_configuration": float(target["summary"]["ships_per_month"]),
        "universe_speed_now": problem["speedup_factor"],
        "universe_speed_when_saved": float(target["inputs"]["universe_speedup_factor"]),
        "planets": len(problem["planets"]),
        "planets_to_found": sum(1 for planet in problem["planets"] if not planet["founded"]),
        "planets_that_changed_since_saving": int((~checks["same_planet"]).sum()),
        "steps": len(steps),
        "steps_by_kind": {kind: sum(1 for step in steps if step["kind"] == kind) for kind in PATH_STEP_KINDS},
        "cost": {
            resource: int(round(sum(step["cost"][position] for step in steps)))
            for position, resource in enumerate(RESOURCE_KEYS)
        },
        "cost_spice_eq": int(round(sum(step["cost_spice_eq"] for step in steps))),
        "stock_now": dict(zip(RESOURCE_KEYS, problem["start_stock"])),
    }
    return checks, problem, summary


PATH_TARGET_PATH: Path = METADATA_DIR / PATH_TARGET_FILE_NAME
path_target_checks_df, PATH_PROBLEM, PATH_TARGET_SUMMARY = load_path_problem(
    PATH_TARGET_PATH,
    empire_planet_states,
    PLANET_PARAM_CONTEXTS,
    live_planet_states,
    colony_researches,
    colony_founding_hours=PATH_COLONY_FOUNDING_HOURS,
    demolition_time_factor=PATH_DEMOLITION_TIME_FACTOR,
)
path_steps_summary_df = summarize_path_steps(PATH_PROBLEM)

print_section_data(
    f"Section 11.2 data: meta parameters, the configuration to reach and the {len(PATH_PROBLEM['steps'])} steps to it",
    [
        ("PATH_TARGET_FILE_NAME", PATH_TARGET_FILE_NAME),
        (
            "reward window settings",
            {
                "PATH_REWARD_WEEKS": PATH_REWARD_WEEKS,
                "PATH_WEEKLY_DISCOUNT": PATH_WEEKLY_DISCOUNT,
            },
        ),
        ("PATH_RESOURCE_SPARE_PERCENT", PATH_RESOURCE_SPARE_PERCENT),
        ("PATH_FLEET_COMMAND_CENTER_LEVEL", PATH_FLEET_COMMAND_CENTER_LEVEL),
        ("PATH_ATTACKS_PER_DAY", PATH_ATTACKS_PER_DAY),
        ("PATH_COLONY_FOUNDING_HOURS", PATH_COLONY_FOUNDING_HOURS),
        ("PATH_DEMOLITION_TIME_FACTOR", PATH_DEMOLITION_TIME_FACTOR),
        (
            "payback rule settings",
            {
                "PATH_SCARCITY_FOCUS": PATH_SCARCITY_FOCUS,
                "PATH_CRITICAL_QUEUE_SHARE": PATH_CRITICAL_QUEUE_SHARE,
                "PATH_PATIENCE": PATH_PATIENCE,
                "PATH_SPEED_WEIGHT": PATH_SPEED_WEIGHT,
                "PATH_REVALUE_EVERY_STEPS": PATH_REVALUE_EVERY_STEPS,
            },
        ),
        (
            "genetic algorithm settings",
            {
                "PATH_SEARCH_TOLERANCE": PATH_SEARCH_TOLERANCE,
                "PATH_GA_POPULATION_SIZE": PATH_GA_POPULATION_SIZE,
                "PATH_GA_GENERATIONS": PATH_GA_GENERATIONS,
                "PATH_GA_TOURNAMENT_SIZE": PATH_GA_TOURNAMENT_SIZE,
                "PATH_GA_CROSSOVER_RATE": PATH_GA_CROSSOVER_RATE,
                "PATH_GA_MUTATION_RATE": PATH_GA_MUTATION_RATE,
                "PATH_GA_ELITE_COUNT": PATH_GA_ELITE_COUNT,
                "PATH_GA_RANDOM_SEED": PATH_GA_RANDOM_SEED,
            },
        ),
        ("PATH_PROGRESS_DAYS", PATH_PROGRESS_DAYS),
        ("PATH_OUTPUT_SHARES", PATH_OUTPUT_SHARES),
        ("PATH_ACTIONS_FILE_NAME", PATH_ACTIONS_FILE_NAME),
        ("WAREHOUSE_KEYS", WAREHOUSE_KEYS),
        ("PATH_STEP_KINDS", PATH_STEP_KINDS),
        ("PATH_TARGET_PATH", str(PATH_TARGET_PATH)),
        ("PATH_TARGET_SUMMARY", PATH_TARGET_SUMMARY),
        ("path_target_checks_df", path_target_checks_df),
        (
            "PATH_PROBLEM (summary)",
            {
                "planets": [planet["coords"] for planet in PATH_PROBLEM["planets"]],
                "central_planet": PATH_PROBLEM["planets"][PATH_PROBLEM["central_planet"]]["coords"],
                "planet_that_pays_for_research": (
                    PATH_PROBLEM["planets"][PATH_PROBLEM["lab_planet"]]["coords"]
                    if PATH_PROBLEM["lab_planet"] >= 0
                    else None
                ),
                "research_levels_now": PATH_PROBLEM["research_levels"],
                "research_levels_planned": PATH_PROBLEM["research_target"],
                "steps": len(PATH_PROBLEM["steps"]),
            },
        ),
        ("path_steps_summary_df", path_steps_summary_df),
    ],
)

### 11.3 Income from Attacks: Fleet Base (`b10`) Slots and the Median Attack

The mean loot of Section 3 is sensitive to outliers, so Section 11 uses the **median attack**: for each resource, the median of what the recorded attacks brought (zero on losses).

`median_attack_income` converts `round_trip_times_arr` (Section 4) into the **median round-trip flight time** and computes how many attacks one fleet slot launches per hour (`3600 / median_round_trip_sec`). In SpaceInvasion Classic, each founded planet's **Fleet base (`b10`, Fleet Command Center)** grants `1 + b10` simultaneous fleet slots (`1 + PATH_FLEET_COMMAND_CENTER_LEVEL` when overridden in Section 11.1):

- With `PATH_ATTACKS_PER_DAY = None`, every active fleet slot adds `income_per_slot_per_hour`, so founding colonies and upgrading `Fleet base` (`b10`) dynamically grow the empire's raid income along the path.
- Setting `PATH_ATTACKS_PER_DAY` overrides the slot model with a fixed daily attack count (`0` disables attacks).

- `median_attack_df` — per resource (plus `spice_eq` and `net_spice_eq`): median and mean per attack, gross income per fleet-slot hour, and starting gross income per hour today.
- `ATTACK_INCOME_SUMMARY` — attack count, median round-trip seconds, attacks per slot-hour, starting fleet slots, `income_per_slot_per_hour`, `fixed_income_per_hour`, and starting `income_per_hour`.

In [ ]:
def median_attack_income(
    attacks: pd.DataFrame,
    round_trip_sec: np.ndarray,
    planets: list[PlanetState],
    *,
    attacks_per_day: float | None = None,
    fleet_command_center_level: int | None = None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Gross attack income per fleet-slot hour and at today's Fleet base (`b10`) levels.

    Each founded planet provides `1 + b10` simultaneous fleet slots (`1 + fleet_command_center_level` when
    overridden), and each slot launches `3600 / median(round_trip_sec)` median-loot attacks per hour unless
    `attacks_per_day` overrides the total daily pace.
    """
    if fleet_command_center_level is not None and int(fleet_command_center_level) < 0:
        raise ValueError(
            f"fleet_command_center_level must be >= 0 or None, got {fleet_command_center_level!r}"
        )
    fixed_level = None if fleet_command_center_level is None else int(fleet_command_center_level)
    median_rt = float(np.median(round_trip_sec)) if round_trip_sec.size > 0 else 0.0
    per_slot_hour = 3600.0 / median_rt if (median_rt > 0.0 and len(attacks) > 0) else 0.0
    slots_now = sum(
        1 + (planet.buildings.get("b10", 0) if fixed_level is None else fixed_level) for planet in planets
    )
    columns = (*RESOURCE_KEYS, "spice_eq", "net_spice_eq")
    medians = {
        column: float(np.median(attacks[column].to_numpy(dtype=np.float64))) if len(attacks) > 0 else 0.0
        for column in columns
    }
    means = {
        column: float(np.mean(attacks[column].to_numpy(dtype=np.float64))) if len(attacks) > 0 else 0.0
        for column in columns
    }
    if attacks_per_day is None:
        slot_income = {resource: medians[resource] * per_slot_hour for resource in RESOURCE_KEYS}
        fixed_income = dict.fromkeys(RESOURCE_KEYS, 0.0)
        attacks_per_day_now = 24.0 * slots_now * per_slot_hour
        pace_set_by = (
            "Fleet base (b10) slots x median round-trip"
            if fixed_level is None
            else f"PATH_FLEET_COMMAND_CENTER_LEVEL ({fixed_level}) x median round-trip"
        )
    else:
        per_day = max(0.0, float(attacks_per_day))
        slot_income = dict.fromkeys(RESOURCE_KEYS, 0.0)
        fixed_income = {resource: medians[resource] * per_day / 24.0 for resource in RESOURCE_KEYS}
        attacks_per_day_now = per_day
        pace_set_by = "PATH_ATTACKS_PER_DAY"
    income_now = {
        resource: fixed_income[resource] + slots_now * slot_income[resource] for resource in RESOURCE_KEYS
    }
    table = pd.DataFrame(
        [
            {
                "metric": column,
                "attacks": int(len(attacks)),
                "median_per_attack": medians[column],
                "mean_per_attack": means[column],
                "income_per_slot_hour": (
                    slot_income[column] if column in RESOURCE_KEYS else float("nan")
                ),
                "income_per_hour": income_now[column] if column in RESOURCE_KEYS else float("nan"),
            }
            for column in columns
        ]
    )
    summary = {
        "attacks_in_history": int(len(attacks)),
        "median_round_trip_sec": median_rt,
        "attacks_per_slot_per_hour": per_slot_hour,
        "fleet_slots_now": int(slots_now),
        "fleet_command_center_level": fixed_level,
        "attacks_per_day": attacks_per_day_now,
        "pace_set_by": pace_set_by,
        "median_attack_spice_eq": float(game_data.spice_equivalent({r: medians[r] for r in RESOURCE_KEYS})),
        "income_per_slot_per_hour": slot_income,
        "fixed_income_per_hour": fixed_income,
        "income_per_hour": income_now,
        "income_spice_eq_per_hour": float(game_data.spice_equivalent(income_now)),
    }
    return table, summary


median_attack_df, ATTACK_INCOME_SUMMARY = median_attack_income(
    attacks_df,
    round_trip_times_arr,
    live_planet_states,
    attacks_per_day=PATH_ATTACKS_PER_DAY,
    fleet_command_center_level=PATH_FLEET_COMMAND_CENTER_LEVEL,
)

print_section_data(
    f"Section 11.3 data: {ATTACK_INCOME_SUMMARY['fleet_slots_now']} fleet slots, "
    f"{ATTACK_INCOME_SUMMARY['income_spice_eq_per_hour']:,.1f} gross attack SpiceEq/h",
    [
        ("median_attack_df", median_attack_df),
        ("ATTACK_INCOME_SUMMARY", ATTACK_INCOME_SUMMARY),
    ],
)

### 11.4 Simulator, Weekly Reward Ceiling and Lower Bound

`EmpirePathSimulator` plays a path problem forward over the `PATH_REWARD_WEEKS` horizon (`168 * W` hours, or an explicit `horizon_hours` override). `run(strategy)` returns the start and end hour of every step started within the window plus the exact resources produced in each week `1..W` and their discounted SpiceEq reward.

- `building_level_flows` and `planet_hourly_output` compute what a planet makes: the hourly flows of each building level, then the power ratio and the smelting ratio, matching `config_optimizer.ipynb`.
- Time moves from event to event up to `horizon_hours`: a step ends, the stock reaches the cost of the reserved step, or the strategy reviews its estimates. No new step starts at or after `horizon_hours`.
- Every hour, each resource keeps `1 - spare%` (`PATH_RESOURCE_SPARE_PERCENT`) of both planet output and dynamic `Fleet base` (`b10`) attack loot (`1 + b10` slots per founded planet); the remaining `spare%` is set aside for building ships or raid losses (`spare_diverted`). Across each elapsed interval inside `[0, horizon_hours]`, net kept planet production is integrated week by week into `weekly_resources`, valued in SpiceEq (`weekly_spice_eq`), and weighted by `weekly_discount ** (w - 1)` into `discounted_reward`. If all steps finish early (`"reached"`) or stall (`"stuck"`), standing planets continue producing through week `W`.
- `PATH_REWARD_UPPER_BOUND` is an upper bound on `discounted_reward`: the discounted SpiceEq that the empire would produce if every planet already stood at the higher of today's and the configuration's mine levels at full power from hour zero. `lower_bounds` also keeps the full-configuration time lower bound (`PATH_LOWER_BOUND_HOURS`).

- `PRODUCTION_FLOW_KEYS`, `PATH_FLOW_BUILDING_KEYS`, `PATH_MINE_KEYS`, `PATH_ACCELERATOR_KEYS`, `PATH_HOURS_PER_WEEK`, `PATH_NEVER` — flow keys, building groups, hours per week and infinity.
- `PATH_SIMULATOR` — the simulator of `PATH_PROBLEM` with the attack income of Section 11.3 and the `W`-week discount window.
- `path_final_output_df` — hourly output of each planet once the full configuration stands, checked against `config_optimizer.ipynb`.
- `path_lower_bounds_df`, `PATH_LOWER_BOUND_HOURS`, `PATH_REWARD_UPPER_BOUND` — full-configuration time lower bounds and the `W`-week discounted reward ceiling.

In [ ]:
PRODUCTION_FLOW_KEYS: tuple[str, ...] = (
    *RESOURCE_KEYS,
    "pig_iron_used",
    "spice_used",
    "energy_produced",
    "energy_used",
)
# Buildings with hourly flows, the mines among them, and the two that shorten the building times of their planet.
PATH_FLOW_BUILDING_KEYS: tuple[str, ...] = ("b1", "b2", "b3", "b4", "b5", "b6")
PATH_MINE_KEYS: tuple[str, ...] = ("b1", "b2", "b3", "b4")
PATH_ACCELERATOR_KEYS: tuple[str, ...] = ("b7", "b17")
PATH_HOURS_PER_WEEK: float = 168.0
PATH_NEVER: float = float("inf")


def building_level_flows(building_key: str, level: int, context: dict[str, Any]) -> list[float]:
    """Hourly flows of one building level at full power on the context's planet, in `PRODUCTION_FLOW_KEYS` order."""
    economy = building_level_economy(int(building_key[1:]), level, context)
    output = {produced_resource(building_key): economy["production"]}
    return [
        *(float(output.get(resource, 0.0)) for resource in RESOURCE_KEYS),
        economy["pig_iron_used"],
        economy["spice_used"],
        max(0.0, economy["energy"]),
        max(0.0, -economy["energy"]),
    ]


def planet_hourly_output(totals: Any) -> tuple[float, float, float, float, float, float, float]:
    """What a planet makes per hour, from the summed flows of its buildings in `PRODUCTION_FLOW_KEYS` order.

    Returns `(pig_iron_mined, pig_iron_left, metal, kryptonite, spice, power_ratio, smelting_ratio)`, with
    the rules of the configuration model. The mines and the Blast furnace run at `power_ratio`, the share
    of the energy that the planet has. While the furnace runs it makes `metal` from the planet's own Pig
    iron, 2 per Metal, and `pig_iron_left` remains. While it is off, the planet keeps all of
    `pig_iron_mined` and makes no Metal.
    """
    pig_iron, metal, kryptonite, spice, pig_iron_used, spice_used, energy_produced, energy_used = totals
    power_ratio = min(1.0, energy_produced / energy_used) if energy_used > 0.0 else 1.0
    mined = pig_iron * power_ratio
    wanted = pig_iron_used * power_ratio
    smelting_ratio = min(1.0, mined / wanted) if wanted > 0.0 else 1.0
    return (
        mined,
        mined - wanted * smelting_ratio,
        metal * power_ratio * smelting_ratio,
        kryptonite * power_ratio,
        spice * power_ratio - spice_used,
        power_ratio,
        smelting_ratio,
    )


class EmpirePathSimulator:
    """Plays the empire forward from today until every step of a path problem is done.

    `run(strategy)` returns when each step started and ended, in hours from now. The strategy only says
    which steps it wants and in which order (`strategy.proposals`). The rules are the same for every
    strategy:

    - Resources are pooled. A step is paid when it starts and counts when it ends.
    - A planet builds or tears down one level at a time, the account researches one level at a time, and
      a shipyard works on one order at a time. Colonies are founded side by side.
    - A step starts only on a founded planet, with its prerequisites standing, with warehouses on the paying
      planet that hold each resource of its cost, and, for a level to build, with a free field.
    - The first step that the strategy wants and the stock does not cover is the reserved step: the empire
      saves for it. A step further down the list still starts when paying for it delays the reserved step
      by no more than `strategy.patience` times its own duration.
    - The Blast furnaces run only while the reserved step still lacks Metal.
    """

    def __init__(
        self,
        problem: dict[str, Any],
        attack_income: dict[str, float],
        *,
        slot_attack_income: dict[str, float] | None = None,
        fleet_command_center_level: int | None = None,
        spare_percent: dict[str, float] | None = None,
        reward_weeks: int = 26,
        weekly_discount: float = 0.90,
    ) -> None:
        if int(reward_weeks) < 1:
            raise ValueError(f"reward_weeks must be at least 1, got {reward_weeks!r}")
        if not (0.0 < float(weekly_discount) <= 1.0):
            raise ValueError(f"weekly_discount must be in (0, 1], got {weekly_discount!r}")
        if fleet_command_center_level is not None and int(fleet_command_center_level) < 0:
            raise ValueError(
                f"fleet_command_center_level must be >= 0 or None, got {fleet_command_center_level!r}"
            )
        if spare_percent is not None:
            if set(spare_percent) != set(RESOURCE_KEYS):
                raise ValueError(f"spare_percent must set {RESOURCE_KEYS}, got {sorted(spare_percent)}")
            for resource, pct in spare_percent.items():
                if not (0.0 <= float(pct) < 100.0):
                    raise ValueError(f"spare_percent[{resource!r}] must be in [0, 100), got {pct!r}")
        self.spare_percent = tuple(
            float(spare_percent[resource]) if spare_percent is not None else 0.0 for resource in RESOURCE_KEYS
        )
        self.keep = tuple(1.0 - pct / 100.0 for pct in self.spare_percent)
        self.fleet_command_center_level = (
            None if fleet_command_center_level is None else int(fleet_command_center_level)
        )
        self.reward_weeks = int(reward_weeks)
        self.weekly_discount = float(weekly_discount)
        self.horizon_hours = PATH_HOURS_PER_WEEK * self.reward_weeks
        self.week_weights = tuple(self.weekly_discount**week for week in range(self.reward_weeks))
        suffix = [0.0] * (self.reward_weeks + 1)
        for week in range(self.reward_weeks - 1, -1, -1):
            suffix[week] = suffix[week + 1] + PATH_HOURS_PER_WEEK * self.week_weights[week]
        self.suffix_discounted_hours = tuple(suffix)
        self.merchant_rates = tuple(
            1.0 / float(game_data.MERCHANT_RATES_IN_SPICE[resource]) for resource in RESOURCE_KEYS
        )
        self.problem = problem
        self.planets = problem["planets"]
        self.steps = problem["steps"]
        self.speedup = float(problem["speedup_factor"])
        self.fixed_attack = tuple(float(attack_income[resource]) for resource in RESOURCE_KEYS)
        self.slot_attack = tuple(
            float(slot_attack_income[resource]) if slot_attack_income is not None else 0.0
            for resource in RESOURCE_KEYS
        )
        start_slots = sum(
            1
            + (
                planet["levels"].get("b10", 0)
                if self.fleet_command_center_level is None
                else self.fleet_command_center_level
            )
            for planet in self.planets
            if planet["founded"]
        )
        self.attack = tuple(
            self.fixed_attack[pos] + start_slots * self.slot_attack[pos] for pos in range(4)
        )
        self.lab_planet = int(problem["lab_planet"])
        self.central_planet = int(problem["central_planet"])
        self.planet_count = len(self.planets)
        self.research_queue = self.planet_count
        self.queue_count = 2 * self.planet_count + 1
        self.temperature = [int(planet["temperature_c"]) for planet in self.planets]
        self.fields_max = [int(planet["fields_max"]) for planet in self.planets]

        # The steps as parallel lists: a run reads them hundreds of thousands of times.
        self.kind = [step["kind"] for step in self.steps]
        self.step_planet = [int(step["planet"]) for step in self.steps]
        self.key = [step["key"] for step in self.steps]
        self.level_to = [int(step["level_to"]) for step in self.steps]
        self.cost = [tuple(step["cost"]) for step in self.steps]
        # A step is paid for when the stock reaches its cost, give or take the rounding of the arithmetic.
        self.cost_floor = [tuple(amount - (1e-9 * amount + 1e-6) for amount in step["cost"]) for step in self.steps]
        self.cost_spice_eq = [float(step["cost_spice_eq"]) for step in self.steps]
        self.base_hours = [float(step["base_hours"]) for step in self.steps]
        self.queue = [int(step["queue"]) for step in self.steps]
        self.pay_planet = [int(step["pay_planet"]) for step in self.steps]
        self.requires = [tuple(step["requires"]) for step in self.steps]
        self.warehouse_levels = [tuple(step["warehouse_levels"]) for step in self.steps]

        # Chains: the steps of one building on one planet, or of one research, in the order of their levels.
        chains: dict[tuple[str, int, str], list[int]] = {}
        for step in self.steps:
            chains.setdefault(tuple(step["chain"]), []).append(int(step["id"]))
        self.chain_steps = list(chains.values())
        self.step_chain = [0] * len(self.steps)
        self.build_chains: list[dict[str, int]] = [{} for _ in self.planets]
        self.demolish_chains: list[list[int]] = [[] for _ in self.planets]
        self.ship_chains: list[list[int]] = [[] for _ in self.planets]
        self.colonize_step = [-1] * self.planet_count
        self.research_chains: dict[str, int] = {}
        for chain_index, ((kind, planet, key), members) in enumerate(chains.items()):
            for member in members:
                self.step_chain[member] = chain_index
            if kind == "build":
                self.build_chains[planet][key] = chain_index
            elif kind == "demolish":
                self.demolish_chains[planet].append(chain_index)
            elif kind == "ships":
                self.ship_chains[planet].append(chain_index)
            elif kind == "colonize":
                self.colonize_step[planet] = members[0]
            else:
                self.research_chains[key] = chain_index

        # The owners (a planet, or `planet_count` for the research) with a step that needs a given research.
        self.research_dependents: dict[str, set[int]] = {}
        for step in self.steps:
            owner = self.research_queue if step["planet"] < 0 else int(step["planet"])
            for required_owner, key, _level in step["requires"]:
                if required_owner < 0:
                    self.research_dependents.setdefault(key, set()).add(owner)

        self._flow_cache: dict[tuple[str, int, int, int], tuple[float, ...]] = {}
        self._solar_cache: dict[tuple[int, int], float] = {}
        self._accelerator_cache: dict[tuple[int, int, int], float] = {}

        # The empire once the configuration stands: its levels, its time divisors and what it supplies per hour.
        final_research = problem["research_target"]
        self.final_levels = [dict(planet["target_levels"]) for planet in self.planets]
        self.final_solar = [
            int(planet["solar_cells"]) + int(planet["missing_ships"].get("s9", 0)) for planet in self.planets
        ]
        self.final_building_divisor = [
            building_time_divisor(levels.get("b7", 0), levels.get("b17", 0), self.speedup)
            for levels in self.final_levels
        ]
        self.final_shipyard_divisor = [
            shipyard_time_divisor(levels.get("b9", 0), levels.get("b17", 0), self.speedup)
            for levels in self.final_levels
        ]
        self.final_research_divisor = research_time_divisor(
            self.final_levels[self.lab_planet].get("b8", 0) if self.lab_planet >= 0 else 0, self.speedup
        )
        self.final_output = [
            planet_hourly_output(
                self.planet_totals(planet, levels, self.final_solar[planet], final_research["r1"], final_research["r2"])
            )
            for planet, levels in enumerate(self.final_levels)
        ]
        self.final_fleet_slots = self.fleet_slots_of(self.final_levels, [True] * self.planet_count)
        self.final_supply = self.supply_of(self.final_output, [True] * self.planet_count, self.final_fleet_slots)
        # Gross planet output per hour of the configuration (matches config_optimizer_result.json) and net kept output.
        self.final_output_total = tuple(
            sum(output[position] for output in self.final_output) for position in (1, 2, 3, 4)
        )
        self.final_net_output_total = tuple(
            self.keep[pos] * self.final_output_total[pos] for pos in range(4)
        )
        # Steps whose standalone time at the configuration's speed fits in the W-week horizon.
        self.in_horizon = [False] * len(self.steps)
        horizon_need = [0.0, 0.0, 0.0, 0.0]
        for step in range(len(self.steps)):
            planet, kind = self.step_planet[step], self.kind[step]
            divisor = (
                1.0
                if kind == "colonize"
                else self.final_research_divisor
                if kind == "research"
                else self.final_shipyard_divisor[planet]
                if kind == "ships"
                else self.final_building_divisor[planet]
            )
            if self.base_hours[step] / divisor <= self.horizon_hours:
                self.in_horizon[step] = True
                for position in range(4):
                    horizon_need[position] += self.cost[step][position]
        self.horizon_need_init = tuple(horizon_need)
        self.reward_ceiling_spice_eq_per_hour = self._reward_ceiling_per_hour()
        self.reward_upper_bound = self.reward_ceiling_spice_eq_per_hour * self.suffix_discounted_hours[0]

    def _reward_ceiling_per_hour(self) -> float:
        """Highest SpiceEq per hour the founded planets can ever make between today and the configuration."""
        mining = max(self.problem["research_levels"].get("r1", 0), self.problem["research_target"].get("r1", 0))
        energy = max(self.problem["research_levels"].get("r2", 0), self.problem["research_target"].get("r2", 0))
        rates = self.merchant_rates
        total = 0.0
        for planet in range(self.planet_count):
            top = {
                key: max(self.planets[planet]["levels"].get(key, 0), self.final_levels[planet].get(key, 0))
                for key in PATH_MINE_KEYS
            }
            flows = {
                key: self.level_flows(key, level, self.temperature[planet], mining, energy)
                for key, level in top.items()
                if level > 0
            }
            mined = flows["b1"][0] if "b1" in flows else 0.0
            smelted = min(flows["b2"][1], mined / 2.0) if "b2" in flows else 0.0
            kryptonite = flows["b3"][2] if "b3" in flows else 0.0
            spice = flows["b4"][3] if "b4" in flows else 0.0
            keep = self.keep
            smelt_bonus = max(0.0, smelted * (keep[1] * rates[1] - 2.0 * keep[0] * rates[0]))
            total += (
                mined * keep[0] * rates[0]
                + smelt_bonus
                + kryptonite * keep[2] * rates[2]
                + spice * keep[3] * rates[3]
            )
        return total

    def remaining_discounted_hours(self, hour: float) -> float:
        """Discounted hours remaining between `hour` and `horizon_hours` (`sum` of `gamma**(w-1)` over hours left)."""
        if hour >= self.horizon_hours:
            return 0.0
        if hour <= 0.0:
            return self.suffix_discounted_hours[0]
        week = int(hour // PATH_HOURS_PER_WEEK)
        if week >= self.reward_weeks:
            return 0.0
        in_week = (week + 1) * PATH_HOURS_PER_WEEK - hour
        return self.week_weights[week] * in_week + self.suffix_discounted_hours[week + 1]

    # ------------------------------------------------------------------ production

    def level_flows(self, key: str, level: int, temperature: int, mining: int, energy: int) -> tuple[float, ...]:
        """Hourly flows of a building level, cached. Mining technology only changes the mines and Energy
        technology only the power plants, so each cached row depends on one of the two."""
        token = (key, level, temperature, mining if key in PATH_MINE_KEYS else energy)
        flows = self._flow_cache.get(token)
        if flows is None:
            context = {
                "speedup_factor": self.speedup,
                "planet_temperature_c": temperature,
                "r1_mining_tech": mining,
                "r2_energy_tech": energy,
            }
            flows = self._flow_cache[token] = tuple(building_level_flows(key, level, context))
        return flows

    def planet_totals(
        self, planet: int, levels: dict[str, int], solar_cells: int, mining: int, energy: int
    ) -> list[float]:
        """Summed hourly flows of a planet's buildings and Solar cells at the given research levels."""
        temperature = self.temperature[planet]
        totals = [0.0] * len(PRODUCTION_FLOW_KEYS)
        for key in PATH_FLOW_BUILDING_KEYS:
            level = levels.get(key, 0)
            if level > 0:
                for position, flow in enumerate(self.level_flows(key, level, temperature, mining, energy)):
                    totals[position] += flow
        if solar_cells > 0:
            token = (temperature, energy)
            per_cell = self._solar_cache.get(token)
            if per_cell is None:
                per_cell = self._solar_cache[token] = solar_cell_energy_per_unit(temperature) * technology_multiplier(
                    "r2", energy
                )
            totals[6] += solar_cells * per_cell
        return totals

    def changed_totals(self, planet: int, changes: dict[str, int]) -> list[float]:
        """The planet's summed flows of the moment, with some building levels replaced by those of `changes`."""
        totals = list(self.totals[planet])
        levels = self.levels[planet]
        temperature = self.temperature[planet]
        mining, energy = self.research["r1"], self.research["r2"]
        for key, new_level in changes.items():
            if key not in PATH_FLOW_BUILDING_KEYS:
                continue
            old_level = levels.get(key, 0)
            if old_level > 0:
                for position, flow in enumerate(self.level_flows(key, old_level, temperature, mining, energy)):
                    totals[position] -= flow
            if new_level > 0:
                for position, flow in enumerate(self.level_flows(key, new_level, temperature, mining, energy)):
                    totals[position] += flow
        return totals

    def fleet_slots_of(self, levels_list: list[dict[str, int]], founded: list[bool]) -> int:
        """Active simultaneous fleet slots (`1 + b10` per founded planet, or `1 + fixed` override)."""
        fixed = self.fleet_command_center_level
        return sum(
            1 + (levels_list[planet].get("b10", 0) if fixed is None else fixed)
            for planet in range(self.planet_count)
            if founded[planet]
        )

    def gross_attack_of(self, slots: int) -> tuple[float, float, float, float]:
        """Gross hourly attack loot at `slots` active fleet slots (before deducting spare%)."""
        return tuple(self.fixed_attack[pos] + slots * self.slot_attack[pos] for pos in range(4))

    def supply_of(
        self, outputs: list[tuple[float, ...]], founded: list[bool], slots: int | None = None
    ) -> tuple[float, float, float, float]:
        """Net kept hourly supply (`1 - spare%` of founded planets + fleet-slot attacks)."""
        active_slots = self.fleet_slots_of(self.levels, founded) if slots is None else int(slots)
        gross = list(self.gross_attack_of(active_slots))
        for output, is_founded in zip(outputs, founded):
            if is_founded:
                gross[0] += output[0]
                gross[1] += output[2]
                gross[2] += output[3]
                gross[3] += output[4]
        return tuple(self.keep[pos] * gross[pos] for pos in range(4))

    def _refresh_planet(self, planet: int) -> None:
        """Recompute the flows and the output of a planet after one of its levels or a research changed."""
        self.totals[planet] = self.planet_totals(
            planet, self.levels[planet], self.solar[planet], self.research["r1"], self.research["r2"]
        )
        self.output[planet] = planet_hourly_output(self.totals[planet])

    def _refresh_rates(self) -> None:
        """Net kept hourly income of the empire (`1 - spare%`) with furnaces off/on, and active fleet slots."""
        mined = left = metal = kry = spice = 0.0
        for planet in range(self.planet_count):
            if self.founded[planet]:
                out = self.output[planet]
                mined += out[0]
                left += out[1]
                metal += out[2]
                kry += out[3]
                spice += out[4]
        self.fleet_slots = self.fleet_slots_of(self.levels, self.founded)
        attack = self.gross_attack_of(self.fleet_slots)
        self.attack_now = attack
        keep = self.keep
        self.supply = (
            keep[0] * (mined + attack[0]),
            keep[1] * (metal + attack[1]),
            keep[2] * (kry + attack[2]),
            keep[3] * (spice + attack[3]),
        )
        self.supply_version += 1
        self.furnace_metal = metal
        self.rate_off = (
            keep[0] * (mined + attack[0]),
            keep[1] * attack[1],
            keep[2] * (kry + attack[2]),
            keep[3] * (spice + attack[3]),
        )
        self.rate_on = (
            keep[0] * (left + attack[0]),
            keep[1] * (metal + attack[1]),
            keep[2] * (kry + attack[2]),
            keep[3] * (spice + attack[3]),
        )

    def lower_bounds(self) -> list[tuple[str, float]]:
        """Hours that no path can beat, as `(limit, hours)` rows. The largest is the lower bound of the path.

        - A queue cannot be done sooner than its work takes at its best speed (`queue_work`), after the
          planet is founded.
        - A resource cannot arrive sooner than the highest supply that the empire can ever have delivers
          what the steps cost beyond the stock. That supply takes every mine at the highest level it has on
          the way, at full power and with the planned research. Pig iron also pays for the Metal that the
          furnaces must smelt, 2 per Metal.
        """
        self._load_start()
        count = self.planet_count
        least, _ = self.queue_work()
        founding = [
            self.base_hours[self.colonize_step[planet]] if self.colonize_step[planet] >= 0 else 0.0
            for planet in range(count)
        ]
        rows = [
            (f"building queue of {self.planets[planet]['coords']}", founding[planet] + least[planet])
            for planet in range(count)
        ]
        rows.append(("research queue", least[count]))
        rows.append(("shipyards", max(founding[planet] + least[count + 1 + planet] for planet in range(count))))

        mining = max(self.problem["research_levels"].get("r1", 0), self.problem["research_target"].get("r1", 0))
        energy = max(self.problem["research_levels"].get("r2", 0), self.problem["research_target"].get("r2", 0))
        top_levels = [
            {
                key: max(self.levels[planet].get(key, 0), self.final_levels[planet].get(key, 0))
                for key in (*PATH_MINE_KEYS, "b10")
            }
            for planet in range(count)
        ]
        top_slots = self.fleet_slots_of(top_levels, [True] * count)
        top_attack = self.gross_attack_of(top_slots)
        ceiling = [0.0, 0.0, 0.0, 0.0]
        for planet in range(count):
            flows = {
                key: self.level_flows(key, level, self.temperature[planet], mining, energy)
                for key, level in top_levels[planet].items()
                if key in PATH_MINE_KEYS and level > 0
            }
            pig_iron = flows["b1"][0] if "b1" in flows else 0.0
            ceiling[0] += pig_iron
            ceiling[1] += min(flows["b2"][1], pig_iron / 2.0) if "b2" in flows else 0.0
            ceiling[2] += flows["b3"][2] if "b3" in flows else 0.0
            ceiling[3] += flows["b4"][3] if "b4" in flows else 0.0
        keep = self.keep
        metal_missing = max(0.0, self.need[1] - self.stock[1])
        missing = (
            max(0.0, self.need[0] - self.stock[0] + 2.0 * metal_missing),
            metal_missing,
            max(0.0, self.need[2] - self.stock[2]),
            max(0.0, self.need[3] - self.stock[3]),
        )
        smelt_extra = max(0.0, 2.0 * (keep[1] - keep[0]) * ceiling[1])
        supply = (
            keep[0] * (ceiling[0] + top_attack[0]) + 2.0 * keep[1] * top_attack[1] + smelt_extra,
            keep[1] * (ceiling[1] + top_attack[1]),
            keep[2] * (ceiling[2] + top_attack[2]),
            keep[3] * (ceiling[3] + top_attack[3]),
        )
        labels = ("Pig iron, with the Metal to smelt", "Metal", "Kryptonite", "Spice")
        for position in range(4):
            if missing[position] <= 0.0:
                hours = 0.0
            else:
                hours = missing[position] / supply[position] if supply[position] > 0.0 else PATH_NEVER
            rows.append((f"supply of {labels[position]}", hours))
        return rows

    # ------------------------------------------------------------------ run

    def run(self, strategy: Any, *, horizon_hours: float | None = None) -> dict[str, Any]:
        """Simulate the path that `strategy` produces up to `horizon_hours` (defaults to `168 * reward_weeks`).

        `status` is `"reached"` when every step finishes by the limit, `"horizon"` when the `W`-week window
        ends with steps still unstarted or running, and `"stuck"` when no step can ever start again. Weekly
        production and `discounted_reward` always cover all `reward_weeks` weeks.
        """
        limit = self.horizon_hours if horizon_hours is None else float(horizon_hours)
        self._reset(strategy, limit_hours=limit)
        events = 0
        event_limit = 80 * len(self.steps) + 10_000
        status = "reached"
        while True:
            if self.clock < limit:
                self._decide()
            if self.unstarted == 0 and not self.heap:
                status = "reached"
                break
            if self.clock >= limit:
                status = "horizon"
                break
            until_done = self.heap[0][0] - self.clock if self.heap else PATH_NEVER
            until_stock = self._hours_to_stock_event()
            until_review = PATH_NEVER
            if self.unstarted > 0 and (self.heap or until_stock < PATH_NEVER):
                until_review = self.review_at - self.clock
            elapsed = min(until_done, until_stock, until_review)
            if elapsed == PATH_NEVER:
                if self.reserved < 0 and limit < PATH_NEVER:
                    self._advance(max(0.0, limit - self.clock), ends_a_step=False)
                    status = "reached" if (self.unstarted == 0 and not self.heap) else "horizon"
                else:
                    status = "stuck"
                break
            until_limit = limit - self.clock
            if until_limit < elapsed:
                self._advance(max(0.0, until_limit), ends_a_step=False)
                status = "reached" if (self.unstarted == 0 and not self.heap) else "horizon"
                break
            events += 1
            if events > event_limit:
                raise RuntimeError(f"The path simulation did not finish within {event_limit} events")
            self._advance(max(0.0, elapsed), ends_a_step=elapsed == until_done)
        if self.clock < self.horizon_hours:
            self._accrue_weekly(self.clock, self.horizon_hours - self.clock)
        rates = self.merchant_rates
        weekly_resources = [tuple(row) for row in self.weekly_resources]
        weekly_spice_eq = tuple(
            row[0] * rates[0] + row[1] * rates[1] + row[2] * rates[2] + row[3] * rates[3] for row in weekly_resources
        )
        discounted_reward = float(
            sum(weight * amount for weight, amount in zip(self.week_weights, weekly_spice_eq))
        )
        completed = sum(1 for end in self.end if end is not None and end <= self.clock)
        started_count = len(self.steps) - self.unstarted
        return {
            "status": status,
            "hours": self.clock if status == "reached" else PATH_NEVER,
            "stopped_at_hours": self.clock,
            "order": list(self.order),
            "start": list(self.start),
            "end": list(self.end),
            "steps_started": started_count,
            "steps_completed": completed,
            "steps_running": started_count - completed,
            "waiting_for_step": self.reserved if status == "stuck" else -1,
            "metal_smelted": self.metal_smelted,
            "smelting_spans": [tuple(span) for span in self.smelting_spans],
            "events": events,
            "reviews": self.reviews,
            "final_stock": tuple(self.stock),
            "final_levels": [dict(levels) for levels in self.levels],
            "final_research": dict(self.research),
            "final_solar": list(self.solar),
            "final_founded": list(self.founded),
            "start_fleet_slots": int(self.start_fleet_slots),
            "final_fleet_slots": int(self.fleet_slots),
            "loot_collected": tuple(self.loot_collected),
            "spare_diverted": tuple(self.spare_diverted),
            "weekly_resources": weekly_resources,
            "weekly_spice_eq": weekly_spice_eq,
            "discounted_reward": discounted_reward,
            "total_spice_eq_produced": float(sum(weekly_spice_eq)),
        }

    def _reset(self, strategy: Any, *, limit_hours: float | None = None) -> None:
        self._load_start()
        self.run_limit_hours = self.horizon_hours if limit_hours is None else float(limit_hours)
        self.strategy = strategy
        strategy.begin(self)
        self._review()

    def _load_start(self) -> None:
        """Set the state of today: the levels, the stock, and every step still to start."""
        count = self.planet_count
        self.clock = 0.0
        self.stock = [float(amount) for amount in self.problem["start_stock"]]
        self.levels = [dict(planet["levels"]) for planet in self.planets]
        self.research = dict(self.problem["research_levels"])
        self.solar = [int(planet["solar_cells"]) for planet in self.planets]
        self.founded = [bool(planet["founded"]) for planet in self.planets]
        # Fields in use: the levels that stand plus the level under construction.
        self.fields_used = [sum(levels.values()) for levels in self.levels]
        self.next_step = [0] * len(self.chain_steps)
        self.queue_step = [-1] * self.queue_count
        self.queue_end = [0.0] * self.queue_count
        self.heap: list[tuple[float, int]] = []
        self.started = [False] * len(self.steps)
        # The steps in the order in which they start. Two steps can start in the same hour; this order tells
        # which came first, such as the founding of a colony before the first level on it.
        self.order: list[int] = []
        self.start: list[float | None] = [None] * len(self.steps)
        self.end: list[float | None] = [None] * len(self.steps)
        self.unstarted = len(self.steps)
        # What the steps not started yet cost, and the hours of work they are on each queue before the time divisors.
        self.need = [sum(cost[position] for cost in self.cost) for position in range(4)]
        self.work_accelerators = [0.0] * count
        self.work_buildings = [0.0] * count
        self.work_ships = [0.0] * count
        self.work_research = 0.0
        for step in range(len(self.steps)):
            self._count_work(step, 1.0)
        self.supply_version = 0
        self.totals = [[0.0] * len(PRODUCTION_FLOW_KEYS) for _ in range(count)]
        self.output: list[tuple[float, ...]] = [(0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0)] * count
        for planet in range(count):
            self._refresh_planet(planet)
        self._refresh_rates()
        self.reserved = -1
        self.smelting = False
        self.metal_smelted = 0.0
        # The spans of hours during which the furnaces ran.
        self.smelting_spans: list[list[float]] = []
        self.reviews = 0
        self.completions_since_review = 0
        self.review_at = PATH_NEVER
        self.review_every = 1
        # The planets (and, as owner `planet_count`, the research) whose steps the strategy must look at again.
        self.dirty = set(range(count + 1))
        self.run_limit_hours = self.horizon_hours
        self.horizon_need = list(self.horizon_need_init)
        self.weekly_resources = [[0.0, 0.0, 0.0, 0.0] for _ in range(self.reward_weeks)]
        self.loot_collected = [0.0, 0.0, 0.0, 0.0]
        self.spare_diverted = [0.0, 0.0, 0.0, 0.0]
        self.start_fleet_slots = self.fleet_slots

    def _count_work(self, step: int, sign: float) -> None:
        """Add (or, with `sign` -1, remove) a step's base hours to the work still to start on its queue."""
        kind, planet = self.kind[step], self.step_planet[step]
        if kind == "research":
            self.work_research += sign * self.base_hours[step]
        elif kind == "ships":
            self.work_ships[planet] += sign * self.base_hours[step]
        elif kind == "build" and self.key[step] in PATH_ACCELERATOR_KEYS:
            self.work_accelerators[planet] += sign * self.base_hours[step]
        elif kind in ("build", "demolish"):
            self.work_buildings[planet] += sign * self.base_hours[step]

    def accelerator_hours(self, planet: int) -> float:
        """Least hours for the accelerator levels of a planet that have not started yet.

        The Development center and the Microsystem accelerator speed each other up, so the order matters:
        this is the fastest order, each level at the speed that the levels before it give. Prerequisites
        are left out, which can only make the result smaller.
        """
        pending: list[list[int]] = []
        for key in PATH_ACCELERATOR_KEYS:
            chain = self.build_chains[planet].get(key)
            pending.append(self.chain_steps[chain][self.next_step[chain]:] if chain is not None else [])
        centers, microsystems = pending
        if not centers and not microsystems:
            return 0.0
        token = (planet, len(centers), len(microsystems))
        cached = self._accelerator_cache.get(token)
        if cached is not None:
            return cached
        # The level that stands, or is under way, before the first level that has not started.
        center_level = self.level_to[centers[0]] - 1 if centers else self.final_levels[planet].get("b7", 0)
        microsystem_level = (
            self.level_to[microsystems[0]] - 1 if microsystems else self.final_levels[planet].get("b17", 0)
        )
        best = [[PATH_NEVER] * (len(microsystems) + 1) for _ in range(len(centers) + 1)]
        best[0][0] = 0.0
        for done_centers in range(len(centers) + 1):
            for done_microsystems in range(len(microsystems) + 1):
                hours = best[done_centers][done_microsystems]
                divisor = building_time_divisor(
                    center_level + done_centers, microsystem_level + done_microsystems, self.speedup
                )
                if done_centers < len(centers):
                    longer = hours + self.base_hours[centers[done_centers]] / divisor
                    if longer < best[done_centers + 1][done_microsystems]:
                        best[done_centers + 1][done_microsystems] = longer
                if done_microsystems < len(microsystems):
                    longer = hours + self.base_hours[microsystems[done_microsystems]] / divisor
                    if longer < best[done_centers][done_microsystems + 1]:
                        best[done_centers][done_microsystems + 1] = longer
        self._accelerator_cache[token] = best[-1][-1]
        return best[-1][-1]

    def queue_work(self) -> tuple[list[float], list[float]]:
        """Hours of work left on every queue, the running step included: `(least, at_todays_speed)`.

        `least` is the soonest the queue can be done: a planet's accelerator levels first and in their
        fastest order, everything else at the speed once the accelerators of the configuration stand (for
        the research: with the Research lab of the configuration). `at_todays_speed` is what the same work
        takes when no accelerator level is added.
        """
        count = self.planet_count
        least = [0.0] * self.queue_count
        for queue in range(self.queue_count):
            if self.queue_step[queue] >= 0:
                least[queue] = max(0.0, self.queue_end[queue] - self.clock)
        slow = list(least)
        for planet in range(count):
            levels = self.levels[planet]
            divisor = building_time_divisor(levels.get("b7", 0), levels.get("b17", 0), self.speedup)
            least[planet] += (
                self.accelerator_hours(planet) + self.work_buildings[planet] / self.final_building_divisor[planet]
            )
            slow[planet] += (self.work_accelerators[planet] + self.work_buildings[planet]) / divisor
            yard = count + 1 + planet
            least[yard] += self.work_ships[planet] / self.final_shipyard_divisor[planet]
            slow[yard] += self.work_ships[planet] / shipyard_time_divisor(
                levels.get("b9", 0), levels.get("b17", 0), self.speedup
            )
        least[count] += self.work_research / self.final_research_divisor
        lab_level = self.levels[self.lab_planet].get("b8", 0) if self.lab_planet >= 0 else 0
        slow[count] += self.work_research / research_time_divisor(lab_level, self.speedup)
        return least, slow

    def missing_resources(self) -> tuple[float, float, float, float]:
        """What the steps not started yet cost beyond the stock. Pig iron includes the 2 per Metal still to smelt."""
        metal = max(0.0, self.need[1] - self.stock[1])
        return (
            max(0.0, self.need[0] + 2.0 * metal - self.stock[0]),
            metal,
            max(0.0, self.need[2] - self.stock[2]),
            max(0.0, self.need[3] - self.stock[3]),
        )

    def horizon_missing_resources(self) -> tuple[float, float, float, float]:
        """Missing resources among steps whose standalone duration fits inside the W-week horizon."""
        metal = max(0.0, self.horizon_need[1] - self.stock[1])
        return (
            max(0.0, self.horizon_need[0] + 2.0 * metal - self.stock[0]),
            metal,
            max(0.0, self.horizon_need[2] - self.stock[2]),
            max(0.0, self.horizon_need[3] - self.stock[3]),
        )

    def _accrue_weekly(self, start_hour: float, elapsed: float) -> None:
        """Integrate net kept founded-planet production (`1 - spare%`) into weekly buckets."""
        if elapsed <= 0.0 or start_hour >= self.horizon_hours:
            return
        end_hour = min(self.horizon_hours, start_hour + elapsed)
        g0 = g1 = g2 = g3 = 0.0
        for planet in range(self.planet_count):
            if self.founded[planet]:
                made = self.output[planet]
                g0 += made[1]
                g1 += made[2]
                g2 += made[3]
                g3 += made[4]
        keep = self.keep
        r0, r1, r2, r3 = keep[0] * g0, keep[1] * g1, keep[2] * g2, keep[3] * g3
        week = int(start_hour // PATH_HOURS_PER_WEEK)
        cursor = start_hour
        while cursor < end_hour and week < self.reward_weeks:
            boundary = min(end_hour, (week + 1) * PATH_HOURS_PER_WEEK)
            span = boundary - cursor
            if span > 0.0:
                bucket = self.weekly_resources[week]
                bucket[0] += r0 * span
                bucket[1] += r1 * span
                bucket[2] += r2 * span
                bucket[3] += r3 * span
            cursor = boundary
            week += 1

    def duration(self, step: int) -> float:
        """Hours that a step takes when it starts now, with the accelerators that stand now."""
        kind, planet = self.kind[step], self.step_planet[step]
        if kind == "colonize":
            return self.base_hours[step]
        if kind == "research":
            return self.base_hours[step] / research_time_divisor(
                self.levels[self.lab_planet].get("b8", 0), self.speedup
            )
        levels = self.levels[planet]
        if kind == "ships":
            return self.base_hours[step] / shipyard_time_divisor(
                levels.get("b9", 0), levels.get("b17", 0), self.speedup
            )
        return self.base_hours[step] / building_time_divisor(levels.get("b7", 0), levels.get("b17", 0), self.speedup)

    def blockers(self, step: int) -> list[tuple]:
        """What keeps a step from starting, apart from its queue and the resources.

        `("colony", planet)`: the planet is not founded. `("level", owner, key, level)`: a prerequisite is
        missing (`owner` -1 for a research). `("storage", planet, resource index, level)`: a warehouse of
        the paying planet is too small. `("field", planet)`: the planet has no free field.
        """
        blockers: list[tuple] = []
        kind, planet = self.kind[step], self.step_planet[step]
        if planet >= 0 and kind != "colonize" and not self.founded[planet]:
            blockers.append(("colony", planet))
        for owner, key, level in self.requires[step]:
            have = self.research.get(key, 0) if owner < 0 else self.levels[owner].get(key, 0)
            if have < level:
                blockers.append(("level", owner, key, level))
        pay_planet = self.pay_planet[step]
        pay_levels = self.levels[pay_planet]
        for position, needed in enumerate(self.warehouse_levels[step]):
            if needed > 0 and needed > pay_levels.get(WAREHOUSE_KEYS[position], 0):
                blockers.append(("storage", pay_planet, position, needed))
        if kind == "build" and self.fields_used[planet] >= self.fields_max[planet]:
            blockers.append(("field", planet))
        return blockers

    def is_next_of_its_chain(self, step: int) -> bool:
        """True when every earlier level of the step's building or research has started."""
        chain = self.step_chain[step]
        position = self.next_step[chain]
        return position < len(self.chain_steps[chain]) and self.chain_steps[chain][position] == step

    def wait_hours(self, stock: list[float], step: int) -> float:
        """Hours until `stock` and the income cover the cost of `step`, the furnaces smelting exactly the
        Metal that lacks. Infinite when nothing supplies a resource that lacks."""
        cost, floor = self.cost[step], self.cost_floor[step]
        wait = 0.0
        for position in (2, 3):
            if stock[position] < floor[position]:
                rate = self.rate_off[position]
                if rate <= 0.0:
                    return PATH_NEVER
                wait = max(wait, (cost[position] - stock[position]) / rate)
        pig_iron_lacking = cost[0] - stock[0]
        smelting_hours = 0.0
        if stock[1] < floor[1]:
            if self.rate_on[1] <= 0.0:
                return PATH_NEVER
            smelting_hours = (cost[1] - stock[1]) / self.rate_on[1]
            pig_iron_lacking -= self.rate_on[0] * smelting_hours
        if pig_iron_lacking > cost[0] - floor[0]:
            if self.rate_off[0] <= 0.0:
                return PATH_NEVER
            smelting_hours += pig_iron_lacking / self.rate_off[0]
        return max(wait, smelting_hours)

    def _hours_to_stock_event(self) -> float:
        """Hours until the stock next matters: the Metal of the reserved step is there, or all of its cost."""
        if self.reserved < 0:
            return PATH_NEVER
        cost, floor = self.cost[self.reserved], self.cost_floor[self.reserved]
        if self.smelting:
            rate = self.rate_on[1]
            return (cost[1] - self.stock[1]) / rate if rate > 0.0 else PATH_NEVER
        wait = 0.0
        for position in (0, 2, 3):
            if self.stock[position] < floor[position]:
                rate = self.rate_off[position]
                if rate <= 0.0:
                    return PATH_NEVER
                wait = max(wait, (cost[position] - self.stock[position]) / rate)
        return wait

    def _advance(self, elapsed: float, *, ends_a_step: bool) -> None:
        """Let `elapsed` hours pass: collect the income, finish the steps that end, review when it is time.

        The income is added for `elapsed` itself, not for a difference of clock readings: late in a long
        path, an hour count of millions cannot resolve the seconds that a cheap step still waits for.
        """
        if elapsed > 0.0:
            self._accrue_weekly(self.clock, elapsed)
            rates = self.rate_on if self.smelting else self.rate_off
            gross_attack = self.attack_now
            keep = self.keep
            for position in range(4):
                self.loot_collected[position] += keep[position] * gross_attack[position] * elapsed
                if keep[position] > 0.0:
                    gross_rate = rates[position] / keep[position]
                    self.spare_diverted[position] += (1.0 - keep[position]) * gross_rate * elapsed
            for position in range(4):
                self.stock[position] += rates[position] * elapsed
            if self.smelting:
                self.metal_smelted += self.furnace_metal * elapsed
                if self.smelting_spans and self.smelting_spans[-1][1] == self.clock:
                    self.smelting_spans[-1][1] = self.clock + elapsed
                else:
                    self.smelting_spans.append([self.clock, self.clock + elapsed])
        self.clock = self.heap[0][0] if ends_a_step else self.clock + elapsed
        while self.heap and self.heap[0][0] <= self.clock:
            self._complete(heapq.heappop(self.heap)[1])
        if self.completions_since_review >= self.review_every or self.clock >= self.review_at:
            self._review()

    def _review(self) -> None:
        """Let the strategy re-estimate the rest of the path, and plan the next review.

        The strategy returns the hours it expects the rest to take, after how many finished steps it wants
        the next review, and whether its view changed enough to look at every step again. A review also
        comes after 2 % of those hours, and never sooner than the clock can tell apart.
        """
        self.reviews += 1
        self.completions_since_review = 0
        horizon, self.review_every, look_again = self.strategy.review(self)
        if 0.0 < horizon < PATH_NEVER:
            self.review_at = self.clock + max(0.02 * horizon, 1e-3, 1e-9 * self.clock)
        else:
            self.review_at = PATH_NEVER
        if look_again:
            self.dirty.update(range(self.planet_count + 1))

    def _start(self, step: int) -> None:
        cost = self.cost[step]
        for position in range(4):
            self.stock[position] = max(0.0, self.stock[position] - cost[position])
            self.need[position] -= cost[position]
            if self.in_horizon[step]:
                self.horizon_need[position] = max(0.0, self.horizon_need[position] - cost[position])
        hours = self.duration(step)
        self.started[step] = True
        self.order.append(step)
        self.start[step] = self.clock
        self.end[step] = self.clock + hours
        heapq.heappush(self.heap, (self.clock + hours, step))
        self.unstarted -= 1
        self.next_step[self.step_chain[step]] += 1
        self._count_work(step, -1.0)
        queue, planet = self.queue[step], self.step_planet[step]
        if queue >= 0:
            self.queue_step[queue] = step
            self.queue_end[queue] = self.clock + hours
        if self.kind[step] == "build":
            self.fields_used[planet] += 1
        if planet < 0:
            # The next research level may need a warehouse or a Research lab level that can be built meanwhile.
            self.dirty.add(self.research_queue)

    def _complete(self, step: int) -> None:
        kind, planet, key = self.kind[step], self.step_planet[step], self.key[step]
        if self.queue[step] >= 0:
            self.queue_step[self.queue[step]] = -1
        self.completions_since_review += 1
        if kind == "research":
            self.research[key] = self.level_to[step]
            self.dirty.add(self.research_queue)
            # The level can be the prerequisite of a step on any planet that waits for this research.
            self.dirty.update(self.research_dependents.get(key, ()))
            if key in ("r1", "r2"):
                # Mining and Energy technology change what every planet makes.
                for other in range(self.planet_count):
                    self._refresh_planet(other)
                self._refresh_rates()
                self.dirty.update(range(self.planet_count))
            return
        if kind == "colonize":
            self.founded[planet] = True
            self._refresh_rates()
        elif kind == "ships":
            if key == "s9":
                self.solar[planet] += self.level_to[step]
                self._refresh_planet(planet)
                self._refresh_rates()
        else:
            if self.level_to[step] > 0:
                self.levels[planet][key] = self.level_to[step]
            else:
                self.levels[planet].pop(key, None)
            if kind == "demolish":
                self.fields_used[planet] -= 1
            if key in PATH_FLOW_BUILDING_KEYS:
                self._refresh_planet(planet)
                if self.founded[planet]:
                    self._refresh_rates()
            elif key == "b10" and self.founded[planet] and self.fleet_command_center_level is None:
                self._refresh_rates()
        self.dirty.add(planet)
        if planet == self.lab_planet:
            # The Research lab and the warehouses that research is paid from stand on this planet.
            self.dirty.add(self.research_queue)
        if planet == self.central_planet:
            # Colony ships are paid from the warehouses of the central planet.
            self.dirty.update(other for other in range(self.planet_count) if not self.founded[other])

    def _full_horizon_hours(self) -> float:
        """Optimistic hours needed for all remaining steps (queues at best speed vs remaining cost at final supply)."""
        work, _ = self.queue_work()
        missing = self.missing_resources()
        horizon = max(work)
        for position in range(4):
            if missing[position] > 0.0:
                supply = self.final_supply[position]
                horizon = max(horizon, missing[position] / supply if supply > 0.0 else PATH_NEVER)
        return horizon

    def _decide(self) -> None:
        """Start every step that the strategy wants now and the empire can pay; note what it saves for."""
        for _ in range(self.queue_count + 2):
            if not self._start_proposals(self.strategy.proposals(self, forced=False)):
                break
        if self.reserved < 0 and not self.heap and self.unstarted > 0:
            allow_forced = (
                self.clock == 0.0
                or self.run_limit_hours == PATH_NEVER
                or self._full_horizon_hours() <= self.run_limit_hours
            )
            if allow_forced:
                # Nothing runs and the strategy wants nothing: take whatever can start, so that the path goes on.
                for _ in range(self.queue_count + 2):
                    if not self._start_proposals(self.strategy.proposals(self, forced=True)):
                        break

    def _start_proposals(self, proposals: list[tuple[tuple, int]]) -> bool:
        """Start the proposed steps that the stock pays for, best first; return whether any started.

        The first proposal that the stock does not cover becomes the reserved step. A later proposal still
        starts when paying for it delays the reserved step by no more than `patience` times its own
        duration and, on the queue of the reserved step, when it is done before that step can be paid.
        Whatever the strategy proposes, a step only starts or is saved for when the rules allow it.
        """
        reserved = -1
        limit = 0.0
        started_any = False
        stock = self.stock
        patience = self.strategy.patience
        for _priority, step in proposals:
            queue = self.queue[step]
            if self.started[step] or (queue >= 0 and self.queue_step[queue] >= 0):
                continue
            floor = self.cost_floor[step]
            if stock[0] >= floor[0] and stock[1] >= floor[1] and stock[2] >= floor[2] and stock[3] >= floor[3]:
                if reserved >= 0:
                    hours = self.duration(step)
                    if queue >= 0 and queue == self.queue[reserved] and hours > limit:
                        continue
                    cost = self.cost[step]
                    delayed = self.wait_hours([stock[position] - cost[position] for position in range(4)], reserved)
                    if delayed > limit + patience * hours + 1e-9 * limit + 1e-9:
                        continue
                    if not self.is_next_of_its_chain(step) or self.blockers(step):
                        continue
                    limit = max(limit, delayed)
                elif not self.is_next_of_its_chain(step) or self.blockers(step):
                    continue
                self._start(step)
                started_any = True
            elif reserved < 0 and self.is_next_of_its_chain(step) and not self.blockers(step):
                reserved = step
                limit = self.wait_hours(stock, step)
        self.reserved = reserved
        self.smelting = reserved >= 0 and stock[1] < self.cost_floor[reserved][1]
        return started_any


def path_final_output_table(simulator: EmpirePathSimulator) -> pd.DataFrame:
    """What every planet makes per hour once the configuration stands, next to what the configuration saved.

    The path model must end where the configuration model says: `matches_saved` is True when the Metal,
    Kryptonite, Spice and Pig iron of a planet agree with the saved numbers (which are rounded to 0.1).
    """
    rows: list[dict[str, Any]] = []
    for planet, output in zip(simulator.planets, simulator.final_output):
        saved = planet["saved_production"]
        made = {
            "pig_iron_per_hour": output[1],
            "metal_per_hour": output[2],
            "kryptonite_per_hour": output[3],
            "spice_per_hour": output[4],
        }
        rows.append(
            {
                "slot_index": planet["slot_index"],
                "coords": planet["coords"],
                **made,
                "power_ratio": output[5],
                "smelting_ratio": output[6],
                **{f"saved_{name}": float(saved.get(name, float("nan"))) for name in made},
                "matches_saved": all(
                    abs(value - float(saved.get(name, float("nan")))) <= 0.051 + 1e-9 * abs(value)
                    for name, value in made.items()
                ),
            }
        )
    return pd.DataFrame(rows)


PATH_SIMULATOR = EmpirePathSimulator(
    PATH_PROBLEM,
    ATTACK_INCOME_SUMMARY["fixed_income_per_hour"],
    slot_attack_income=ATTACK_INCOME_SUMMARY["income_per_slot_per_hour"],
    fleet_command_center_level=PATH_FLEET_COMMAND_CENTER_LEVEL,
    spare_percent=PATH_RESOURCE_SPARE_PERCENT,
    reward_weeks=PATH_REWARD_WEEKS,
    weekly_discount=PATH_WEEKLY_DISCOUNT,
)
path_final_output_df = path_final_output_table(PATH_SIMULATOR)
path_lower_bounds_df = (
    pd.DataFrame(
        [
            {"limit": limit, "hours": hours, "time": fmt_path_time(hours)}
            for limit, hours in PATH_SIMULATOR.lower_bounds()
        ]
    )
    .sort_values("hours", ascending=False, kind="stable")
    .reset_index(drop=True)
)
PATH_LOWER_BOUND_HOURS: float = float(path_lower_bounds_df["hours"].iloc[0])
PATH_REWARD_UPPER_BOUND: float = float(PATH_SIMULATOR.reward_upper_bound)

print_section_data(
    f"Section 11.4 data: simulator ({PATH_REWARD_WEEKS}w, discount {PATH_WEEKLY_DISCOUNT:.2f}), "
    f"reward ceiling {PATH_REWARD_UPPER_BOUND:,.1f} SpiceEq, full bound {fmt_path_time(PATH_LOWER_BOUND_HOURS)}",
    [
        ("PRODUCTION_FLOW_KEYS", PRODUCTION_FLOW_KEYS),
        ("PATH_FLOW_BUILDING_KEYS", PATH_FLOW_BUILDING_KEYS),
        ("PATH_MINE_KEYS", PATH_MINE_KEYS),
        ("PATH_ACCELERATOR_KEYS", PATH_ACCELERATOR_KEYS),
        ("PATH_HOURS_PER_WEEK", PATH_HOURS_PER_WEEK),
        ("PATH_NEVER", PATH_NEVER),
        (
            "PATH_SIMULATOR (EmpirePathSimulator summary)",
            {
                "planets": PATH_SIMULATOR.planet_count,
                "steps": len(PATH_SIMULATOR.steps),
                "horizon_steps": sum(PATH_SIMULATOR.in_horizon),
                "chains": len(PATH_SIMULATOR.chain_steps),
                "queues": PATH_SIMULATOR.queue_count,
                "reward_weeks": PATH_SIMULATOR.reward_weeks,
                "weekly_discount": PATH_SIMULATOR.weekly_discount,
                "horizon_hours": PATH_SIMULATOR.horizon_hours,
                "spare_percent": dict(zip(RESOURCE_KEYS, PATH_SIMULATOR.spare_percent)),
                "start_fleet_slots": PATH_SIMULATOR.fleet_slots_of(
                    [p["levels"] for p in PATH_SIMULATOR.planets],
                    [p["founded"] for p in PATH_SIMULATOR.planets],
                ),
                "final_fleet_slots": PATH_SIMULATOR.final_fleet_slots,
                "attack_income_per_hour_now": dict(zip(RESOURCE_KEYS, PATH_SIMULATOR.attack)),
                "slot_attack_income_per_hour": dict(zip(RESOURCE_KEYS, PATH_SIMULATOR.slot_attack)),
                "supply_per_hour_of_the_configuration": {
                    resource: int(round(amount)) for resource, amount in zip(RESOURCE_KEYS, PATH_SIMULATOR.final_supply)
                },
                "output_per_hour_of_the_configuration": {
                    resource: int(round(amount))
                    for resource, amount in zip(RESOURCE_KEYS, PATH_SIMULATOR.final_output_total)
                },
            },
        ),
        ("path_final_output_df", path_final_output_df),
        ("path_lower_bounds_df", path_lower_bounds_df),
        ("PATH_LOWER_BOUND_HOURS", PATH_LOWER_BOUND_HOURS),
        ("PATH_REWARD_UPPER_BOUND", PATH_REWARD_UPPER_BOUND),
    ],
)

### 11.5 Discounted-Payback Strategies

A `PathStrategy` decides which steps the empire wants next across the `PATH_REWARD_WEEKS` window. Three deterministic strategies are simulated here.

**Cheapest step first** wants every step that can start and takes the cheapest in SpiceEq. It is the baseline.

**Discounted payback** ranks a step by the higher of two numbers:

- its **NPV-weighted payback**: the steady-state output value that the step adds per hour multiplied by the fraction of remaining discounted hours left after it finishes (`remaining_discounted_hours(clock + duration) / remaining_discounted_hours(clock)`), divided by its cost plus opportunity cost during build. Any step that cannot finish before the end of week `W` gets economic score `0`. Mines bundle the Fusion plant levels they need, blocked steps propagate their bundle rank down to their unblocked enablers (warehouses, prerequisites, demolitions, colony founding), and accelerators (`b7`, `b17`) join a bundle whenever cutting build duration raises its discounted payback.
- its **urgency**: accelerators (`b7`, `b17`, `b8`) receive speed urgency scaled by `speed_weight` and the remaining share of the `W`-week discounted window whenever planet or research work needs speed; non-producing steps only become urgent if the entire configuration fits inside the `W`-week window (or once economic steps are exhausted).

`"merchant"` values resources at fixed Merchant rates (`6:2.5:3:1`). `"scarcity"` scales near-term horizon scarcity (`horizon_missing_resources` vs current supply) with exponent `focus`, so `900,000`-year tail levels never distort early-week prices.

- `PATH_STRATEGIES` — cheapest step first, discounted payback at Merchant values, and discounted payback at scarcity values.
- `PATH_RUNS` — the `W`-week simulation of each strategy.
- `path_strategies_df` — one row per strategy: settings, status, `discounted_reward`, total `W`-week SpiceEq, week-1 and week-`W` output, and actions started/completed.

In [ ]:
class PathStrategy:
    """Which steps the empire wants next, and in which order.

    `rule="cheapest"` wants every step that can start and takes the cheapest first, at the Merchant's
    exchange rates. It tears a level down as soon as its planet is full. It is the plain baseline.

    `rule="payback"` ranks a step by the higher of two numbers, both in "per hour":

    - its **payback**: the value that the step adds to the hourly supply, divided by the value of its cost.
      A mine counts together with the Fusion levels that it needs to run, and a step that is blocked passes
      its payback on to the steps that unblock it (a warehouse, a prerequisite, a field, a colony), with
      their cost added to its own. A colony is worth the payback of its first levels.
    - its **urgency**, only on a critical queue: the hourly supply divided by the value of its cost, that
      is, one over the hours the empire needs to earn it. A queue is critical when the work left on it is
      at least `critical_share` of the hours that the rest of the path is expected to take. An accelerator
      on such a queue is as many times more urgent as the time it saves exceeds the time it takes.

    A step that neither pays back nor is urgent waits: a research level that only unlocks something, a
    shipyard, the fleet. So a step that makes the empire richer goes before a step that only has to be
    done, unless the queue of that step would hold up the end of the path.

    `values` says what a unit of a resource is worth. `"merchant"` uses the Merchant's exchange rates.
    `"scarcity"` divides by the hourly supply of the resource and multiplies by `share ** focus`, where
    `share` compares the hours that the supply needs for what the remaining steps still cost with the same
    hours of the scarcest resource. `bias` multiplies the four values.
    """

    def __init__(
        self,
        name: str,
        *,
        rule: str,
        values: str = "merchant",
        focus: float = 1.0,
        critical_share: float = 0.9,
        patience: float = 0.25,
        speed_weight: float = 1.5,
        bias: tuple[float, ...] = (1.0, 1.0, 1.0, 1.0),
        revalue_every_steps: int = 25,
    ) -> None:
        if rule not in ("cheapest", "payback") or values not in ("merchant", "scarcity"):
            raise ValueError(f"Unknown rule {rule!r} or values {values!r}")
        self.name = name
        self.rule = rule
        self.value_mode = values
        self.focus = float(focus)
        self.critical_share = float(critical_share)
        self.patience = float(patience)
        self.speed_weight = float(speed_weight)
        self.bias = tuple(float(value) for value in bias)
        self.revalue_every = max(1, int(revalue_every_steps))

    def parameters(self) -> dict[str, Any]:
        """The settings of the strategy, for the tables."""
        if self.rule == "cheapest":
            return {"rule": self.rule}
        return {
            "rule": self.rule,
            "values": self.value_mode,
            "focus": self.focus,
            "critical_share": self.critical_share,
            "patience": self.patience,
            "speed_weight": self.speed_weight,
            "bias": self.bias,
        }

    # ------------------------------------------------------------------ called by the simulator

    def begin(self, sim: EmpirePathSimulator) -> None:
        """Forget the previous run."""
        owners = sim.planet_count + 1
        self.forced = False
        # What the strategy wants is kept twice: as it is, and as it is when every step is wanted (`forced`).
        # Each view has its own list per owner, its own owners to look at again, and its own ranked list.
        self.views: dict[bool, dict[str, Any]] = {
            forced: {
                "entries": [[] for _ in range(owners)],
                "requests": [{} for _ in range(owners)],
                "dirty": set(range(owners)),
                "ranked": None,
            }
            for forced in (False, True)
        }
        self.entries: list[list[tuple[tuple, int]]] = self.views[False]["entries"]
        self.requests: list[dict[int, tuple]] = self.views[False]["requests"]
        self.research_gains: dict[str, tuple[int, tuple[float, ...]]] = {}
        self.critical = [False] * sim.queue_count
        self.needs_speed = [False] * sim.queue_count
        self.merchant_values = tuple(1.0 / float(game_data.MERCHANT_RATES_IN_SPICE[r]) for r in RESOURCE_KEYS)
        self.values = self.merchant_values
        self.income_value = 0.0
        self.horizon = PATH_NEVER
        self.colony_cache: dict[tuple, float] = {}

    def review(self, sim: EmpirePathSimulator) -> tuple[float, int, bool]:
        """Re-estimate the rest of the path: its length, the critical queues and the values of the resources.

        The rest of the path takes at least as long as the work left on the busiest queue, and at least as
        long as the finished configuration would need to supply what the steps not started yet still cost.
        The longer of the two is the horizon.

        Returns the horizon, the number of finished steps after which to review again, and whether the
        picture changed enough to rank every step again: a queue became critical or stopped being so, or
        a value or the supply moved by more than 15 %. Smaller changes keep the ranks as they are.
        """
        if self.rule == "cheapest":
            return PATH_NEVER, len(sim.steps) + 1, False
        work, slow_work = sim.queue_work()
        bounded = sim.run_limit_hours < PATH_NEVER
        missing = sim.horizon_missing_resources() if bounded else sim.missing_resources()
        full_missing = sim.missing_resources()
        full_horizon = max(work)
        for position in range(4):
            if full_missing[position] > 0.0:
                supply = sim.final_supply[position]
                full_horizon = max(full_horizon, full_missing[position] / supply if supply > 0.0 else PATH_NEVER)
        rem_hours = max(0.0, sim.run_limit_hours - sim.clock) if bounded else full_horizon
        horizon = rem_hours if (bounded and rem_hours > 0.0) else full_horizon
        self.horizon = horizon
        measurable = 0.0 < horizon < PATH_NEVER
        can_finish_all = (not bounded) or (full_horizon <= sim.run_limit_hours)
        critical = [
            can_finish_all and measurable and hours > 0.0 and hours >= self.critical_share * horizon for hours in work
        ]
        # A queue needs speed when, without another accelerator level, its work exceeds critical_share of the window.
        needs_speed = [
            measurable and hours > 0.0 and hours >= self.critical_share * horizon for hours in slow_work
        ]

        # Hours that today's supply needs for what is still missing of each resource.
        supply_hours = [
            (missing[position] / sim.supply[position] if sim.supply[position] > 0.0 else PATH_NEVER)
            if missing[position] > 0.0
            else 0.0
            for position in range(4)
        ]
        starved = any(hours == PATH_NEVER for hours in supply_hours)
        if self.value_mode == "scarcity":
            longest = max((hours for hours in supply_hours if hours < PATH_NEVER), default=0.0)
            values = []
            for position in range(4):
                if supply_hours[position] == PATH_NEVER:
                    share = 1.0
                elif starved or longest <= 0.0:
                    share = 0.01 if starved else 1.0
                else:
                    share = max(0.01, supply_hours[position] / longest)
                values.append(self.bias[position] * share**self.focus / max(sim.supply[position], 1e-9))
        else:
            values = [self.bias[position] * self.merchant_values[position] for position in range(4)]
        income_value = sum(values[position] * sim.supply[position] for position in range(4))

        def moved(new: float, old: float) -> bool:
            return abs(new - old) > 0.15 * abs(old)

        changed = (
            self.income_value <= 0.0
            or critical != self.critical
            or needs_speed != self.needs_speed
            or moved(income_value, self.income_value)
            or any(moved(new, old) for new, old in zip(values, self.values))
        )
        if changed:
            self.critical, self.needs_speed = critical, needs_speed
            self.values, self.income_value = tuple(values), income_value
            self.colony_cache.clear()
        # While nothing supplies a resource that is needed, every finished step can change the picture.
        return horizon, (1 if starved else self.revalue_every), changed

    def proposals(self, sim: EmpirePathSimulator, *, forced: bool) -> list[tuple[tuple, int]]:
        """The steps that the strategy wants, best first. The simulator starts those that the rules allow.

        `forced` asks for every step that can start, wanted or not: the simulator uses it when nothing runs
        and nothing is wanted, so that the path goes on.
        """
        if sim.dirty:
            for view in self.views.values():
                view["dirty"].update(sim.dirty)
            sim.dirty.clear()
        view = self.views[forced]
        if not view["dirty"] and view["ranked"] is not None:
            return view["ranked"]
        self.forced = forced
        self.entries, self.requests = view["entries"], view["requests"]
        for owner in sorted(view["dirty"]):
            if owner == sim.research_queue:
                self._scan_research(sim)
            else:
                self._scan_planet(sim, owner)
        view["dirty"].clear()
        wanted: dict[int, tuple] = {}
        for entries in self.entries:
            for priority, step in entries:
                if step not in wanted or priority > wanted[step]:
                    wanted[step] = priority
        for requests in self.requests:
            for step, priority in requests.items():
                if step not in wanted or priority > wanted[step]:
                    wanted[step] = priority
        view["ranked"] = sorted(((priority, step) for step, priority in wanted.items()), reverse=True)
        return view["ranked"]

    # ------------------------------------------------------------------ values and ranks

    def _value(self, amounts: Any) -> float:
        values = self.values
        return values[0] * amounts[0] + values[1] * amounts[1] + values[2] * amounts[2] + values[3] * amounts[3]

    def _gain(self, sim: EmpirePathSimulator, planet: int, changes: dict[str, int]) -> tuple[float, ...]:
        """Net kept steady-state hourly output (`1 - spare%`) added on a founded planet by `changes`."""
        after = planet_hourly_output(sim.changed_totals(planet, changes))
        before = sim.output[planet]
        keep = sim.keep
        dl_b10 = 0
        if "b10" in changes and sim.fleet_command_center_level is None:
            dl_b10 = max(0, int(changes["b10"]) - sim.levels[planet].get("b10", 0))
        return tuple(
            keep[pos] * (after[pos + 1] - before[pos + 1] + dl_b10 * sim.slot_attack[pos]) for pos in range(4)
        )

    def _rank(
        self, sim: EmpirePathSimulator, added: float, price: float, urgent: float, hours: float
    ) -> float:
        """Discounted payback vs urgency over the remaining W-week window (`0` if an upgrade finishes past week W)."""
        if price <= 0.0:
            return PATH_NEVER if (added > 0.0 or urgent > 0.0) else 0.0
        if sim.run_limit_hours < PATH_NEVER:
            now_window = sim.remaining_discounted_hours(sim.clock)
            future_window = sim.remaining_discounted_hours(sim.clock + hours)
            if now_window > 0.0 and future_window <= 0.0:
                return 0.0
            window_factor = (future_window / now_window) if now_window > 0.0 else 1.0
            total_window = sim.suffix_discounted_hours[0]
            rem_share = (future_window / total_window) if total_window > 0.0 else 1.0
        else:
            window_factor = rem_share = 1.0
        rank = (added * window_factor) / (price + added * hours) if added > 0.0 else 0.0
        if urgent > 0.0:
            rank = max(rank, urgent * rem_share * self.income_value / price)
        return rank

    def _offer(
        self,
        sim: EmpirePathSimulator,
        step: int,
        gain: Any,
        urgent: float,
        entries: list[tuple[tuple, int]],
        requests: dict[int, tuple],
        *,
        extra_enablers: list[int] | None = None,
        bundle_planet: int = -1,
        accelerators: Any = (),
    ) -> None:
        """Offer a step that can start, or ask for the steps that it still needs.

        A step that is blocked passes its rank on to the steps that unblock it. The rank is then that of the
        bundle: the step together with everything it still needs. `accelerators` lists accelerator levels
        of the planet, each with the steps it needs itself: one of them joins the bundle when the step
        then pays back sooner, the shorter building time weighed against the cost of the accelerator.
        """
        blockers = sim.blockers(step)
        # Every step that must come first, with what blocks that step in turn.
        enablers: dict[int, list[tuple]] = {}
        if blockers:
            self._collect_enablers(sim, blockers, enablers)
        for extra in extra_enablers or ():
            if extra not in enablers:
                enablers[extra] = sim.blockers(extra)
                if enablers[extra]:
                    self._collect_enablers(sim, enablers[extra], enablers)
        if blockers and not enablers:
            return  # what blocks the step is under way: only waiting helps
        cost = list(sim.cost[step])
        for enabler in enablers:
            enabler_cost = sim.cost[enabler]
            for position in range(4):
                cost[position] += enabler_cost[position]
        if enablers and bundle_planet >= 0:
            changes = {sim.key[step]: sim.level_to[step]}
            for enabler in enablers:
                if sim.kind[enabler] == "build" and sim.step_planet[enabler] == bundle_planet:
                    changes[sim.key[enabler]] = max(changes.get(sim.key[enabler], 0), sim.level_to[enabler])
            gain = self._gain(sim, bundle_planet, changes)
        added = self._value(gain)
        own_hours = other_hours = 0.0
        if added > 0.0 or urgent > 0.0:
            own_hours = sim.duration(step)
            queue = sim.queue[step]
            other_hours = sum(sim.duration(enabler) for enabler in enablers if sim.queue[enabler] == queue)
        price = self._value(cost)
        rank = self._rank(sim, added, price, urgent, own_hours + other_hours)
        if own_hours > 0.0 and added > 0.0:
            for accelerator, needs, faster, needs_price, needs_hours in accelerators:
                if accelerator in enablers:
                    continue
                faster_rank = self._rank(
                    sim, added, price + needs_price, urgent, own_hours / faster + other_hours + needs_hours
                )
                if faster_rank > rank:
                    rank = faster_rank
                    price += needs_price
                    other_hours += needs_hours
                    own_hours /= faster
                    enablers = {**enablers, **needs}
        if rank <= 0.0:
            return
        if not enablers:
            entries.append(((rank, -price, -step), step))
            return
        for enabler, enabler_blockers in enablers.items():
            if not enabler_blockers and sim.is_next_of_its_chain(enabler):
                priority = (rank, -price, -enabler)
                if enabler not in requests or priority > requests[enabler]:
                    requests[enabler] = priority

    def _collect_enablers(
        self, sim: EmpirePathSimulator, blockers: list[tuple], enablers: dict[int, list[tuple]]
    ) -> None:
        """Add to `enablers` the steps not started yet that remove `blockers`, each with what blocks it in turn,
        and what those steps need themselves.

        A blocker that a running step already removes adds nothing: only waiting helps there.
        """
        added: list[int] = []
        for blocker in blockers:
            reason = blocker[0]
            candidates: list[int] = []
            if reason == "colony":
                step = sim.colonize_step[blocker[1]]
                if step >= 0 and not sim.started[step]:
                    candidates.append(step)
            elif reason == "field":
                planet = blocker[1]
                running = sim.queue_step[planet]
                if not (running >= 0 and sim.kind[running] == "demolish"):
                    step = self._demolition(sim, planet)
                    if step >= 0:
                        candidates.append(step)
            else:
                if reason == "level":
                    _, owner, key, level = blocker
                    chain = sim.research_chains.get(key) if owner < 0 else sim.build_chains[owner].get(key)
                else:
                    _, owner, position, level = blocker
                    chain = sim.build_chains[owner].get(WAREHOUSE_KEYS[position])
                if chain is not None:
                    for step in sim.chain_steps[chain][sim.next_step[chain]:]:
                        if sim.level_to[step] > level:
                            break
                        candidates.append(step)
            for step in candidates:
                if step not in enablers:
                    enablers[step] = sim.blockers(step)
                    added.append(step)
        for step in added:
            if enablers[step] and len(enablers) < 400:
                self._collect_enablers(sim, enablers[step], enablers)

    def _energy_steps(self, sim: EmpirePathSimulator, planet: int, totals: list[float]) -> list[int]:
        """Fusion plant levels to build before a mine level, so that the planet's energy still covers its
        mines afterwards. `totals` are the planet's flows with that mine level. Empty when the energy
        suffices or no Fusion level is left."""
        energy_produced, energy_used = totals[6], totals[7]
        if energy_used <= energy_produced + 1e-9:
            return []
        chain = sim.build_chains[planet].get("b5")
        if chain is None or sim.next_step[chain] >= len(sim.chain_steps[chain]):
            return []
        temperature = sim.temperature[planet]
        mining, energy = sim.research["r1"], sim.research["r2"]
        standing = sim.levels[planet].get("b5", 0)
        standing_energy = sim.level_flows("b5", standing, temperature, mining, energy)[6] if standing > 0 else 0.0
        needed: list[int] = []
        for step in sim.chain_steps[chain][sim.next_step[chain]:]:
            needed.append(step)
            fusion_energy = sim.level_flows("b5", sim.level_to[step], temperature, mining, energy)[6]
            if energy_used <= energy_produced - standing_energy + fusion_energy + 1e-9:
                break
        return needed

    def _demolition(self, sim: EmpirePathSimulator, planet: int) -> int:
        """The level to tear down next on a planet: the one whose loss costs least. -1 when there is none."""
        best_step, best_loss = -1, PATH_NEVER
        for chain in sim.demolish_chains[planet]:
            position = sim.next_step[chain]
            if position >= len(sim.chain_steps[chain]):
                continue
            step = sim.chain_steps[chain][position]
            loss = 0.0
            if sim.founded[planet]:
                loss = -self._value(self._gain(sim, planet, {sim.key[step]: sim.level_to[step]}))
            if loss < best_loss:
                best_step, best_loss = step, loss
        return best_step

    def _research_gain(self, sim: EmpirePathSimulator, key: str) -> tuple[float, ...]:
        """What the next level of Mining or Energy technology adds to the hourly supply of the founded planets.

        The result is kept until the supply of the empire or a research level changes.
        """
        token = sim.supply_version
        cached = self.research_gains.get(key)
        if cached is not None and cached[0] == token:
            return cached[1]
        mining = sim.research["r1"] + (1 if key == "r1" else 0)
        energy = sim.research["r2"] + (1 if key == "r2" else 0)
        outputs = [
            planet_hourly_output(sim.planet_totals(planet, sim.levels[planet], sim.solar[planet], mining, energy))
            for planet in range(sim.planet_count)
        ]
        gain = tuple(
            sim.keep[pos]
            * sum(outputs[p][pos + 1] - sim.output[p][pos + 1] for p in range(sim.planet_count) if sim.founded[p])
            for pos in range(4)
        )
        self.research_gains[key] = (token, gain)
        return gain

    def _colony_payback(self, sim: EmpirePathSimulator, planet: int, depth: int = 25) -> float:
        """Payback of founding a colony: the best ratio, over its first `depth` levels, of the value that
        those levels add per hour to the value of what they cost together with the Colony ship.

        The levels are picked one at a time, always the mine that pays back soonest, each with the Fusion
        levels that it needs. Colonies with the same plan and temperature share the result until the next
        review.
        """
        target = sim.planets[planet]["target_levels"]
        token = (sim.temperature[planet], tuple(sorted(target.items())), tuple(sorted(sim.levels[planet].items())))
        cached = self.colony_cache.get(token)
        if cached is not None:
            return cached
        mining, energy = sim.research["r1"], sim.research["r2"]
        solar = sim.solar[planet]
        values = self.values

        keep = sim.keep
        fixed_cc = sim.fleet_command_center_level

        def supply_value(levels: dict[str, int]) -> float:
            output = planet_hourly_output(sim.planet_totals(planet, levels, solar, mining, energy))
            slots = 1 + (levels.get("b10", 0) if fixed_cc is None else fixed_cc)
            return sum(
                values[pos] * keep[pos] * (output[pos + 1] + slots * sim.slot_attack[pos]) for pos in range(4)
            )

        def level_price(key: str, level: int) -> float:
            cost = full_cost(game_data.building_cost(int(key[1:]), level))
            return sum(values[position] * float(cost[resource]) for position, resource in enumerate(RESOURCE_KEYS))

        levels = dict(sim.levels[planet])
        ship = sim.colonize_step[planet]
        spent = self._value(sim.cost[ship]) if ship >= 0 else 0.0
        # Before founding, the slot contributes 0; founding immediately activates its base fleet slot(s).
        current = supply_value(levels)
        added = current
        best = (added / spent) if spent > 0.0 else 0.0
        for _ in range(depth):
            choice: tuple[float, dict[str, int], float, float] | None = None
            for key in PATH_MINE_KEYS:
                level = levels.get(key, 0) + 1
                if level > target.get(key, 0):
                    continue
                trial = dict(levels)
                trial[key] = level
                price = level_price(key, level)
                # Add Fusion levels until the energy covers the mines again, as far as the plan has them.
                while True:
                    totals = sim.planet_totals(planet, trial, solar, mining, energy)
                    fusion = trial.get("b5", 0) + 1
                    if totals[7] <= totals[6] + 1e-9 or fusion > target.get("b5", 0):
                        break
                    trial["b5"] = fusion
                    price += level_price("b5", fusion)
                value = supply_value(trial) - current
                ratio = value / price if price > 0.0 else 0.0
                if value > 0.0 and (choice is None or ratio > choice[0]):
                    choice = (ratio, trial, value, price)
            if choice is None:
                break
            _, levels, value, price = choice
            current += value
            added += value
            spent += price
            if spent > 0.0:
                best = max(best, added / spent)
        self.colony_cache[token] = best
        return best

    # ------------------------------------------------------------------ the steps of a planet and of the research

    def _scan_planet(self, sim: EmpirePathSimulator, planet: int) -> None:
        """Work out which steps of a planet can start and how much the strategy wants each of them."""
        entries: list[tuple[tuple, int]] = []
        requests: dict[int, tuple] = {}
        self.entries[planet], self.requests[planet] = entries, requests
        cheapest = self.rule == "cheapest"
        forced = self.forced
        nothing = (0.0, 0.0, 0.0, 0.0)

        if not sim.founded[planet]:
            step = sim.colonize_step[planet]
            if step < 0 or sim.started[step]:
                return
            blockers = sim.blockers(step)
            if cheapest:
                if not blockers:
                    entries.append(((-sim.cost_spice_eq[step], 0.0, -step), step))
                return
            # The payback of a colony is that of its first levels, not of the Colony ship alone.
            price = self._value(sim.cost[step])
            colony_hours = sim.duration(step)
            if sim.run_limit_hours < PATH_NEVER and sim.clock + colony_hours >= sim.run_limit_hours and not forced:
                return
            forced_rank = self._rank(sim, 0.0, price, 1.0 if forced else 0.0, colony_hours)
            rank = max(self._colony_payback(sim, planet), forced_rank)
            if rank <= 0.0:
                return
            if not blockers:
                entries.append(((rank, -price, -step), step))
            else:
                enablers: dict[int, list[tuple]] = {}
                self._collect_enablers(sim, blockers, enablers)
                for enabler, enabler_blockers in enablers.items():
                    if not enabler_blockers and sim.is_next_of_its_chain(enabler):
                        requests[enabler] = (rank, -price, -enabler)
            return

        yard = sim.planet_count + 1 + planet
        if cheapest:
            for chain in (*sim.build_chains[planet].values(), *sim.ship_chains[planet]):
                position = sim.next_step[chain]
                if position < len(sim.chain_steps[chain]):
                    step = sim.chain_steps[chain][position]
                    if not sim.blockers(step):
                        entries.append(((-sim.cost_spice_eq[step], 0.0, -step), step))
            # No look-ahead: a level is torn down as soon as the planet is full.
            if sim.fields_used[planet] >= sim.fields_max[planet] or forced:
                step = self._demolition(sim, planet)
                if step >= 0 and not sim.blockers(step):
                    entries.append(((0.0, 0.0, -step), step))
            return

        urgent = 1.0 if (self.critical[planet] or forced) else 0.0
        needs_speed = self.needs_speed[planet]
        research_needs_speed = planet == sim.lab_planet and self.needs_speed[sim.research_queue]
        before = sim.output[planet]
        # The next accelerator levels of the planet, each with what it needs, how much faster it makes the
        # planet build, and what it and its needs cost and take.
        accelerators = []
        for key in PATH_ACCELERATOR_KEYS:
            chain = sim.build_chains[planet].get(key)
            if chain is None or sim.next_step[chain] >= len(sim.chain_steps[chain]):
                continue
            step = sim.chain_steps[chain][sim.next_step[chain]]
            needs = {step: sim.blockers(step)}
            if needs[step]:
                self._collect_enablers(sim, needs[step], needs)
            level = sim.level_to[step]
            accelerators.append(
                (
                    step,
                    needs,
                    2.0 if key == "b17" else (level + 1.0) / level,
                    sum(self._value(sim.cost[need]) for need in needs),
                    sum(sim.duration(need) for need in needs if sim.queue[need] == planet),
                )
            )
        builds_left = 0
        for key, chain in sim.build_chains[planet].items():
            position = sim.next_step[chain]
            if position >= len(sim.chain_steps[chain]):
                continue
            builds_left += 1
            step = sim.chain_steps[chain][position]
            level = sim.level_to[step]
            if key in PATH_FLOW_BUILDING_KEYS:
                totals = sim.changed_totals(planet, {key: level})
                after = planet_hourly_output(totals)
                gain = tuple(sim.keep[pos] * (after[pos + 1] - before[pos + 1]) for pos in range(4))
                extra = self._energy_steps(sim, planet, totals) if key in PATH_MINE_KEYS else None
                self._offer(
                    sim, step, gain, urgent, entries, requests,
                    extra_enablers=extra, bundle_planet=planet, accelerators=accelerators,
                )
            elif key == "b10" and sim.fleet_command_center_level is None and any(x > 0.0 for x in sim.slot_attack):
                slot_gain = tuple(sim.keep[pos] * sim.slot_attack[pos] for pos in range(4))
                self._offer(
                    sim, step, slot_gain, urgent, entries, requests,
                    bundle_planet=planet, accelerators=accelerators,
                )
            elif key in PATH_ACCELERATOR_KEYS:
                saves = self.speed_weight * (0.5 if key == "b17" else 1.0 / (level + 1.0))
                speed_urgent = max(urgent, saves if needs_speed else 0.0)
                if speed_urgent > 0.0:
                    self._offer(sim, step, nothing, speed_urgent, entries, requests)
            elif key == "b8" and research_needs_speed:
                self._offer(sim, step, nothing, max(urgent, self.speed_weight / (level + 1.0)), entries, requests)
            elif urgent > 0.0:
                self._offer(sim, step, nothing, urgent, entries, requests)
        yard_urgent = 1.0 if (self.critical[yard] or forced) else 0.0
        if yard_urgent > 0.0:
            for chain in sim.ship_chains[planet]:
                position = sim.next_step[chain]
                if position < len(sim.chain_steps[chain]):
                    self._offer(sim, sim.chain_steps[chain][position], nothing, yard_urgent, entries, requests)
        # A level is torn down when a step needs its field (`_collect_enablers`), or as the last work of the planet.
        if urgent > 0.0 and builds_left == 0:
            step = self._demolition(sim, planet)
            if step >= 0 and not sim.blockers(step):
                entries.append(((PATH_NEVER, 0.0, -step), step))

    def _scan_research(self, sim: EmpirePathSimulator) -> None:
        """Work out which research levels can start and how much the strategy wants each of them."""
        owner = sim.research_queue
        entries: list[tuple[tuple, int]] = []
        requests: dict[int, tuple] = {}
        self.entries[owner], self.requests[owner] = entries, requests
        urgent = 1.0 if (self.critical[owner] or self.forced) else 0.0
        for key, chain in sim.research_chains.items():
            position = sim.next_step[chain]
            if position >= len(sim.chain_steps[chain]):
                continue
            step = sim.chain_steps[chain][position]
            if self.rule == "cheapest":
                if not sim.blockers(step):
                    entries.append(((-sim.cost_spice_eq[step], 0.0, -step), step))
                continue
            if key in ("r1", "r2"):
                self._offer(sim, step, self._research_gain(sim, key), urgent, entries, requests)
            elif urgent > 0.0:
                self._offer(sim, step, (0.0, 0.0, 0.0, 0.0), urgent, entries, requests)


def strategy_runs_table(
    strategies: list[PathStrategy], runs: dict[str, dict[str, Any]], reward_upper_bound: float
) -> pd.DataFrame:
    """One row per strategy: its settings, W-week discounted reward, weekly endpoints and actions taken."""
    rows: list[dict[str, Any]] = []
    for strategy in strategies:
        run = runs[strategy.name]
        settings = strategy.parameters()
        hours = float(run["hours"])
        reward = float(run["discounted_reward"])
        weekly = run["weekly_spice_eq"]
        rows.append(
            {
                "strategy": strategy.name,
                "rule": settings["rule"],
                "values": settings.get("values", "merchant"),
                "focus": settings.get("focus", float("nan")),
                "critical_share": settings.get("critical_share", float("nan")),
                "patience": settings.get("patience", float("nan")),
                "speed_weight": settings.get("speed_weight", float("nan")),
                "status": run["status"],
                "discounted_reward": reward,
                "total_spice_eq_produced": float(run["total_spice_eq_produced"]),
                "week_1_spice_eq": float(weekly[0]),
                "last_week_spice_eq": float(weekly[-1]),
                "share_of_reward_ceiling_percent": (
                    100.0 * reward / reward_upper_bound if reward_upper_bound > 0.0 else 0.0
                ),
                "steps_started": int(run["steps_started"]),
                "steps_completed": int(run["steps_completed"]),
                "hours": hours,
                "time": fmt_path_time(hours),
                "metal_smelted": float(run["metal_smelted"]),
            }
        )
    return pd.DataFrame(rows)


PATH_STRATEGIES: list[PathStrategy] = [
    PathStrategy("cheapest step first", rule="cheapest", patience=PATH_PATIENCE),
    PathStrategy(
        "discounted payback at Merchant values",
        rule="payback",
        values="merchant",
        critical_share=PATH_CRITICAL_QUEUE_SHARE,
        patience=PATH_PATIENCE,
        speed_weight=PATH_SPEED_WEIGHT,
        revalue_every_steps=PATH_REVALUE_EVERY_STEPS,
    ),
    PathStrategy(
        "discounted payback at scarcity values",
        rule="payback",
        values="scarcity",
        focus=PATH_SCARCITY_FOCUS,
        critical_share=PATH_CRITICAL_QUEUE_SHARE,
        patience=PATH_PATIENCE,
        speed_weight=PATH_SPEED_WEIGHT,
        revalue_every_steps=PATH_REVALUE_EVERY_STEPS,
    ),
]
PATH_RUNS: dict[str, dict[str, Any]] = {strategy.name: PATH_SIMULATOR.run(strategy) for strategy in PATH_STRATEGIES}
path_strategies_df = strategy_runs_table(PATH_STRATEGIES, PATH_RUNS, PATH_REWARD_UPPER_BOUND)

print_section_data(
    "Section 11.5 data: the three strategies and their W-week discounted rewards",
    [
        ("PATH_STRATEGIES (settings)", {strategy.name: strategy.parameters() for strategy in PATH_STRATEGIES}),
        (
            "PATH_RUNS (summary)",
            {
                name: {
                    "status": run["status"],
                    "discounted_reward": round(run["discounted_reward"], 1),
                    "total_spice_eq_produced": round(run["total_spice_eq_produced"], 1),
                    "steps_started": run["steps_started"],
                    "steps_completed": run["steps_completed"],
                    "events": run["events"],
                    "reviews": run["reviews"],
                }
                for name, run in PATH_RUNS.items()
            },
        ),
        ("path_strategies_df", path_strategies_df),
    ],
)

### 11.6 Genetic Algorithm (Maximizing Discounted Weekly Production)

`tune_path_strategy` searches for the 8 continuous parameters of `PATH_GENE_RANGES` that **maximize** the `W`-week `discounted_reward` (`sum` over `w = 1..W` of `gamma ** (w - 1) * R_w`, breaking ties by shorter completion time).

- The genes are `focus`, `critical_share`, `patience`, `speed_weight`, and one base-2 log value multiplier per resource (`value_pig_iron`, `value_metal`, `value_kryptonite`, `value_spice`).
- Generation 0 includes `start_genes` (the scarcity settings of Section 11.1) plus random chromosomes, guaranteeing the GA never scores below its starting scarcity rule. Each subsequent generation keeps the `elite_count` highest-reward chromosomes and fills the population via tournament selection, uniform blend crossover and Gaussian mutation.
- The same seed gives the exact same result.

`search_fastest_path` picks the highest-reward strategy. It skips the GA when no strategy can start any step (cold start), when the GA is switched off (`generations <= 0` or `population_size < 2`), or when the best named strategy is already within `PATH_SEARCH_TOLERANCE` of `PATH_REWARD_UPPER_BOUND`.

- `PATH_GENE_RANGES` — bounds of the 8 genes.
- `PATH_SEARCH_SUMMARY` — whether the GA ran and why, simulation count, tuned genes/reward, and winning strategy.
- `path_ga_history_df` — per-generation `best_reward`, `mean_reward`, `best_so_far_reward` and simulation count.
- `BEST_PATH_STRATEGY`, `BEST_PATH_RUN` — the winning strategy and its `W`-week simulation.

In [ ]:
# The numbers of the discounted payback rule that the genetic algorithm tunes, with the range of each.
# The four `value_*` genes are the base-2 logarithms of the value multipliers (`bias`) of the resources.
PATH_GENE_RANGES: dict[str, tuple[float, float]] = {
    "focus": (0.0, 3.0),
    "critical_share": (0.2, 1.0),
    "patience": (0.0, 1.0),
    "speed_weight": (0.0, 3.0),
    "value_pig_iron": (-2.0, 2.0),
    "value_metal": (-2.0, 2.0),
    "value_kryptonite": (-2.0, 2.0),
    "value_spice": (-2.0, 2.0),
}


def strategy_from_genes(name: str, genes: Any, revalue_every_steps: int) -> PathStrategy:
    """The discounted payback strategy at scarcity values described by an 8-gene chromosome."""
    focus, critical_share, patience, speed_weight, *value_genes = (float(gene) for gene in genes)
    return PathStrategy(
        name,
        rule="payback",
        values="scarcity",
        focus=focus,
        critical_share=critical_share,
        patience=patience,
        speed_weight=speed_weight,
        bias=tuple(2.0**gene for gene in value_genes),
        revalue_every_steps=revalue_every_steps,
    )


def path_fitness_key(run: dict[str, Any]) -> tuple[float, float]:
    """Fitness ordering of a run: higher `discounted_reward` wins, breaking ties by earlier completion."""
    return (float(run["discounted_reward"]), -float(run["hours"]))


def tune_path_strategy(
    simulator: EmpirePathSimulator,
    start_genes: Any,
    *,
    population_size: int,
    generations: int,
    tournament_size: int,
    crossover_rate: float,
    mutation_rate: float,
    elite_count: int,
    random_seed: int,
    revalue_every_steps: int,
) -> tuple[np.ndarray, dict[str, Any], pd.DataFrame]:
    """Tune the discounted payback rule with a real-coded GA maximizing W-week `discounted_reward`."""
    rng = np.random.default_rng(int(random_seed))
    low = np.array([bounds[0] for bounds in PATH_GENE_RANGES.values()], dtype=np.float64)
    high = np.array([bounds[1] for bounds in PATH_GENE_RANGES.values()], dtype=np.float64)
    size = max(2, int(population_size))
    elites = min(max(1, int(elite_count)), size - 1)
    results: dict[tuple[float, ...], dict[str, Any]] = {}

    def simulate(genes: np.ndarray) -> dict[str, Any]:
        token = tuple(round(float(gene), 9) for gene in genes)
        if token not in results:
            results[token] = simulator.run(strategy_from_genes("tuned", genes, revalue_every_steps))
        return results[token]

    def pick_parent(population: np.ndarray, keys: list[tuple[float, float]]) -> np.ndarray:
        contenders = rng.integers(0, size, size=max(1, int(tournament_size)))
        winner = max(contenders, key=lambda idx: keys[int(idx)])
        return population[int(winner)]

    first = np.clip(np.asarray(start_genes, dtype=np.float64), low, high)
    population = np.vstack([first, rng.uniform(low, high, (size - 1, len(low)))])
    history: list[dict[str, Any]] = []
    best_genes, best_result = population[0].copy(), simulate(population[0])
    for generation in range(int(generations) + 1):
        runs = [simulate(genes) for genes in population]
        keys = [path_fitness_key(run) for run in runs]
        order = sorted(range(size), key=lambda idx: keys[idx], reverse=True)
        if keys[order[0]] > path_fitness_key(best_result):
            best_genes, best_result = population[order[0]].copy(), runs[order[0]]
        rewards = np.array([float(run["discounted_reward"]) for run in runs], dtype=np.float64)
        stuck = sum(1 for run in runs if run["status"] == "stuck")
        history.append(
            {
                "generation": generation,
                "best_reward": float(rewards[order[0]]),
                "mean_reward": float(rewards.mean()),
                "paths_that_got_stuck": int(stuck),
                "best_so_far_reward": float(best_result["discounted_reward"]),
                "simulations_so_far": len(results),
            }
        )
        if generation == int(generations):
            break
        children = [population[index].copy() for index in order[:elites]]
        while len(children) < size:
            mother, father = pick_parent(population, keys), pick_parent(population, keys)
            child = mother.copy()
            if rng.random() < crossover_rate:
                share = rng.random(len(low))
                child = share * mother + (1.0 - share) * father
            moves = rng.random(len(low)) < mutation_rate
            child = np.clip(child + moves * rng.normal(0.0, 0.15, len(low)) * (high - low), low, high)
            children.append(child)
        population = np.vstack(children)
    return best_genes, best_result, pd.DataFrame(history)


def search_fastest_path(
    simulator: EmpirePathSimulator,
    strategies: list[PathStrategy],
    runs: dict[str, dict[str, Any]],
    reward_upper_bound: float,
    *,
    tolerance: float,
    start_genes: Any,
    population_size: int,
    generations: int,
    tournament_size: int,
    crossover_rate: float,
    mutation_rate: float,
    elite_count: int,
    random_seed: int,
    revalue_every_steps: int,
) -> tuple[PathStrategy, dict[str, Any], pd.DataFrame, dict[str, Any]]:
    """Pick the strategy with the highest W-week `discounted_reward`, tuning with the GA when room remains."""
    active = [strategy for strategy in strategies if runs[strategy.name]["steps_started"] > 0]
    history = pd.DataFrame(
        columns=[
            "generation",
            "best_reward",
            "mean_reward",
            "paths_that_got_stuck",
            "best_so_far_reward",
            "simulations_so_far",
        ]
    )
    summary: dict[str, Any] = {
        "reward_upper_bound": reward_upper_bound,
        "genetic_algorithm_ran": False,
        "simulations_of_the_genetic_algorithm": 0,
        "tuned_genes": None,
        "tuned_reward": None,
    }
    if not active and len(simulator.steps) > 0:
        summary["why"] = "no strategy can start any step"
        summary["best_strategy"] = strategies[-1].name
        summary["best_reward"] = float(runs[strategies[-1].name]["discounted_reward"])
        return strategies[-1], runs[strategies[-1].name], history, summary
    candidates = active if active else list(strategies)
    best = max(candidates, key=lambda strategy: path_fitness_key(runs[strategy.name]))
    best_run = runs[best.name]
    best_reward = float(best_run["discounted_reward"])
    headroom = (
        max(0.0, 1.0 - best_reward / reward_upper_bound) if reward_upper_bound > 0.0 else 0.0
    )
    if generations <= 0 or population_size < 2:
        summary["why"] = "the genetic algorithm is switched off"
    elif headroom <= tolerance:
        summary["why"] = (
            f"{best.name} is within {100.0 * headroom:.3f} % of the reward ceiling "
            f"(tolerance {100.0 * tolerance:.3f} %)"
        )
    else:
        genes, tuned_run, history = tune_path_strategy(
            simulator,
            start_genes,
            population_size=population_size,
            generations=generations,
            tournament_size=tournament_size,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            elite_count=elite_count,
            random_seed=random_seed,
            revalue_every_steps=revalue_every_steps,
        )
        summary["genetic_algorithm_ran"] = True
        summary["simulations_of_the_genetic_algorithm"] = int(history["simulations_so_far"].iloc[-1])
        summary["tuned_genes"] = {name: float(gene) for name, gene in zip(PATH_GENE_RANGES, genes)}
        summary["tuned_reward"] = float(tuned_run["discounted_reward"])
        summary["why"] = f"{best.name} left {100.0 * headroom:.2f} % headroom below the reward ceiling"
        if path_fitness_key(tuned_run) > path_fitness_key(best_run):
            best = strategy_from_genes("discounted payback tuned by the genetic algorithm", genes, revalue_every_steps)
            best_run = tuned_run
    summary["best_strategy"] = best.name
    summary["best_reward"] = float(best_run["discounted_reward"])
    return best, best_run, history, summary


BEST_PATH_STRATEGY, BEST_PATH_RUN, path_ga_history_df, PATH_SEARCH_SUMMARY = search_fastest_path(
    PATH_SIMULATOR,
    PATH_STRATEGIES,
    PATH_RUNS,
    PATH_REWARD_UPPER_BOUND,
    tolerance=PATH_SEARCH_TOLERANCE,
    start_genes=(
        PATH_SCARCITY_FOCUS,
        PATH_CRITICAL_QUEUE_SHARE,
        PATH_PATIENCE,
        PATH_SPEED_WEIGHT,
        0.0,
        0.0,
        0.0,
        0.0,
    ),
    population_size=PATH_GA_POPULATION_SIZE,
    generations=PATH_GA_GENERATIONS,
    tournament_size=PATH_GA_TOURNAMENT_SIZE,
    crossover_rate=PATH_GA_CROSSOVER_RATE,
    mutation_rate=PATH_GA_MUTATION_RATE,
    elite_count=PATH_GA_ELITE_COUNT,
    random_seed=PATH_GA_RANDOM_SEED,
    revalue_every_steps=PATH_REVALUE_EVERY_STEPS,
)

print_section_data(
    f"Section 11.6 data: the search, best strategy {BEST_PATH_STRATEGY.name!r} "
    f"({BEST_PATH_RUN['discounted_reward']:,.1f} discounted SpiceEq)",
    [
        ("PATH_GENE_RANGES", PATH_GENE_RANGES),
        ("PATH_SEARCH_SUMMARY", PATH_SEARCH_SUMMARY),
        ("path_ga_history_df", path_ga_history_df),
        ("BEST_PATH_STRATEGY (settings)", {"name": BEST_PATH_STRATEGY.name, **BEST_PATH_STRATEGY.parameters()}),
        (
            "BEST_PATH_RUN (summary)",
            {
                "status": BEST_PATH_RUN["status"],
                "discounted_reward": round(BEST_PATH_RUN["discounted_reward"], 1),
                "total_spice_eq_produced": round(BEST_PATH_RUN["total_spice_eq_produced"], 1),
                "stopped_at_hours": BEST_PATH_RUN["stopped_at_hours"],
                "steps_started": BEST_PATH_RUN["steps_started"],
                "steps_completed": BEST_PATH_RUN["steps_completed"],
                "events": BEST_PATH_RUN["events"],
                "reviews": BEST_PATH_RUN["reviews"],
            },
        ),
    ],
)

### 11.7 Best `W`-Week Path, Weekly Production and Max Factor Levels

The highest-reward `W`-week path in detail:

- `BEST_PATH_SUMMARY` — discounted SpiceEq reward, undiscounted `W`-week production, week-1 and week-`W` SpiceEq, actions started/completed/running within the `W` weeks, colonies founded, and resource totals.
- `best_path_weekly_df` — one row per week `1..PATH_REWARD_WEEKS`: discount weight `gamma ** (w - 1)`, resources produced, weekly `spice_eq`, `discounted_spice_eq` and cumulative discounted reward.
- `best_path_levels_df` — max level reached across the empire by the end of week `W` for every factor (`colonies`, every `building` in catalog order, and every `research` in catalog order), alongside starting max, in-progress max and saved target max.
- `best_path_output_df`, `best_path_progress_df`, `best_path_queues_df` — output milestones, day-by-day checkpoints up to week `W`, and per-queue busy/idle hours.
- `best_path_steps_df` — every action started within the `W` weeks, in the causal order in which they start.

In [ ]:
def path_output_timeline(simulator: EmpirePathSimulator, run: dict[str, Any]) -> tuple[np.ndarray, np.ndarray]:
    """Hourly output of the empire along a run up to `stopped_at_hours`: `(hours, output)`."""
    count = simulator.planet_count
    limit = float(run["stopped_at_hours"])
    levels = [dict(planet["levels"]) for planet in simulator.planets]
    research = dict(simulator.problem["research_levels"])
    solar = [int(planet["solar_cells"]) for planet in simulator.planets]
    founded = [bool(planet["founded"]) for planet in simulator.planets]

    keep = simulator.keep

    def planet_output(planet: int) -> tuple[float, ...]:
        made = planet_hourly_output(
            simulator.planet_totals(planet, levels[planet], solar[planet], research["r1"], research["r2"])
        )
        return (keep[0] * made[1], keep[1] * made[2], keep[2] * made[3], keep[3] * made[4])

    outputs = [planet_output(planet) for planet in range(count)]

    def total() -> list[float]:
        return [sum(outputs[planet][position] for planet in range(count) if founded[planet]) for position in range(4)]

    finished = sorted(
        (float(end), step)
        for step, end in enumerate(run["end"])
        if end is not None and float(end) <= limit + 1e-12
    )
    hours = [0.0]
    rows = [total()]
    for end, step in finished:
        kind, planet, key = simulator.kind[step], simulator.step_planet[step], simulator.key[step]
        if kind == "research":
            research[key] = simulator.level_to[step]
            if key in ("r1", "r2"):
                outputs = [planet_output(other) for other in range(count)]
        elif kind == "colonize":
            founded[planet] = True
        elif kind == "ships":
            if key == "s9":
                solar[planet] += simulator.level_to[step]
                outputs[planet] = planet_output(planet)
        else:
            if simulator.level_to[step] > 0:
                levels[planet][key] = simulator.level_to[step]
            else:
                levels[planet].pop(key, None)
            if key in PATH_FLOW_BUILDING_KEYS:
                outputs[planet] = planet_output(planet)
        hours.append(float(end))
        rows.append(total())
    return np.array(hours, dtype=np.float64), np.array(rows, dtype=np.float64)


def path_weekly_production_table(simulator: EmpirePathSimulator, run: dict[str, Any]) -> pd.DataFrame:
    """One row per week `1..reward_weeks`: resources produced, SpiceEq, discount weight and discounted reward."""
    cumulative = 0.0
    rows: list[dict[str, Any]] = []
    for week_idx in range(simulator.reward_weeks):
        weight = float(simulator.week_weights[week_idx])
        amounts = run["weekly_resources"][week_idx]
        spice_eq = float(run["weekly_spice_eq"][week_idx])
        discounted = weight * spice_eq
        cumulative += discounted
        rows.append(
            {
                "week": week_idx + 1,
                "discount_weight": weight,
                **{resource: float(amounts[pos]) for pos, resource in enumerate(RESOURCE_KEYS)},
                "spice_eq": spice_eq,
                "discounted_spice_eq": discounted,
                "cumulative_discounted_spice_eq": cumulative,
            }
        )
    return pd.DataFrame(rows)


def path_factor_levels_table(simulator: EmpirePathSimulator, run: dict[str, Any]) -> pd.DataFrame:
    """Max level reached across the empire at the end of the run for colonies, buildings and researches."""
    limit = float(run["stopped_at_hours"])
    rows: list[dict[str, Any]] = [
        {
            "factor_kind": "colonies",
            "key": "colonies",
            "name": "Colonies founded",
            "start_max_level": sum(1 for planet in simulator.planets if planet["founded"]),
            "reached_max_level": sum(1 for flag in run["final_founded"] if flag),
            "started_max_level": (
                sum(1 for planet in simulator.planets if planet["founded"])
                + sum(
                    1
                    for step in run["order"]
                    if simulator.kind[step] == "colonize" and float(run["start"][step]) <= limit
                )
            ),
            "target_max_level": simulator.planet_count,
            "planets_at_reached_max": sum(1 for flag in run["final_founded"] if flag),
        }
    ]
    building_keys = sorted(
        {
            key
            for planet in simulator.planets
            for key, level in (*planet["levels"].items(), *planet["target_levels"].items())
            if level > 0
        },
        key=lambda item: int(item[1:]),
    )
    started_building_max: dict[str, int] = {
        key: max((planet["levels"].get(key, 0) for planet in simulator.planets), default=0) for key in building_keys
    }
    for step in run["order"]:
        if simulator.kind[step] == "build" and simulator.key[step] in started_building_max:
            started_building_max[simulator.key[step]] = max(
                started_building_max[simulator.key[step]], simulator.level_to[step]
            )
    for key in building_keys:
        building_id = int(key[1:])
        reached = max((levels.get(key, 0) for levels in run["final_levels"]), default=0)
        rows.append(
            {
                "factor_kind": "building",
                "key": key,
                "name": str(game_data.BUILDING_BY_ID[building_id]["name"]),
                "start_max_level": max((planet["levels"].get(key, 0) for planet in simulator.planets), default=0),
                "reached_max_level": reached,
                "started_max_level": max(reached, started_building_max[key]),
                "target_max_level": max(
                    (planet["target_levels"].get(key, 0) for planet in simulator.planets), default=0
                ),
                "planets_at_reached_max": (
                    sum(1 for levels in run["final_levels"] if levels.get(key, 0) == reached) if reached > 0 else 0
                ),
            }
        )
    research_keys = sorted(
        {
            key
            for key, level in (
                *simulator.problem["research_levels"].items(),
                *simulator.problem["research_target"].items(),
            )
            if level > 0
        },
        key=lambda item: int(item[1:]),
    )
    started_research: dict[str, int] = {
        key: int(simulator.problem["research_levels"].get(key, 0)) for key in research_keys
    }
    for step in run["order"]:
        if simulator.kind[step] == "research" and simulator.key[step] in started_research:
            started_research[simulator.key[step]] = max(started_research[simulator.key[step]], simulator.level_to[step])
    for key in research_keys:
        reached = int(run["final_research"].get(key, 0))
        rows.append(
            {
                "factor_kind": "research",
                "key": key,
                "name": str(game_data.RESEARCH_BY_ID[key]["name"]),
                "start_max_level": int(simulator.problem["research_levels"].get(key, 0)),
                "reached_max_level": reached,
                "started_max_level": max(reached, started_research[key]),
                "target_max_level": int(simulator.problem["research_target"].get(key, 0)),
                "planets_at_reached_max": 1 if reached > 0 else 0,
            }
        )
    return pd.DataFrame(rows)


def path_steps_table(simulator: EmpirePathSimulator, run: dict[str, Any]) -> pd.DataFrame:
    """Every step started within the run window, in the causal order in which the simulator started them."""
    rows: list[dict[str, Any]] = []
    for order, step in enumerate(run["order"], start=1):
        details = simulator.steps[step]
        start = float(run["start"][step])
        end = float(run["end"][step])
        planet = details["planet"]
        rows.append(
            {
                "order": order,
                "start_hours": start,
                "start": fmt_path_time(start),
                "duration_hours": end - start,
                "end_hours": end,
                "end": fmt_path_time(end),
                "planet": simulator.planets[planet]["coords"] if planet >= 0 else "account",
                "kind": details["kind"],
                "key": details["key"],
                "name": details["name"],
                "from_level": details["level_from"],
                "to_level": details["level_to"],
                **dict(zip(RESOURCE_KEYS, details["cost"])),
                "cost_spice_eq": details["cost_spice_eq"],
            }
        )
    columns = [
        "order", "start_hours", "start", "duration_hours", "end_hours", "end", "planet", "kind", "key", "name",
        "from_level", "to_level", *RESOURCE_KEYS, "cost_spice_eq",
    ]
    return pd.DataFrame(rows, columns=columns)


def path_queues_table(simulator: EmpirePathSimulator, run: dict[str, Any], bounds: pd.DataFrame) -> pd.DataFrame:
    """How busy each building queue and the research queue were within the run window."""
    count = simulator.planet_count
    limit = float(run["stopped_at_hours"])
    least = dict(zip(bounds["limit"], bounds["hours"]))
    rows: list[dict[str, Any]] = []
    for queue in range(count + 1):
        if queue < count:
            name = f"building queue of {simulator.planets[queue]['coords']}"
            colonize = simulator.colonize_step[queue]
            free_from = (
                min(limit, float(run["end"][colonize]))
                if colonize >= 0 and run["end"][colonize] is not None
                else (0.0 if simulator.planets[queue]["founded"] else limit)
            )
        else:
            name, free_from = "research queue", 0.0
        spans = [
            (float(run["start"][step]), min(limit, float(run["end"][step])))
            for step in range(len(simulator.steps))
            if simulator.queue[step] == queue and run["start"][step] is not None
        ]
        busy = sum(max(0.0, end - start) for start, end in spans)
        last_end = max((end for _, end in spans), default=free_from)
        rows.append(
            {
                "queue": name,
                "steps": len(spans),
                "free_from_hours": free_from,
                "busy_hours": busy,
                "idle_hours": max(0.0, last_end - free_from - busy),
                "last_end_hours": last_end,
                "last_end": fmt_path_time(last_end),
                "least_hours": float(least.get(name, 0.0)),
            }
        )
    return pd.DataFrame(rows)


def path_progress_table(
    simulator: EmpirePathSimulator, run: dict[str, Any], after_days: tuple[float, ...]
) -> pd.DataFrame:
    """Where the empire stands at given days of a run: steps done and running, and its hourly output."""
    hours, output = path_output_timeline(simulator, run)
    last = float(run["stopped_at_hours"])
    final_value = float(game_data.spice_equivalent(dict(zip(RESOURCE_KEYS, simulator.final_net_output_total))))
    moments = sorted({0.0, *(24.0 * float(days) for days in after_days if 24.0 * float(days) < last), last})
    starts = np.sort(np.array([start for start in run["start"] if start is not None], dtype=np.float64))
    ends = hours[1:]
    rows: list[dict[str, Any]] = []
    for moment in moments:
        done = int(np.searchsorted(ends, moment, side="right"))
        begun = int(np.searchsorted(starts, moment, side="right"))
        made = output[done]
        value = float(game_data.spice_equivalent(dict(zip(RESOURCE_KEYS, made))))
        rows.append(
            {
                "after_days": moment / 24.0,
                "after": fmt_path_time(moment),
                "steps_done": done,
                "steps_running": begun - done,
                **{f"{resource}_per_hour": float(made[position]) for position, resource in enumerate(RESOURCE_KEYS)},
                "output_spice_eq_per_hour": value,
                "share_of_final_output": value / final_value if final_value > 0.0 else 0.0,
            }
        )
    return pd.DataFrame(rows)


def path_output_shares_table(
    simulator: EmpirePathSimulator, run: dict[str, Any], shares: tuple[float, ...]
) -> pd.DataFrame:
    """When the hourly output of a run first reaches given shares of the output of the configuration."""
    hours, output = path_output_timeline(simulator, run)
    rates = np.array(simulator.merchant_rates, dtype=np.float64)
    values = output @ rates
    final_value = float(np.array(simulator.final_net_output_total) @ rates)
    rows: list[dict[str, Any]] = []
    for share in shares:
        reached = np.flatnonzero(values >= float(share) * final_value * (1.0 - 1e-12)) if final_value > 0.0 else []
        moment = float(hours[reached[0]]) if len(reached) > 0 else float("inf")
        rows.append(
            {
                "share_of_final_output": float(share),
                "output_spice_eq_per_hour": float(share) * final_value,
                "reached_after_hours": moment,
                "reached_after": fmt_path_time(moment),
                "steps_done_by_then": int(reached[0]) if len(reached) > 0 else int(len(hours) - 1),
            }
        )
    return pd.DataFrame(rows)


def summarize_best_path(
    simulator: EmpirePathSimulator,
    strategy: PathStrategy,
    run: dict[str, Any],
    bounds: pd.DataFrame,
    queues: pd.DataFrame,
    factor_levels: pd.DataFrame,
) -> dict[str, Any]:
    """The winning W-week path in numbers: discounted reward, weekly growth, factor levels and spend."""
    hours = float(run["hours"])
    bound_hours = float(bounds["hours"].iloc[0])
    reached = run["status"] == "reached"
    started_steps = run["order"]
    cost_spent = [
        sum(simulator.cost[step][position] for step in started_steps) for position in range(4)
    ]
    loot = list(run["loot_collected"])
    spare = list(run["spare_diverted"])
    cost_value = float(game_data.spice_equivalent(dict(zip(RESOURCE_KEYS, cost_spent))))
    loot_value = float(game_data.spice_equivalent(dict(zip(RESOURCE_KEYS, loot))))
    _, output = path_output_timeline(simulator, run)
    waiting_for = run["waiting_for_step"]
    busiest = queues.sort_values("busy_hours", ascending=False, kind="stable").iloc[0]
    weekly = run["weekly_spice_eq"]
    buildings_reached = {
        str(row.name): f"{int(row.reached_max_level)}/{int(row.target_max_level)}"
        for row in factor_levels[factor_levels["factor_kind"] == "building"].itertuples(index=False)
    }
    research_reached = {
        str(row.name): f"{int(row.reached_max_level)}/{int(row.target_max_level)}"
        for row in factor_levels[factor_levels["factor_kind"] == "research"].itertuples(index=False)
    }
    return {
        "strategy": strategy.name,
        "status": run["status"],
        "reward_weeks": simulator.reward_weeks,
        "weekly_discount": simulator.weekly_discount,
        "horizon_hours": simulator.horizon_hours,
        "horizon": fmt_path_time(simulator.horizon_hours),
        "discounted_reward": float(run["discounted_reward"]),
        "reward_upper_bound": float(simulator.reward_upper_bound),
        "share_of_reward_ceiling_percent": (
            100.0 * float(run["discounted_reward"]) / simulator.reward_upper_bound
            if simulator.reward_upper_bound > 0.0
            else 0.0
        ),
        "total_spice_eq_produced": float(run["total_spice_eq_produced"]),
        "week_1_spice_eq": float(weekly[0]),
        "last_week_spice_eq": float(weekly[-1]),
        "hours": hours,
        "time": fmt_path_time(hours),
        "lower_bound_hours": bound_hours,
        "lower_bound": fmt_path_time(bound_hours),
        "lower_bound_set_by": str(bounds["limit"].iloc[0]),
        "steps": len(simulator.steps),
        "steps_started": int(run["steps_started"]),
        "steps_completed": int(run["steps_completed"]),
        "steps_running": int(run["steps_running"]),
        "stopped_at_hours": float(run["stopped_at_hours"]),
        "waiting_for": (
            f"{simulator.steps[waiting_for]['name']} {simulator.steps[waiting_for]['level_to']}"
            if waiting_for >= 0
            else None
        ),
        "colonies_founded": sum(1 for flag in run["final_founded"] if flag),
        "colonies_target": simulator.planet_count,
        "start_fleet_slots": int(run["start_fleet_slots"]),
        "final_fleet_slots": int(run["final_fleet_slots"]),
        "attacks_per_slot_per_hour": float(ATTACK_INCOME_SUMMARY["attacks_per_slot_per_hour"]),
        "spare_percent": dict(zip(RESOURCE_KEYS, simulator.spare_percent)),
        "spare_diverted": {resource: int(round(amount)) for resource, amount in zip(RESOURCE_KEYS, spare)},
        "max_building_levels_reached": buildings_reached,
        "research_levels_reached": research_reached,
        "busiest_queue": str(busiest["queue"]),
        "busy_hours_of_that_queue": float(busiest["busy_hours"]),
        "cost_spent": {resource: int(round(amount)) for resource, amount in zip(RESOURCE_KEYS, cost_spent)},
        "metal_smelted": int(round(run["metal_smelted"])),
        "loot_counted": {resource: int(round(amount)) for resource, amount in zip(RESOURCE_KEYS, loot)},
        "loot_share_of_cost_spent": loot_value / cost_value if cost_value > 0.0 else 0.0,
        "ends_with_the_output_of_the_configuration": bool(
            reached and np.allclose(output[-1], simulator.final_net_output_total, rtol=1e-9, atol=1e-6)
        ),
    }


best_path_weekly_df = path_weekly_production_table(PATH_SIMULATOR, BEST_PATH_RUN)
best_path_levels_df = path_factor_levels_table(PATH_SIMULATOR, BEST_PATH_RUN)
best_path_steps_df = path_steps_table(PATH_SIMULATOR, BEST_PATH_RUN)
best_path_queues_df = path_queues_table(PATH_SIMULATOR, BEST_PATH_RUN, path_lower_bounds_df)
best_path_progress_df = path_progress_table(PATH_SIMULATOR, BEST_PATH_RUN, PATH_PROGRESS_DAYS)
best_path_output_df = path_output_shares_table(PATH_SIMULATOR, BEST_PATH_RUN, PATH_OUTPUT_SHARES)
BEST_PATH_SUMMARY: dict[str, Any] = summarize_best_path(
    PATH_SIMULATOR,
    BEST_PATH_STRATEGY,
    BEST_PATH_RUN,
    path_lower_bounds_df,
    best_path_queues_df,
    best_path_levels_df,
)

print_section_data(
    (
        f"Section 11.7 data: best {PATH_REWARD_WEEKS}-week path, "
        f"{BEST_PATH_SUMMARY['discounted_reward']:,.1f} discounted SpiceEq with {BEST_PATH_SUMMARY['strategy']!r}"
        if BEST_PATH_SUMMARY["status"] != "stuck"
        else "Section 11.7 data: the configuration is not reached, the path stops after "
        f"{BEST_PATH_SUMMARY['steps_started']} of {BEST_PATH_SUMMARY['steps']} steps"
    ),
    [
        ("BEST_PATH_SUMMARY", BEST_PATH_SUMMARY),
        ("best_path_weekly_df", best_path_weekly_df),
        ("best_path_levels_df", best_path_levels_df),
        ("best_path_output_df", best_path_output_df),
        ("best_path_progress_df", best_path_progress_df),
        ("best_path_queues_df", best_path_queues_df),
        ("best_path_steps_df", best_path_steps_df),
    ],
)

### 11.8 Save the `W`-Week Path as Text

The last cell writes the `PATH_REWARD_WEEKS` weeks of actions into `METADATA_DIR` as `full_planner_path.txt` (`PATH_ACTIONS_FILE_NAME`), replacing the previous file. It is the only file that this notebook writes.

The file begins with a 14-line header reporting:

1. when the file was written, the saved configuration, and the winning strategy;
2. the `W`-week horizon, weekly discount factor, `Attacks & spare:` (starting -> ending fleet slots from `Fleet base` `b10` + colonies, attacks/hr, and per-resource spare %), and net `Reward:` (discounted SpiceEq, total `W`-week net SpiceEq, week-1 and week-`W` production);
3. the `Result:` (`W`-week horizon completion counts, full configuration reached early, or stuck reason);
4. the **max level reached of each factor** across the empire by the end of week `W`:
   - `Colonies reached: <founded>/<total> founded`
   - `Max building levels reached: <Building> <reached>/<target>, ...`
   - `Research levels reached: <Technology> <reached>/<target>, ...`

Then every action started within those `W` weeks follows on its own numbered line, in causal start order:

```
   1. at 0s on 02:380:12: build Pig iron mine level 22 (takes 44m 20s)
   2. at 12m 05s: research Energy technology level 5 (takes 6m 07s)
```

- `path_action_lines` formats the `W`-week actions from `best_path_steps_df`; `path_actions_text` prepends the 14-line header.
- `save_path_actions` writes `PATH_ACTIONS_PATH` and returns `PATH_ACTIONS_SUMMARY`.

In [ ]:
def path_action_lines(steps_table: pd.DataFrame) -> list[str]:
    """One numbered line per step started in the W-week window, in causal start order."""
    width = len(str(len(steps_table)))
    lines: list[str] = []
    for row in steps_table.itertuples(index=False):
        if row.kind == "build":
            action = f"build {row.name} level {row.to_level}"
        elif row.kind == "demolish":
            action = f"tear down {row.name} to level {row.to_level}"
        elif row.kind == "research":
            action = f"research {row.name} level {row.to_level}"
        elif row.kind == "ships":
            action = f"build {row.to_level} x {row.name}"
        else:
            action = "found the colony"
        where = "" if row.planet == "account" else f" on {row.planet}"
        lines.append(
            f"{row.order:>{width}}. at {row.start}{where}: {action} (takes {fmt_path_time(row.duration_hours)})"
        )
    return lines


def format_factor_levels(entries: dict[str, str]) -> str:
    """Comma-separated `<Factor> <reached>/<target>` summary for the text header (`none` when empty)."""
    return ", ".join(f"{name} {value}" for name, value in entries.items()) if entries else "none"


def path_actions_text(lines: list[str], summary: dict[str, Any], target_summary: dict[str, Any]) -> str:
    """The text of the W-week path file: a 14-line header with slots, spare%, reward & max factor levels."""
    weeks = int(summary["reward_weeks"])
    discount = float(summary["weekly_discount"])
    if summary["status"] == "reached":
        result = f"configuration reached after {summary['time']} within the {weeks}-week horizon ({len(lines)} actions)"
    elif summary["status"] == "horizon":
        result = (
            f"{weeks}-week horizon ended with {summary['steps_started']} of {summary['steps']} actions started "
            f"({summary['steps_completed']} finished, {summary['steps_running']} running)"
        )
    else:
        result = (
            f"NOT reached. The path stops after {summary['steps_started']} of {summary['steps']} actions: "
            f"the empire can never pay for {summary['waiting_for']}"
        )
    written = datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
    buildings_line = format_factor_levels(summary["max_building_levels_reached"])
    research_line = format_factor_levels(summary["research_levels_reached"])
    s0, s1 = int(summary["start_fleet_slots"]), int(summary["final_fleet_slots"])
    per_slot = float(summary["attacks_per_slot_per_hour"])
    spare_txt = ", ".join(f"{res} {pct:.1f}%" for res, pct in summary["spare_percent"].items())
    header = [
        f"Best {weeks}-week path toward the saved configuration (notebooks/full_planner.ipynb, Section 11)",
        f"Written: {written}",
        f"Configuration: {target_summary['file']}, saved {target_summary['saved_at']}",
        f"Strategy: {summary['strategy']}",
        f"Horizon: {weeks} weeks ({summary['horizon']}), weekly discount {discount:.2f}",
        f"Attacks & spare: {s0} -> {s1} fleet slots ({s0 * per_slot:.2f} -> {s1 * per_slot:.2f} attacks/h); "
        f"spare {spare_txt}",
        f"Reward: {summary['discounted_reward']:,.1f} SpiceEq net discounted "
        f"({summary['total_spice_eq_produced']:,.1f} net SpiceEq produced in {weeks} weeks; "
        f"week 1: {summary['week_1_spice_eq']:,.1f}, week {weeks}: {summary['last_week_spice_eq']:,.1f})",
        f"Result: {result}",
        f"Colonies reached: {summary['colonies_founded']}/{summary['colonies_target']} founded",
        f"Max building levels reached: {buildings_line}",
        f"Research levels reached: {research_line}",
        f"Actions: {len(lines)} within {weeks} weeks, in the order in which they start. "
        "Times count from the moment this file was written.",
        "Planets are named by their coordinates. "
        'EMPTY:nn is an empty colony slot: "found the colony" comes first there.',
        "",
    ]
    return "\n".join([*header, *lines]) + "\n"


def save_path_actions(
    path: Path, steps_table: pd.DataFrame, summary: dict[str, Any], target_summary: dict[str, Any]
) -> dict[str, Any]:
    """Write the W-week actions and max factor levels header to `path` as text; return what was written."""
    lines = path_action_lines(steps_table)
    text = path_actions_text(lines, summary, target_summary)
    path.write_text(text, encoding="utf-8")
    return {
        "file": path.name,
        "folder": str(path.parent),
        "status": summary["status"],
        "reward_weeks": int(summary["reward_weeks"]),
        "discounted_reward": round(float(summary["discounted_reward"]), 1),
        "actions": len(lines),
        "lines": text.count("\n"),
        "first_action": lines[0] if lines else None,
        "last_action": lines[-1] if lines else None,
    }


PATH_ACTIONS_PATH: Path = METADATA_DIR / PATH_ACTIONS_FILE_NAME
PATH_ACTIONS_SUMMARY: dict[str, Any] = save_path_actions(
    PATH_ACTIONS_PATH, best_path_steps_df, BEST_PATH_SUMMARY, PATH_TARGET_SUMMARY
)

print_section_data(
    f"Section 11.8 data: the path as text, {PATH_ACTIONS_SUMMARY['actions']} actions "
    f"({PATH_REWARD_WEEKS} weeks) saved in {PATH_ACTIONS_PATH.name}",
    [
        ("PATH_ACTIONS_PATH", str(PATH_ACTIONS_PATH)),
        ("PATH_ACTIONS_SUMMARY", PATH_ACTIONS_SUMMARY),
    ],
)